# TPI IA 2026P · Comprensión visual de escenas

**Grupo 15 · Curso 5K1**

**Equipo Kaggle:** Grupo 15 - 5K1

## Integrantes

- Gonzalo Ezequiel Majlis (95047)
- Tomas Morinigo (95639)
- Ramiro Figueroa Manzi (96306)
- Franco Manuel Lucero (96311)

Esta notebook es la entrega reproducible de la **Etapa 1 — Promoción**. Implementa la pipeline
imagen → detección open-vocabulary → SAM2 → SceneJSON → QA determinista/LLM, junto con
trazabilidad, evaluación por etapa, experimento controlado y análisis de propagación de errores.


## 1. Preparación y reproducibilidad

El código de inferencia se reconstruye en celdas etiquetadas `inference`. Los módulos que leen
anotaciones se etiquetan `ground-truth` y solo se usan en EDA/evaluación. Ninguna predicción,
respuesta o SceneJSON usa GT. Las semillas, versiones, hardware y modelos quedan en el manifest.


In [ ]:

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/kaggle/working/tpi-ia-2026")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
SRC_ROOT = str(PROJECT_ROOT / "src")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print("Directorio de trabajo:", Path.cwd())

for directory in ['.', 'configs', 'configs/experiments', 'scripts', 'src/eda', 'src/eval', 'src/scene', 'src/scene/detection', 'src/scene/io', 'src/scene/llm', 'src/scene/llm/prompts', 'src/scene/masks', 'src/scene/qa', 'src/scene/scene', 'src/scene/segmentation', 'src/scene/tracing']:
    Path(directory).mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile configs/base.yaml
# Configuración de la línea base. Los experimentos son overrides en configs/experiments/*.yaml.
seed: 42
device: auto            # auto | cpu | cuda | mps  (auto: cuda si hay, si no cpu)

paths:
  # Raíz de la fuente de datos (misma estructura que el mount de Kaggle).
  data_root: data/dataset
  outputs: outputs/runs
  # Caché por etapa compartida entre runs (detecciones crudas y máscaras), por hash de config.
  cache: outputs/cache
  submission: submission.csv   # nombre del archivo dentro del run
  dev_images: development/images
  test_images: tpi_ia_2026_competition_public/test_images
  test_index: tpi_ia_2026_competition_public/test_images.csv
  categories: tpi_ia_2026_competition_public/categories.csv
  questions: tpi_2026_qa_public/questions_dev.jsonl
  scene_schema: scene_schema.json
  sample_submission: sample_submission.csv

dataset:
  image_extensions: [".jpg", ".jpeg", ".png"]

vocab:
  mode: clean               # clean | synonyms (sinónimos LVIS, experimento E1-30)
  prompt_template: "a photo of a {}"
  max_synonyms: null        # tope de sinónimos por categoría (null = todos)
  # Fuente pública de sinónimos LVIS v1 (se descarga en tiempo de ejecución y se cachea).
  lvis_categories_url: https://raw.githubusercontent.com/facebookresearch/detectron2/main/detectron2/data/datasets/lvis_v1_categories.py
  synonyms_cache: data/cache/lvis_synonyms.json
  # Nombres de la competencia que no coinciden con el nombre LVIS (valor = nombre LVIS crudo).
  lvis_aliases:
    laptop: laptop_computer
    cell phone: cellular_telephone
    computer monitor: monitor_(computer_equipment) computer_monitor

detection:
  detector: owlv2
  owlv2:
    model_id: google/owlv2-base-patch16-ensemble
    # Piso de score para las detecciones crudas que se cachean: permite barrer el umbral
    # (E1-29) sin volver a correr el modelo.
    score_floor: 0.01
    raw_topk: 300           # tope de detecciones crudas por imagen (caché)
    labels_per_box: 1       # queries que se conservan por caja (EDA H6: hay objetos con 2 clases)
    query_batch_size: null  # null = todas las queries en una pasada
  score_threshold: 0.18     # E1-29: máx. F1 en tuning (80 img); holdout F1 0,666 vs 0,619 con 0,1 (D-14)
  per_class_threshold: {}   # nombre de categoría → umbral (solo experimentos)
  nms_iou: 0.5              # NMS por clase (EDA H6: solo 1,7 % de duplicados reales con IoU ≥ 0,5)
  cross_class_nms:
    enabled: false          # D-09: fuera de la línea base, se evalúa en E1-30
    iou: 0.8
  max_detections: 100       # tope por imagen (EDA: máx. 66 instancias GT; schema admite 300)

segmentation:
  model_id: facebook/sam2.1-hiera-large
  multimask_output: true    # 3 máscaras por caja; queda la de mayor IoU predicho
  box_batch_size: 32        # cajas por pasada del decoder (el embedding de imagen se reutiliza)
  mask_threshold: 0.0       # umbral sobre los logits de SAM2
  postprocess:
    crop_to_box: true
    box_margin: 0.1         # fracción del ancho/alto de la caja
    largest_component: true
    fill_holes: true
    max_hole_ratio: 0.05    # huecos de hasta 5 % del área de la máscara
    # Post-procesar solo máscaras crudas con área relativa menor a esto (null = todas). E1-30.
    apply_below_area_ratio: null

scene:
  # Umbrales de tamaño sobre area_ratio (el schema no los fija; ver docs/eda_findings.md).
  # Equivalen a 32² y 96² px (convención COCO) sobre una imagen de referencia de 640×480.
  size_thresholds:
    small_max: 0.00333
    medium_max: 0.03
  color:
    k: 3                    # clusters de k-means en Lab
    max_pixels: 5000        # submuestreo de píxeles de la máscara (con la semilla global)
    min_pixels: 20          # por debajo, el color queda desconocido (null)
    # Paleta de 11 colores básicos (RGB de referencia).
    palette:
      black: [20, 20, 20]
      white: [245, 245, 245]
      gray: [128, 128, 128]
      red: [200, 30, 30]
      orange: [240, 140, 30]
      yellow: [240, 220, 40]
      green: [40, 150, 50]
      blue: [40, 80, 200]
      purple: [130, 50, 160]
      pink: [240, 150, 180]
      brown: [120, 75, 40]
  relations:
    # Márgenes como fracción del ancho (horizontal) o alto (vertical) de la imagen: si los
    # centroides están más cerca que eso, no se afirma la relación.
    h_margin: 0.02
    v_margin: 0.02
    overlap_min_iou: 0.01   # IoU de máscaras a partir del cual se dice `overlaps`
    near_max_gap: 0.05      # distancia entre cajas / diagonal de la imagen (extra_relations)
    on_top_min_x_overlap: 0.3  # solape horizontal / ancho del sujeto (extra_relations)
    contact_px: 3           # dilatación para considerar que dos máscaras se tocan
    max_relations: 20000    # tope del schema; se priorizan los pares de mayor confianza

qa:
  matcher:
    # Texto de categoría → category_id: nombre, léxico español, sinónimos LVIS y, por último,
    # embeddings (E1-21). Ruta relativa al directorio de trabajo, como vocab.synonyms_cache.
    lexicon_es: configs/lexicon_es.yaml
    lvis_synonyms: true       # usa vocab.lvis_categories_url / vocab.synonyms_cache
    embeddings:
      enabled: true           # el modelo se carga recién si el léxico no alcanza
      model_id: sentence-transformers/LaBSE
      # D-13: en la calibración, el negativo más alto dio 0,887 ("puerta" → doorknob).
      threshold: 0.9
      batch_size: 64
  executor:
    describe_max_categories: 5   # categorías que se nombran en la descripción determinista
    # Colores en español → nombre de la paleta (scene.color.palette).
    color_aliases:
      negro: black
      blanco: white
      gris: gray
      rojo: red
      naranja: orange
      amarillo: yellow
      verde: green
      azul: blue
      violeta: purple
      morado: purple
      rosa: pink
      rosado: pink
      marron: brown
      marrón: brown
  adapter:
    # Frase del texto de la pregunta → predicado (el benchmark no estructura la relación).
    relation_phrases:
      por encima de: above
      encima de: above
      arriba de: above
      por debajo de: below
      debajo de: below
      abajo de: below
      a la derecha de: right_of
      a la izquierda de: left_of
      superpuesto con: overlaps
      se superpone con: overlaps
      cerca de: near
      sobre: on_top_of

# Solo lo usa src/eval/ y src/eda/ (nunca la inferencia).
eval:
  annotations: development/annotations.json
  iou_thr: 0.5
  centroid_decimals: 2     # tolerancia del chequeo de centroides (el builder redondea a 2)
  error_analysis:
    localization_min_iou: 0.1  # FP con IoU en [0,1; 0,5) contra GT de su clase = mala localización
    gallery_per_group: 4       # imágenes por grupo (correctas, ambiguas, fallidas)
    worst_classes_min_gt: 5    # mínimo de instancias GT para listar una clase como de las peores
  # Bins de frecuencia por cantidad de imágenes donde la categoría es positiva (EDA: 3 a 11).
  frequency_bins:
    rare_max: 4
    common_max: 7
  # IoU de caja a partir del cual dos instancias de la misma clase se consideran duplicadas (EDA).
  eda_duplicate_iou: 0.5

tracing:
  trace_file: trace.jsonl
  manifest_file: run_manifest.json

llm:
  selected: qwen25_3b
  candidates:
    qwen25_3b:
      model_id: Qwen/Qwen2.5-3B-Instruct
      precision: fp16
      chat_template_kwargs: {}
    qwen3_4b:
      model_id: Qwen/Qwen3-4B
      precision: 4bit
      chat_template_kwargs:
        enable_thinking: false
  max_new_tokens: 256
  seed: 42
  trust_remote_code: false
  interpreter:
    prompt_version: interpreter_v2.txt  # D-18: v1 tenía ejemplos del benchmark
    max_attempts: 2
  describer:
    prompt_version: describer_v1.txt
    max_new_tokens: 160
    max_objects: 100
    max_relations: 100
    retry_on_unfaithful: 1
  benchmark:
    warmup: 1
    repeats: 3


In [ ]:
%%writefile configs/delivery.yaml
delivery:
  group_number: "15"
  course: 5K1
  kaggle_team: "Grupo 15 - 5K1"
  members:
    - name: Gonzalo Ezequiel Majlis
      student_id: "95047"
    - name: Tomas Morinigo
      student_id: "95639"
    - name: Ramiro Figueroa Manzi
      student_id: "96306"
    - name: Franco Manuel Lucero
      student_id: "96311"
  notebook:
    output: notebooks/TPI_IA_2026P_15.ipynb
    run_oracle_segmentation: true
    run_threshold_tuning: true
    run_controlled_experiment: true
    run_llm_benchmark: true


In [ ]:
%%writefile configs/experiments/e1_30_pp_solo_chicas.yaml
# E1-30 · Post-proceso de máscaras solo en máscaras chicas (D-15).
# Hipótesis: con cajas GT el post-proceso sube IoU ≥ 0,5 en small (72 % → 75 %) y lo baja en
# medium (96 % → 94 %) y large (97 % → 95 %); aplicarlo solo a las chicas sube el F1 de máscaras.
# Única variable respecto de base.yaml. El umbral es el corte de `small` (scene.size_thresholds).
segmentation:
  postprocess:
    apply_below_area_ratio: 0.00333


In [ ]:
%%writefile configs/experiments/e1_30_sin_postproceso.yaml
# E1-30 · Control: sin post-proceso de máscaras (máscara cruda de SAM2 para todos los objetos).
# Sirve para separar el aporte del post-proceso en small del perjuicio en medium/large.
segmentation:
  postprocess:
    crop_to_box: false
    largest_component: false
    fill_holes: false


In [ ]:
%%writefile configs/kaggle.yaml
# Override para correr en Kaggle: se combina con base.yaml.
device: cuda
paths:
  data_root: /kaggle/input/competitions/tpi-ia-2026-primavera-comprension-visual-de-escenas
  outputs: /kaggle/working/outputs/runs
  cache: /kaggle/working/outputs/cache


In [ ]:
%%writefile configs/lexicon_es.yaml
# Léxico español → categoría del vocabulario (nombre en inglés de categories.csv).
# Traducción manual de los 120 nombres, más variantes regionales comunes. Lo usa
# scene.qa.category_matcher (E1-21). Los acentos y las mayúsculas se normalizan al comparar.
# Un término que aparezca en dos categorías se descarta en ambas (no se adivina).
sofa: [sofá, canapé]
laptop: [computadora portátil, ordenador portátil, portátil, notebook, laptop]
pillow: [almohada, almohadón, cojín]
bench: [banco, banca]
jacket: [chaqueta, campera, chamarra, abrigo]
signboard: [cartel, letrero, rótulo]
backpack: [mochila]
traffic light: [semáforo]
umbrella: [paraguas, sombrilla]
horse: [caballo]
handle: [mango, asa, manija, agarradera]
chair: [silla]
clock: [reloj, reloj de pared]
napkin: [servilleta]
pizza: [pizza]
flag: [bandera]
person: [persona, gente, hombre, mujer, niño, niña]
trousers: [pantalón, pantalones]
trash can: [tacho de basura, cesto de basura, bote de basura, papelera, basurero, contenedor de basura]
sock: [calcetín, media]
vase: [jarrón, florero]
bicycle: [bicicleta, bici]
tennis racket: [raqueta de tenis, raqueta]
necklace: [collar]
cell phone: [teléfono celular, celular, teléfono móvil, móvil, teléfono]
book: [libro]
sunglasses: [gafas de sol, lentes de sol, anteojos de sol]
lamp: [lámpara]
bottle: [botella]
glove: [guante]
shoe: [zapato, zapatilla, calzado]
blanket: [manta, frazada, cobija]
computer keyboard: [teclado de computadora, teclado]
dog: [perro]
jean: [jean, vaquero, pantalón de jean, pantalón vaquero]
short pants: [pantalón corto, pantalones cortos, shorts, short, bermuda]
flower arrangement: [arreglo floral, ramo de flores, ramo, flores]
ski: [esquí]
cat: [gato]
wheel: [rueda]
faucet: [grifo, canilla, llave de agua]
helmet: [casco]
table: [mesa]
drinking glass: [vaso]
headlight: [faro delantero, faro]
streetlight: [farola, farol, poste de luz, luminaria]
television set: [televisor, televisión, tele, tv]
belt: [cinturón]
bird: [pájaro, ave]
knife: [cuchillo]
watch: [reloj de pulsera, reloj pulsera]
handbag: [bolso, cartera, bolso de mano]
street sign: [señal de tránsito, señal de tráfico, señal vial, cartel de calle, letrero de calle]
bowl: [cuenco, bol, tazón]
ski pole: [bastón de esquí]
hat: [sombrero]
plate: [plato]
car: [auto, coche, carro, automóvil]
curtain: [cortina]
elephant: [elefante]
bed: [cama]
drawer: [cajón]
earring: [arete, aro, pendiente]
boat: [barco, bote, lancha, barca]
necktie: [corbata]
skateboard: [patineta, monopatín, skate]
wall socket: [tomacorriente, enchufe, toma de corriente]
fork: [tenedor]
bus: [autobús, colectivo, ómnibus, micro, bus]
giraffe: [jirafa]
toilet: [inodoro, retrete, váter]
surfboard: [tabla de surf]
motorcycle: [motocicleta, moto]
cabinet: [gabinete, armario, alacena]
taillight: [luz trasera, faro trasero]
train: [tren]
baseball cap: [gorra de béisbol, gorra]
airplane: [avión]
jersey: [camiseta, remera, playera, jersey]
pole: [poste, palo]
doorknob: [pomo de puerta, pomo, picaporte, perilla de puerta]
baseball bat: [bate de béisbol, bate]
shirt: [camisa]
license plate: [matrícula, patente, placa]
banana: [banana, plátano]
mirror: [espejo]
kite: [cometa, barrilete, papalote]
frisbee: [frisbee, disco volador]
sink: [lavabo, lavamanos, pileta, fregadero, bacha]
spectacles: [gafas, lentes, anteojos]
button: [botón]
goggles: [antiparras, gafas protectoras, gafas de esquí]
zebra: [cebra]
suitcase: [maleta, valija]
baseball glove: [guante de béisbol]
basket: [canasta, cesta, canasto]
refrigerator: [refrigerador, heladera, nevera]
ski boot: [bota de esquí]
tablecloth: [mantel]
bracelet: [pulsera, brazalete]
teddy bear: [oso de peluche, osito de peluche, peluche]
windshield wiper: [limpiaparabrisas]
computer mouse: [mouse de computadora, mouse, ratón]
box: [caja]
cup: [taza]
dress: [vestido]
computer monitor: [monitor de computadora, monitor, pantalla]
broccoli: [brócoli]
strap: [correa, tira]
knob: [perilla]
plastic bag: [bolsa de plástico, bolsa]
ring: [anillo]
awning: [toldo]
vent: [rejilla de ventilación, ventilación, respiradero]
banner: [pancarta, estandarte]
cow: [vaca]
flowerpot: [maceta, macetero]
tag: [etiqueta]
fireplug: [hidrante, boca de incendio]
boot: [bota]


In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "tpi-ia-2026"
version = "0.1.0"
description = "TPI IA 2026 (UTN FRC): comprensión visual de escenas"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "numpy",
    "pillow",
    "opencv-python",
    "pyyaml",
    "pydantic>=2",
    "jsonschema>=4.18",
    "pycocotools",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "torch",
    "torchvision",  # lo requiere el procesador de SAM2 en transformers
    "transformers",
]

[project.optional-dependencies]
# lvis-api solo se usa como referencia de AP en evaluación.
lvis = ["lvis"]
dev = ["pytest", "ruff", "nbformat>=5.10"]
llm = [
    "accelerate>=0.30",
    "bitsandbytes>=0.43; platform_system == 'Linux'",
]

[tool.setuptools.package-data]
"scene.llm" = ["prompts/*.txt"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra"
# pycocotools todavía usa una API de numpy deprecada; no es un problema nuestro.
filterwarnings = ["ignore::DeprecationWarning:pycocotools.*"]

[tool.ruff]
line-length = 100
target-version = "py310"
extend-exclude = ["notebooks", "data", "outputs"]

[tool.ruff.lint]
select = ["E", "F", "W", "I", "UP", "B"]


In [ ]:
%%writefile scripts/analyze_errors.py
"""Análisis de errores de un run de dev (E1-31): embudo por instancia, tipos de FP, peores clases
y galería de casos.

Uso:
    python scripts/analyze_errors.py --config configs/base.yaml configs/kaggle.yaml --run latest

Necesita la caché de detecciones crudas de la misma config (la deja run_pipeline.py).
Salidas en <run>/error_analysis/: summary.json, funnel.csv, fp_types.csv, per_image.csv,
worst_classes.csv y gallery/<grupo>_<image_id>.png.
"""

from __future__ import annotations

import argparse
import json
from collections import Counter
from pathlib import Path

from eval.annotations import annotations_path, load_annotations
from eval.error_analysis import (
    draw_case,
    fp_types,
    funnel_rows,
    funnel_summary,
    per_image_scores,
    pick_gallery,
    worst_classes,
)
from eval.lvis_federated import evaluate_from_config, predictions_from_scenes
from scene.config import load_config
from scene.detection.base import load_detections
from scene.io.dataset import list_images, load_image, load_vocabulary
from scene.pipeline import load_scenes, stage_keys


def resolve(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = Path(cfg["paths"]["outputs"]).iterdir()
    return sorted(p for p in runs if (p / "scenes").is_dir())[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--run", default="latest")
    parser.add_argument("--raw-dir", help="caché de detecciones crudas (por defecto, la de cfg)")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    ea = cfg["eval"]["error_analysis"]
    run_dir = resolve(cfg, args.run)
    # La config del run manda (umbral, NMS) para recalcular el embudo igual que la pipeline.
    run_cfg = json.loads((run_dir / cfg["tracing"]["manifest_file"]).read_text("utf-8"))["config"]
    run_cfg["paths"] = cfg["paths"]
    run_cfg["eval"] = cfg["eval"]  # los parámetros de evaluación no son parte de la inferencia
    raw_dir = (
        Path(args.raw_dir)
        if args.raw_dir
        else Path(cfg["paths"]["cache"]) / "detections_raw" / stage_keys(run_cfg)["detection"]
    )
    gt = load_annotations(annotations_path(cfg))
    scenes = {i: s for i, s in load_scenes(run_dir).items() if i in gt}
    if not scenes:
        raise SystemExit(f"{run_dir} no tiene escenas de dev")
    ids = sorted(scenes)
    raw = {i: load_detections(raw_dir / f"{i}.json") for i in ids}
    names = {c.id: c.name for c in load_vocabulary(cfg)}

    preds = predictions_from_scenes(scenes)
    box = evaluate_from_config(preds, gt, run_cfg, iou_type="bbox", image_ids=ids)
    segm = evaluate_from_config(preds, gt, run_cfg, iou_type="segm", image_ids=ids)

    funnel = funnel_rows(gt, raw, box, segm, run_cfg)
    fps = fp_types(scenes, gt, box, run_cfg)
    scores = per_image_scores(box)
    worst = worst_classes(box.per_category, names, ea["worst_classes_min_gt"])
    gallery = pick_gallery(scores, fps, ea["gallery_per_group"])

    out = run_dir / "error_analysis"
    out.mkdir(exist_ok=True)
    funnel.assign(category_name=funnel["category_id"].map(names)).to_csv(
        out / "funnel.csv", index=False
    )
    fps.assign(category_name=fps["category_id"].map(names)).to_csv(
        out / "fp_types.csv", index=False
    )
    scores.to_csv(out / "per_image.csv", index=False)
    worst.to_csv(out / "worst_classes.csv", index=False)

    lost = funnel[funnel["lost_at"] != "ok"]
    confusions = Counter(
        f"{names.get(r.category_id)} ← {names.get(int(r.confused_with))}"
        for r in fps[fps["fp_type"] == "class_confusion"].itertuples()
    )
    summary = {
        "run": str(run_dir),
        "n_images": len(ids),
        "funnel": funnel_summary(funnel),
        "fp_types": fps["fp_type"].value_counts().to_dict() if len(fps) else {},
        "fp_types_by_score": (
            fps.assign(bin=(fps["score"] // 0.1 * 0.1).round(1))
            .groupby(["bin", "fp_type"])
            .size()
            .unstack(fill_value=0)
            .to_dict(orient="index")
            if len(fps)
            else {}
        ),
        "top_class_confusions": dict(confusions.most_common(10)),
        "most_lost_classes": lost["category_id"].map(names).value_counts().head(10).to_dict(),
        "worst_classes": worst.head(10).to_dict(orient="records"),
        "gallery": gallery,
    }
    (out / "summary.json").write_text(json.dumps(summary, indent=2, default=str), "utf-8")

    recs = {r.image_id: r for r in list_images(cfg, "dev")}
    for group, group_ids in gallery.items():
        for iid in group_ids:
            draw_case(
                load_image(recs[iid]),
                gt[iid],
                scenes[iid],
                box,
                names,
                out / "gallery" / f"{group}_{iid}.png",
            )
    printable = {k: v for k, v in summary.items() if k != "worst_classes"}
    print(json.dumps(printable, indent=2, default=str))
    print(f"salidas en {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/benchmark_llm.py
"""Benchmark reproducible de los candidatos LLM. Ejecutar en una GPU T4."""

from __future__ import annotations

import argparse
import gc
import json
import platform
from datetime import datetime
from pathlib import Path

from scene.config import load_config, resolve_device
from scene.llm.model import benchmark_llm, build_llm

PROMPTS = [
    "Convertí a JSON DSL: ¿Cuántas tazas aparecen?",
    "Convertí a JSON DSL: ¿La lámpara está por encima de la silla?",
    "Describí brevemente: Total de objetos: 2; cup: 1; table: 1.",
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--candidate", action="append", help="Candidato; por defecto, todos")
    parser.add_argument("--output", type=Path, default=Path("outputs/llm_benchmark.json"))
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    cfg = load_config(*args.config)
    device = resolve_device(cfg.get("device", "auto"))
    if not device.startswith("cuda"):
        raise SystemExit("Este benchmark debe ejecutarse en una GPU CUDA (objetivo: T4)")

    candidates = args.candidate or list(cfg["llm"]["candidates"])
    benchmark_cfg = cfg["llm"]["benchmark"]
    results = []
    for candidate in candidates:
        model = build_llm(cfg, device=device, candidate=candidate)
        try:
            result = benchmark_llm(
                model,
                PROMPTS,
                candidate=candidate,
                warmup=benchmark_cfg["warmup"],
                repeats=benchmark_cfg["repeats"],
            )
            results.append(result.to_dict())
        finally:
            model.close()
            gc.collect()

    import torch

    payload = {
        "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "prompts": len(PROMPTS),
        "results": results,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(args.output)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/compare_runs.py
"""Compara un run experimental contra la línea base (dev) y lo registra en experiments.csv.

Uso:
    python scripts/run_pipeline.py --config configs/base.yaml configs/kaggle.yaml \\
        configs/experiments/e1_30_pp_solo_chicas.yaml --split dev
    python scripts/compare_runs.py --config configs/base.yaml configs/kaggle.yaml \\
        --base outputs/runs/<run_base> --exp latest --name e1_30_pp_solo_chicas \\
        --hypothesis "post-proceso solo en máscaras chicas sube el F1 de máscaras" \\
        --change "segmentation.postprocess.apply_below_area_ratio: null -> 0.00333"

Las métricas se calculan en el mismo split tuning/holdout de E1-29. La decisión se toma con el
holdout. Escribe una fila en <outputs>/../experiments.csv y experiment_<name>.json en el run.
"""

from __future__ import annotations

import argparse
import json
from datetime import datetime
from pathlib import Path

from eval.annotations import annotations_path, load_annotations
from eval.experiments import append_row, dev_splits, run_metrics, stage_cost_s
from scene.config import load_config
from scene.pipeline import load_scenes
from scene.tracing.tracer import read_trace


def resolve(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = Path(cfg["paths"]["outputs"]).iterdir()
    return sorted(p for p in runs if (p / "scenes").is_dir())[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--base", required=True, help="run de la línea base")
    parser.add_argument("--exp", required=True, help="run del experimento ('latest')")
    parser.add_argument("--name", required=True)
    parser.add_argument("--hypothesis", required=True)
    parser.add_argument("--change", required=True, help="la única variable que cambia")
    parser.add_argument("--conclusion", default="", help="se puede completar después")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    base_dir, exp_dir = resolve(cfg, args.base), resolve(cfg, args.exp)
    if base_dir == exp_dir:
        raise SystemExit("--base y --exp son el mismo run")
    gt = load_annotations(annotations_path(cfg))
    splits = dev_splits(gt, cfg)
    base = run_metrics(load_scenes(base_dir), gt, cfg, splits)
    exp = run_metrics(load_scenes(exp_dir), gt, cfg, splits)
    cost_base = stage_cost_s(read_trace(base_dir / cfg["tracing"]["trace_file"]))
    cost_exp = stage_cost_s(read_trace(exp_dir / cfg["tracing"]["trace_file"]))

    row = {
        "date": datetime.now().isoformat(timespec="seconds"),
        "name": args.name,
        "hypothesis": args.hypothesis,
        "change": args.change,
        "base_run": base_dir.name,
        "exp_run": exp_dir.name,
    }
    for key in base:
        row[f"base_{key}"] = round(base[key], 4)
        row[f"exp_{key}"] = round(exp[key], 4)
    row["delta_holdout_f1_box"] = round(exp["holdout_f1_box"] - base["holdout_f1_box"], 4)
    row["delta_holdout_f1_mask"] = round(exp["holdout_f1_mask"] - base["holdout_f1_mask"], 4)
    row["delta_tuning_f1_mask"] = round(exp["tuning_f1_mask"] - base["tuning_f1_mask"], 4)
    # Costo: segundos de segmentación de cada run (la detección suele venir de caché).
    row["base_seg_s"] = round(cost_base["segmentation"], 1)
    row["exp_seg_s"] = round(cost_exp["segmentation"], 1)
    row["conclusion"] = args.conclusion

    csv_path = append_row(Path(cfg["paths"]["outputs"]).parent / "experiments.csv", row)
    (exp_dir / f"experiment_{args.name}.json").write_text(json.dumps(row, indent=2), "utf-8")
    print(f"{'métrica':<28}{'base':>9}{'exp':>9}{'Δ':>9}")
    for key in base:
        print(f"{key:<28}{base[key]:>9.4f}{exp[key]:>9.4f}{exp[key] - base[key]:>+9.4f}")
    print(f"registrado en {csv_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/eval_qa.py
"""Evaluación de QA (E1-24) sobre un run con `questions` y, si están, `questions_llm`.

Uso:
    python scripts/eval_qa.py --config configs/base.yaml configs/kaggle.yaml --run <run>

El run tiene que tener las 20 imágenes del benchmark con `run_qa.py` (y `run_qa_llm.py`) ya
corridos. Arma el SceneJSON oráculo de cada imagen desde el GT y responde con el mismo ejecutor.
Escribe en el run: metrics_qa.json y qa_eval.csv.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from eval.annotations import annotations_path, load_annotations
from eval.oracle import oracle_scene
from eval.qa_eval import evaluate_mode, summarize
from scene.config import load_config
from scene.io.dataset import list_images, load_image, load_questions, load_vocabulary
from scene.pipeline import load_scenes
from scene.qa.category_matcher import load_lexicon, matcher_from_config, spanish_names
from scene.qa.executor import executor_from_config


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--run", required=True)
    args = parser.parse_args()

    cfg = load_config(*args.config)
    run = Path(args.run)
    gt = load_annotations(annotations_path(cfg))
    categories = load_vocabulary(cfg)
    names = {c.id: c.name for c in categories}
    matcher = matcher_from_config(cfg, categories)
    executor = executor_from_config(
        cfg,
        matcher.match_id,
        names,
        display_names=spanish_names(categories, load_lexicon(cfg["qa"]["matcher"]["lexicon_es"])),
    )
    phrases = cfg["qa"]["adapter"]["relation_phrases"]
    benchmark = load_questions(cfg)
    scenes = load_scenes(run)
    recs = {r.image_id: r for r in list_images(cfg, "benchmark")}
    oracles = {iid: oracle_scene(gt[iid], load_image(rec), cfg, names) for iid, rec in recs.items()}

    frames = []
    for mode, field in (("deterministic", "questions"), ("llm", "questions_llm")):
        if any(s.get(field) for s in scenes.values()):
            frames.append(
                evaluate_mode(mode, benchmark, scenes, oracles, executor, phrases, matcher.match_id)
            )
    if not frames:
        raise SystemExit(f"{run} no tiene questions ni questions_llm: correr antes run_qa.py")
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(run / "qa_eval.csv", index=False)
    summary = summarize(df)
    (run / "metrics_qa.json").write_text(json.dumps(summary, indent=2, default=str), "utf-8")
    print(json.dumps(summary, indent=2, default=str))
    wrong = df[(df["correct"] == False) & df["type"].ne("description")]  # noqa: E712
    if len(wrong):
        cols = ["mode", "question_id", "type", "answer", "oracle_answer", "status", "error_source"]
        print(wrong[cols].to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/eval_segmentation.py
"""Segmentación con oráculo (E1-12): SAM2 con cajas GT vs. con cajas predichas.

Uso:
    python scripts/eval_segmentation.py --config configs/base.yaml configs/kaggle.yaml --run latest
    # smoke test en CPU
    python scripts/eval_segmentation.py --limit 2 \\
        --set segmentation.model_id=facebook/sam2.1-hiera-tiny

Con `--run` (un run de run_pipeline.py sobre dev) agrega la comparación con las cajas predichas.
Escribe metrics_segmentation_oracle.json y segmentation_oracle.csv en el run (o en outputs/eval/).
"""

from __future__ import annotations

import argparse
import copy
import json
from pathlib import Path

import pandas as pd

from eval.annotations import annotations_path, load_annotations
from eval.lvis_federated import evaluate_from_config, predictions_from_scenes
from eval.segmentation import oracle_rows, predicted_pairs_summary, summarize_oracle
from scene.config import load_config, resolve_device, set_seed
from scene.io.dataset import list_images, load_image
from scene.pipeline import load_scenes
from scene.segmentation.sam2 import build_segmenter


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--set", nargs="*", default=[], help="overrides clave.sub=valor")
    parser.add_argument("--run", help="run de dev con scenes/ ('latest' = el más reciente)")
    parser.add_argument("--limit", type=int, default=None)
    args = parser.parse_args()

    cfg = load_config(*args.config, dotlist=args.set)
    set_seed(cfg["seed"])
    gt = load_annotations(annotations_path(cfg))
    run_dir = None
    if args.run:
        if args.run == "latest":
            runs = Path(cfg["paths"]["outputs"]).iterdir()
            run_dir = sorted(p for p in runs if (p / "scenes").is_dir())[-1]
        else:
            run_dir = Path(args.run)

    # SAM2 sin post-proceso: la cruda se post-procesa aparte para evaluar las dos.
    raw_cfg = copy.deepcopy(cfg)
    raw_cfg["segmentation"]["postprocess"] = {
        "crop_to_box": False,
        "largest_component": False,
        "fill_holes": False,
        "box_margin": 0.0,
        "max_hole_ratio": 0.0,
    }
    segmenter = build_segmenter(raw_cfg, resolve_device(cfg.get("device", "auto")))
    pp_cfg = cfg["segmentation"]["postprocess"]

    recs = [r for r in list_images(cfg, "dev") if r.image_id in gt][: args.limit]
    small_max = cfg["scene"]["size_thresholds"]["small_max"]
    rows = []
    for k, rec in enumerate(recs, 1):
        rows += oracle_rows(gt[rec.image_id], segmenter, load_image(rec), pp_cfg, small_max)
        print(f"{k}/{len(recs)} {rec.image_id}", flush=True)

    out = {
        "model_id": cfg["segmentation"]["model_id"],
        "postprocess": pp_cfg,
        "oracle_gt_boxes": summarize_oracle(rows, cfg),
    }
    if run_dir is not None:
        scenes = load_scenes(run_dir)
        ids = [r.image_id for r in recs if r.image_id in scenes]
        preds = predictions_from_scenes({i: scenes[i] for i in ids})
        box = evaluate_from_config(preds, gt, cfg, iou_type="bbox", image_ids=ids)
        segm = evaluate_from_config(preds, gt, cfg, iou_type="segm", image_ids=ids)
        out["predicted_boxes"] = {"run": str(run_dir), **predicted_pairs_summary(box, segm)}

    dest = run_dir or Path(cfg["paths"]["outputs"]).parent / "eval"
    dest.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(dest / "segmentation_oracle.csv", index=False)
    (dest / "metrics_segmentation_oracle.json").write_text(json.dumps(out, indent=2), "utf-8")
    print(json.dumps(out, indent=2))
    print(f"salidas en {dest}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/make_submission.py
"""Arma `submission.csv` a partir de los SceneJSON de uno o más runs y lo valida completo.

Uso:
    python scripts/make_submission.py --run outputs/runs/<run_id>
    python scripts/make_submission.py --config configs/base.yaml configs/kaggle.yaml \\
        --run latest --out /kaggle/working/submission.csv

Si una imagen está en varios runs, gana el último de la lista. Falla si falta alguna de las 80
filas o si algún SceneJSON no valida (no escribe un submission inválido).
"""

from __future__ import annotations

import argparse
from pathlib import Path

from scene.config import load_config
from scene.io.submission import build_submission, read_submission, write_submission
from scene.pipeline import load_scenes


def resolve_run(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = sorted(p for p in Path(cfg["paths"]["outputs"]).iterdir() if (p / "scenes").is_dir())
    if not runs:
        raise SystemExit("No hay runs con scenes/: correr antes scripts/run_pipeline.py")
    return runs[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--run", nargs="+", required=True, help="runs con scenes/ o 'latest'")
    parser.add_argument("--out", help="ruta del CSV (por defecto, dentro del último run)")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    runs = [resolve_run(cfg, r) for r in args.run]
    scenes: dict = {}
    for run in runs:
        scenes.update(load_scenes(run))
    rows, errors = build_submission(scenes, cfg)
    if errors:
        for iid, errs in list(errors.items())[:10]:
            print(f"{iid}: {errs[:3]}")
        raise SystemExit(f"{len(errors)} filas con error: no se escribe el submission")

    out = Path(args.out) if args.out else runs[-1] / cfg["paths"]["submission"]
    write_submission(rows, out)
    back = read_submission(out)  # chequeo de ida y vuelta del CSV
    assert list(back) == [iid for iid, _ in rows]
    n_obj = sum(len(s["objects"]) for s in back.values())
    n_rel = sum(len(s["relations"]) for s in back.values())
    size_mb = out.stat().st_size / 2**20
    print(f"{out}: {len(rows)} filas, {n_obj} objetos, {n_rel} relaciones, {size_mb:.1f} MB")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_eda.py
"""EDA del dataset de desarrollo: tablas (CSV), figuras (PNG) y resumen (JSON).

Uso: python scripts/run_eda.py --config configs/base.yaml [--out outputs/eda]
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from eda import plots
from eda.stats import (
    benchmark_table,
    categories_table,
    images_table,
    instances_table,
    overlap_table,
)
from eval.annotations import annotations_path, load_annotations
from scene.config import load_config
from scene.io.dataset import load_questions, load_vocabulary


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--out", default="outputs/eda")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    out = Path(args.out)
    (out / "figures").mkdir(parents=True, exist_ok=True)

    gt = load_annotations(annotations_path(cfg))
    vocab = load_vocabulary(cfg)
    names = {c.id: c.name for c in vocab}
    sizes = cfg["scene"]["size_thresholds"]
    freq = cfg["eval"]["frequency_bins"]

    instances = instances_table(gt)
    images = images_table(gt, len(vocab))
    categories = categories_table(gt, names)
    overlaps = overlap_table(gt, cfg["eval"]["eda_duplicate_iou"])
    bench = benchmark_table(load_questions(cfg), gt, {v: k for k, v in names.items()})
    tables = {
        "instances": instances,
        "images": images,
        "categories": categories,
        "overlaps": overlaps,
        "benchmark": bench,
    }
    for name, df in tables.items():
        df.to_csv(out / f"eda_{name}.csv", index=False)

    figures = {
        "instances_per_image": plots.instances_per_image(
            images, cfg["detection"]["max_detections"]
        ),
        "area_ratio": plots.area_ratio(instances, sizes["small_max"], sizes["medium_max"]),
        "category_frequency": plots.category_frequency(
            categories, freq["rare_max"], freq["common_max"]
        ),
        "federated_status": plots.federated_status(images),
        "occlusion": plots.occlusion(overlaps),
        "position_grid": plots.position_grid(instances),
        "benchmark_types": plots.benchmark_types(bench),
    }
    for name, fig in figures.items():
        fig.savefig(out / "figures" / f"{name}.png", dpi=130, bbox_inches="tight")

    a = instances["area_ratio"]
    n_img = categories["n_images_positive"]
    summary = {
        "n_images": len(images),
        "n_instances": len(instances),
        "instances_per_image": images["n_instances"].describe().round(2).to_dict(),
        "area_ratio_quantiles": a.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(5).to_dict(),
        "share_area_lt_0.1pct": round(float((a < 0.001).mean()), 3),
        "share_area_lt_1pct": round(float((a < 0.01).mean()), 3),
        "size_split": {
            "small": round(float((a < sizes["small_max"]).mean()), 3),
            "medium": round(
                float(((a >= sizes["small_max"]) & (a < sizes["medium_max"])).mean()), 3
            ),
            "large": round(float((a >= sizes["medium_max"]).mean()), 3),
        },
        "category_images_range": [int(n_img.min()), int(n_img.max())],
        "frequency_split": {
            "rare": int((n_img <= freq["rare_max"]).sum()),
            "common": int(((n_img > freq["rare_max"]) & (n_img <= freq["common_max"])).sum()),
            "frequent": int((n_img > freq["common_max"]).sum()),
        },
        "federated_per_image_mean": images[
            ["n_positive", "n_negative", "n_not_exhaustive", "n_unverified"]
        ]
        .mean()
        .round(2)
        .to_dict(),
        "share_instances_overlapping": round(float((overlaps["max_covered"] > 0).mean()), 3),
        "share_instances_covered_gt_50pct": round(float((overlaps["max_covered"] > 0.5).mean()), 3),
        "share_same_class_duplicates": round(float(overlaps["same_class_dup"].mean()), 3),
        "position_share": instances["position"].value_counts(normalize=True).round(3).to_dict(),
        "benchmark_types": bench["type"].value_counts().to_dict(),
    }
    (out / "eda_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_eval.py
"""Evaluación de un run de `run_pipeline.py`: F1 de cajas y máscaras, consistencia del SceneJSON
y exactitud de posición, tamaño y color contra GT (solo imágenes de dev).

Uso:
    python scripts/run_eval.py --run outputs/runs/<run_id>
    python scripts/run_eval.py --config configs/base.yaml configs/kaggle.yaml --run latest

Escribe en el run: metrics_detection.json, metrics_segmentation.json (+ CSV por categoría) y
metrics_scene.json. La consistencia interna se calcula para todas las escenas (también test).
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from eval.annotations import annotations_path, load_annotations
from eval.lvis_federated import evaluate_from_config, predictions_from_scenes, save_metrics
from eval.scene_consistency import attribute_accuracy, internal_consistency, save_scene_metrics
from scene.config import load_config
from scene.io.dataset import list_images, load_image, load_vocabulary
from scene.pipeline import load_scenes


def resolve_run(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = sorted(p for p in Path(cfg["paths"]["outputs"]).iterdir() if (p / "scenes").is_dir())
    if not runs:
        raise SystemExit("No hay runs con scenes/: correr antes scripts/run_pipeline.py")
    return runs[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--run", default="latest")
    parser.add_argument("--no-color", action="store_true", help="no medir color (más rápido)")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    run_dir = resolve_run(cfg, args.run)
    scenes = load_scenes(run_dir)
    out: dict = {"run": str(run_dir), "consistency": internal_consistency(scenes, cfg)}

    gt = load_annotations(annotations_path(cfg))
    dev_ids = [i for i in scenes if i in gt]
    if dev_ids:
        names = {c.id: c.name for c in load_vocabulary(cfg)}
        preds = predictions_from_scenes({i: scenes[i] for i in dev_ids})
        for iou_type, name in (("bbox", "detection"), ("segm", "segmentation")):
            res = evaluate_from_config(preds, gt, cfg, iou_type=iou_type, image_ids=dev_ids)
            save_metrics(res, run_dir, name, names)
            out[name] = {
                k: res.summary[k] for k in ("precision", "recall", "f1", "evaluable_coverage")
            }
            out[name]["by_size_f1"] = {s: v["f1"] for s, v in res.by_size.items()}
        images = None
        if not args.no_color:
            recs = {r.image_id: r for r in list_images(cfg, "dev")}
            images = {i: load_image(recs[i]) for i in dev_ids}
        out["attributes"] = attribute_accuracy(scenes, gt, cfg, images)

    save_scene_metrics({k: out[k] for k in ("consistency", "attributes") if k in out}, run_dir)
    printable = {**out, "consistency": {**out["consistency"]}}
    printable["consistency"].pop("scenes_with_issues")
    print(json.dumps(printable, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_pipeline.py
"""Pipeline completa (detección → SAM2 → SceneJSON) sobre uno o más splits o imágenes sueltas.

Uso:
    # imagen cualquiera (fuera del dataset), en CPU con el SAM2 chico
    python scripts/run_pipeline.py --images foto.jpg --debug \\
        --set segmentation.model_id=facebook/sam2.1-hiera-tiny
    # smoke test
    python scripts/run_pipeline.py --split dev --limit 3 --debug
    # dev completo en Kaggle + F1 de cajas y máscaras
    python scripts/run_pipeline.py --config configs/base.yaml configs/kaggle.yaml --split dev --eval
    # las 80 imágenes del submission (después: scripts/make_submission.py --run <run>)
    python scripts/run_pipeline.py --config configs/base.yaml configs/kaggle.yaml \\
        --split test benchmark

Salidas en outputs/runs/<run_id>/: scenes/ (SceneJSON por imagen), debug/, trace.jsonl,
run_manifest.json y, con --eval, metrics_detection.json y metrics_segmentation.json.
La caché de detecciones crudas y máscaras queda en paths.cache y se reutiliza entre runs.
"""

from __future__ import annotations

import argparse
import json

from scene.config import load_config, start_run
from scene.detection.owlv2 import build_detector
from scene.detection.vocabulary import vocabulary_from_config
from scene.io.dataset import image_record_from_path, list_images, load_vocabulary
from scene.pipeline import STAGES, Pipeline
from scene.segmentation.sam2 import build_segmenter
from scene.tracing.tracer import Tracer, read_trace


def evaluate(cfg: dict, run_dir, scenes: dict, image_ids: list[str], names: dict) -> dict:
    """F1 federado de cajas y máscaras desde los SceneJSON (GT de dev, fuera de la inferencia)."""
    from eval.annotations import annotations_path, load_annotations
    from eval.lvis_federated import evaluate_from_config, predictions_from_scenes, save_metrics

    gt = load_annotations(annotations_path(cfg))
    preds = predictions_from_scenes(scenes)
    out = {}
    for iou_type, name in (("bbox", "detection"), ("segm", "segmentation")):
        result = evaluate_from_config(preds, gt, cfg, iou_type=iou_type, image_ids=image_ids)
        save_metrics(result, run_dir, name, names)
        out[name] = {k: result.summary[k] for k in ("precision", "recall", "f1", "tp", "fp", "fn")}
    return out


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--set", nargs="*", default=[], help="overrides clave.sub=valor")
    parser.add_argument("--split", nargs="+", default=["dev"], choices=["dev", "test", "benchmark"])
    parser.add_argument("--images", nargs="*", help="imágenes sueltas (ignora --split)")
    parser.add_argument("--limit", type=int, default=None, help="tope de imágenes por split")
    parser.add_argument("--from-stage", choices=STAGES, help="ignora la caché desde esta etapa")
    parser.add_argument("--debug", action="store_true", help="PNG con cajas y máscaras")
    parser.add_argument("--eval", action="store_true", help="F1 de cajas y máscaras (solo dev)")
    args = parser.parse_args()

    cfg = load_config(*args.config, dotlist=args.set)
    ctx = start_run(cfg)
    tracer = Tracer(ctx.trace_path, run_id=ctx.run_id, config_hash=ctx.config_hash)
    vocab = vocabulary_from_config(cfg, load_vocabulary(cfg))

    if args.images:
        images = [image_record_from_path(p) for p in args.images]
    else:
        seen, images = set(), []
        for split in args.split:  # benchmark ⊂ dev: no repetir imágenes
            for rec in list_images(cfg, split)[: args.limit]:
                if rec.image_id not in seen:
                    seen.add(rec.image_id)
                    images.append(rec)

    pipe = Pipeline(
        cfg,
        ctx,
        tracer,
        vocab,
        make_detector=lambda: build_detector(cfg, vocab, ctx.device),
        make_segmenter=lambda: build_segmenter(cfg, ctx.device),
        from_stage=args.from_stage,
        debug=args.debug,
    )
    scenes = pipe.run(images)
    ctx.update_manifest("images", [r.image_id for r in images])

    stages: dict[str, dict[str, int]] = {}
    for r in read_trace(ctx.trace_path):
        stages.setdefault(r["stage"], {}).setdefault(r["status"], 0)
        stages[r["stage"]][r["status"]] += 1
    print(f"run {ctx.run_id}: {len(scenes)} escenas, {len(tracer.failed_images)} con error")
    print(json.dumps(stages))
    if tracer.failed_images:
        print("con error:", tracer.failed_images)

    if args.eval:
        if args.images or args.split != ["dev"]:
            raise SystemExit("--eval solo tiene GT para el split dev")
        names = {c.id: c.name for c in vocab.categories}
        summary = evaluate(cfg, ctx.run_dir, scenes, [r.image_id for r in images], names)
        print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_qa.py
"""QA determinista (E1-23): agrega `questions` a los SceneJSON de las imágenes del benchmark.

Uso:
    python scripts/run_qa.py --run outputs/runs/<run_id>
    python scripts/run_qa.py --config configs/base.yaml configs/kaggle.yaml --run latest

Reescribe `scenes/<image_id>.json` del run (reemplaza `questions` si ya estaba), traza cada
imagen en `trace.jsonl` (etapa `qa_deterministic`) y guarda `qa_deterministic.csv`. Después,
`make_submission.py` toma las escenas con sus respuestas.
"""

from __future__ import annotations

import argparse
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from scene.config import config_hash, data_path, load_config
from scene.io.dataset import load_questions, load_vocabulary
from scene.pipeline import load_scenes
from scene.qa.category_matcher import load_lexicon, matcher_from_config, spanish_names
from scene.qa.executor import executor_from_config
from scene.qa.structured_adapter import answer_questions, group_by_image
from scene.scene.schema import assert_valid_scene
from scene.tracing.tracer import Tracer


def resolve_run(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = sorted(p for p in Path(cfg["paths"]["outputs"]).iterdir() if (p / "scenes").is_dir())
    if not runs:
        raise SystemExit("No hay runs con scenes/: correr antes scripts/run_pipeline.py")
    return runs[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--run", default="latest")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    run = resolve_run(cfg, args.run)
    categories = load_vocabulary(cfg)
    matcher = matcher_from_config(cfg, categories)
    lexicon = load_lexicon(cfg["qa"]["matcher"]["lexicon_es"])
    executor = executor_from_config(
        cfg,
        matcher.match_id,
        {c.id: c.name for c in categories},
        display_names=spanish_names(categories, lexicon),
    )
    phrases = cfg["qa"]["adapter"]["relation_phrases"]
    scenes = load_scenes(run)
    tracer = Tracer(run / cfg["tracing"]["trace_file"], run.name, config_hash(cfg))
    schema = data_path(cfg, "scene_schema")

    rows, missing = [], []
    for iid, questions in group_by_image(load_questions(cfg)).items():
        if iid not in scenes:
            missing.append(iid)
            continue
        with (
            tracer.image_guard(iid),
            tracer.stage(iid, "qa_deterministic", n_in=len(questions)) as s,
        ):
            scene = scenes[iid]
            scene["questions"] = answer_questions(questions, scene, executor, phrases)
            assert_valid_scene(scene, schema)
            path = run / "scenes" / f"{iid}.json"
            path.write_text(json.dumps(scene, ensure_ascii=False), encoding="utf-8")
            s.n_out = len(scene["questions"])
            s.artifact_path = str(path)
            s.extra["status"] = dict(Counter(q["status"] for q in scene["questions"]))
            for q in scene["questions"]:
                rows.append(
                    {
                        "image_id": iid,
                        "question_id": q["question_id"],
                        "type": q["type"],
                        "status": q["status"],
                        "normalized_answer": q["normalized_answer"],
                        "confidence": q["confidence"],
                        "n_evidence": len(q["evidence"]),
                        "detail": q.get("status_detail"),
                    }
                )

    df = pd.DataFrame(rows)
    df.to_csv(run / "qa_deterministic.csv", index=False)
    print(f"{run}: {len(df)} respuestas en {df.image_id.nunique() if len(df) else 0} imágenes")
    if len(df):
        print(df.groupby(["type", "status"]).size().to_string())
    if missing or tracer.failed_images:
        raise SystemExit(f"Sin escena: {missing}; con error: {tracer.failed_images}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_qa_llm.py
"""QA con LLM (E1-28): agrega `questions_llm` a los SceneJSON de las imágenes del benchmark.

Uso (en Kaggle, con GPU):
    pip install -e ".[llm]"
    python scripts/run_qa_llm.py --config configs/base.yaml configs/kaggle.yaml --run <run>

El LLM recibe solo el texto de cada pregunta y produce el JSON intermedio (DSL); lo ejecuta el
mismo motor determinista de `run_qa.py`. Reescribe `scenes/<image_id>.json` del run (reemplaza
`questions_llm`), traza cada imagen (etapa `qa_llm`) y guarda `qa_llm.csv`. Después se corre
`make_submission.py`.
"""

from __future__ import annotations

import argparse
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from scene.config import config_hash, data_path, load_config, resolve_device, set_seed
from scene.io.dataset import load_questions, load_vocabulary
from scene.llm.describer import build_describer
from scene.llm.interpreter import build_interpreter
from scene.llm.model import build_llm
from scene.pipeline import load_scenes
from scene.qa.category_matcher import load_lexicon, matcher_from_config, spanish_names
from scene.qa.executor import executor_from_config
from scene.qa.llm_mode import answer_questions_llm
from scene.qa.structured_adapter import group_by_image
from scene.scene.schema import assert_valid_scene
from scene.tracing.tracer import Tracer


def resolve_run(cfg: dict, run: str) -> Path:
    if run != "latest":
        return Path(run)
    runs = sorted(p for p in Path(cfg["paths"]["outputs"]).iterdir() if (p / "scenes").is_dir())
    if not runs:
        raise SystemExit("No hay runs con scenes/: correr antes scripts/run_pipeline.py")
    return runs[-1]


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--set", nargs="*", default=[], help="overrides clave.sub=valor")
    parser.add_argument("--run", default="latest")
    args = parser.parse_args()

    cfg = load_config(*args.config, dotlist=args.set)
    set_seed(cfg["seed"])
    device = resolve_device(cfg.get("device", "auto"))
    run = resolve_run(cfg, args.run)
    categories = load_vocabulary(cfg)
    names = {c.id: c.name for c in categories}
    matcher = matcher_from_config(cfg, categories, device)
    lexicon = load_lexicon(cfg["qa"]["matcher"]["lexicon_es"])
    executor = executor_from_config(
        cfg, matcher.match_id, names, display_names=spanish_names(categories, lexicon)
    )
    llm = build_llm(cfg, device=device)
    interpreter = build_interpreter(
        cfg, llm, category_matcher=matcher.match_id, category_names=names
    )
    describer = build_describer(cfg, llm, category_names=names)

    scenes = load_scenes(run)
    tracer = Tracer(run / cfg["tracing"]["trace_file"], run.name, config_hash(cfg))
    schema = data_path(cfg, "scene_schema")
    candidate = cfg["llm"]["candidates"][cfg["llm"]["selected"]]

    rows, missing = [], []
    for iid, questions in group_by_image(load_questions(cfg)).items():
        if iid not in scenes:
            missing.append(iid)
            continue
        with (
            tracer.image_guard(iid),
            tracer.stage(
                iid,
                "qa_llm",
                params=cfg["llm"],
                model_id=candidate["model_id"],
                n_in=len(questions),
            ) as s,
        ):
            scene = scenes[iid]
            scene["questions_llm"] = answer_questions_llm(
                questions, scene, interpreter, executor, describer, names
            )
            assert_valid_scene(scene, schema)
            path = run / "scenes" / f"{iid}.json"
            path.write_text(json.dumps(scene, ensure_ascii=False), encoding="utf-8")
            s.n_out = len(scene["questions_llm"])
            s.artifact_path = str(path)
            s.extra["status"] = dict(Counter(q["status"] for q in scene["questions_llm"]))
            s.extra["interpreter"] = dict(
                Counter(q["interpreter"]["status"] for q in scene["questions_llm"])
            )
            for q in scene["questions_llm"]:
                rows.append(
                    {
                        "image_id": iid,
                        "question_id": q["question_id"],
                        "type": q["type"],
                        "interpreter_status": q["interpreter"]["status"],
                        "attempts": q["interpreter"]["attempts"],
                        "op": q["interpreted_query"]["op"],
                        "status": q["status"],
                        "normalized_answer": q["normalized_answer"],
                        "confidence": q["confidence"],
                        "describer_status": q["interpreter"].get("describer_status"),
                    }
                )

    df = pd.DataFrame(rows)
    df.to_csv(run / "qa_llm.csv", index=False)
    print(f"{run}: {len(df)} respuestas LLM en {df.image_id.nunique() if len(df) else 0} imágenes")
    if len(df):
        print(df.groupby(["type", "interpreter_status", "status"]).size().to_string())
    if missing or tracer.failed_images:
        raise SystemExit(f"Sin escena: {missing}; con error: {tracer.failed_images}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/tune_thresholds.py
"""Barrido del umbral de detección (y NMS entre clases) sobre la caché de detecciones crudas.

Uso (después de correr la pipeline sobre dev, así la caché tiene las 120 imágenes):
    python scripts/tune_thresholds.py --config configs/base.yaml configs/kaggle.yaml
    python scripts/tune_thresholds.py --raw-dir outputs/runs/<run_id>/detections_raw

Elige el umbral de máximo F1 en tuning (80 imágenes) y lo reporta en holdout (40), junto con
el umbral actual de la config como referencia. No cambia `base.yaml`: el valor elegido se
copia a mano después de revisar la curva.

Salidas en outputs/tuning/<config_hash>/: curve.csv, curve.png, split.json y summary.json.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

from eval.annotations import annotations_path, load_annotations  # noqa: E402
from eval.tuning import choose, split_tuning_holdout, sweep  # noqa: E402
from scene.config import config_hash, load_config  # noqa: E402
from scene.detection.base import load_detections  # noqa: E402
from scene.pipeline import stage_keys  # noqa: E402


def parse_range(text: str) -> list[float]:
    start, stop, step = (float(x) for x in text.split(":"))
    return [round(v, 4) for v in np.arange(start, stop + step / 2, step)]


def plot(curve, out: Path, chosen: dict[str, float]) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for (variant, split), g in curve.groupby(["variant", "split"]):
        style = "-" if split == "tuning" else "--"
        ax.plot(g["threshold"], g["f1"], style, label=f"{variant} · {split}")
    for variant, thr in chosen.items():
        ax.axvline(thr, color="gray", lw=0.8, ls=":")
        ax.annotate(f"{variant}: {thr:.2f}", (thr, ax.get_ylim()[0]), fontsize=8, rotation=90)
    ax.set_xlabel("umbral de score")
    ax.set_ylabel("F1 de cajas (federado, IoU ≥ 0,5)")
    ax.set_title("Ajuste de umbral: tuning (—) y holdout (--)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out, dpi=120)
    plt.close(fig)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", nargs="+", default=["configs/base.yaml"])
    parser.add_argument("--raw-dir", help="carpeta con <image_id>.json crudos (por defecto, caché)")
    parser.add_argument("--thresholds", default="0.02:0.5:0.02", help="inicio:fin:paso")
    parser.add_argument("--holdout", type=int, default=40)
    parser.add_argument("--out", default="outputs/tuning")
    args = parser.parse_args()

    cfg = load_config(*args.config)
    raw_dir = (
        Path(args.raw_dir)
        if args.raw_dir
        else Path(cfg["paths"]["cache"]) / "detections_raw" / stage_keys(cfg)["detection"]
    )
    gt = load_annotations(annotations_path(cfg))
    missing = [i for i in gt if not (raw_dir / f"{i}.json").is_file()]
    if missing:
        raise SystemExit(
            f"Faltan {len(missing)} imágenes de dev en {raw_dir} (por ejemplo {missing[:3]}): "
            "correr antes scripts/run_pipeline.py --split dev"
        )
    raw = {i: load_detections(raw_dir / f"{i}.json") for i in gt}
    floor = cfg["detection"]["owlv2"]["score_floor"]
    thresholds = [t for t in parse_range(args.thresholds) if t >= floor]

    tuning, holdout = split_tuning_holdout(gt, sorted(gt), args.holdout, cfg["seed"])
    splits = {"tuning": tuning, "holdout": holdout}
    cross_iou = cfg["detection"]["cross_class_nms"]["iou"]
    variants = {
        "base": {"cross_class_nms": {"enabled": False, "iou": cross_iou}},
        f"cross_nms_{cross_iou}": {"cross_class_nms": {"enabled": True, "iou": cross_iou}},
    }
    curve = sweep(raw, gt, splits, cfg, thresholds, variants)

    out = Path(args.out) / config_hash(cfg)[:8]
    out.mkdir(parents=True, exist_ok=True)
    curve.to_csv(out / "curve.csv", index=False)
    (out / "split.json").write_text(json.dumps(splits, indent=1), encoding="utf-8")

    current = cfg["detection"]["score_threshold"]
    summary: dict = {"raw_dir": str(raw_dir), "current_threshold": current, "variants": {}}
    for variant in variants:
        best = choose(curve, variant)
        hold = curve[
            (curve["variant"] == variant)
            & (curve["split"] == "holdout")
            & np.isclose(curve["threshold"], best["threshold"])
        ].iloc[0]
        summary["variants"][variant] = {
            "chosen_threshold": float(best["threshold"]),
            "tuning": best[["precision", "recall", "f1", "preds_per_image"]].to_dict(),
            "holdout": hold[["precision", "recall", "f1", "preds_per_image"]].to_dict(),
        }
    ref = curve[
        (curve["variant"] == "base")
        & (curve["split"] == "holdout")
        & np.isclose(curve["threshold"], current)
    ]
    if len(ref):
        summary["holdout_at_current_threshold"] = ref.iloc[0][
            ["precision", "recall", "f1"]
        ].to_dict()
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    plot(
        curve, out / "curve.png", {v: s["chosen_threshold"] for v, s in summary["variants"].items()}
    )
    print(json.dumps(summary, indent=2))
    print(f"salidas en {out}")


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path

_ = Path('src/eda/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/eda/plots.py
"""Gráficos del EDA. Cada función responde una pregunta y devuelve la figura."""

from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

from scene.scene.schema import POSITIONS  # noqa: E402

# Paleta: un solo tono para magnitudes, tinta de texto neutra y grilla recesiva.
SERIES = "#2a78d6"
ACCENT = "#eb6834"  # solo para marcar umbrales/decisiones
SURFACE = "#fcfcfb"
TEXT = "#0b0b0b"
TEXT_2 = "#52514e"
GRID = "#e4e3df"
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#eaf2fc", "#2a78d6", "#0d3a73"])


def _style(ax: plt.Axes, title: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(title, loc="left", color=TEXT, fontsize=11)
    ax.set_xlabel(xlabel, color=TEXT_2)
    ax.set_ylabel(ylabel, color=TEXT_2)
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=TEXT_2)


def _fig(width: float = 7, height: float = 3.6) -> tuple[plt.Figure, plt.Axes]:
    fig, ax = plt.subplots(figsize=(width, height), facecolor=SURFACE)
    return fig, ax


def _vline(ax: plt.Axes, x: float, label: str) -> None:
    ax.axvline(x, color=ACCENT, linewidth=1.5, linestyle="--")
    ax.annotate(
        label, (x, 1), xycoords=("data", "axes fraction"), xytext=(4, -12),
        textcoords="offset points", color=TEXT_2, fontsize=8,
    )  # fmt: skip


def instances_per_image(images: pd.DataFrame, max_detections: int) -> plt.Figure:
    """¿Cuántos objetos hay por imagen? → tope de detecciones."""
    fig, ax = _fig()
    ax.hist(images["n_instances"], bins=30, color=SERIES, edgecolor=SURFACE, linewidth=2)
    _vline(ax, float(images["n_instances"].median()), "mediana")
    _style(ax, "Instancias anotadas por imagen", "instancias", "imágenes")
    ax.text(0.98, 0.9, f"máx = {images['n_instances'].max()} · tope propuesto = {max_detections}",
            transform=ax.transAxes, ha="right", color=TEXT_2, fontsize=9)  # fmt: skip
    return fig


def area_ratio(instances: pd.DataFrame, small_max: float, medium_max: float) -> plt.Figure:
    """¿Qué tan chicos son los objetos? → umbrales de tamaño y necesidad de tiling."""
    fig, ax = _fig()
    vals = np.log10(instances["area_ratio"].clip(lower=1e-6))
    ax.hist(vals, bins=40, color=SERIES, edgecolor=SURFACE, linewidth=2)
    _vline(ax, np.log10(small_max), f"small < {small_max:.2%}")
    _vline(ax, np.log10(medium_max), f"medium < {medium_max:.0%}")
    ax.set_xticks([-5, -4, -3, -2, -1, 0], ["0.001%", "0.01%", "0.1%", "1%", "10%", "100%"])
    _style(ax, "Área relativa de los objetos (escala log)", "área / área de la imagen", "objetos")
    return fig


def category_frequency(categories: pd.DataFrame, rare_max: int, common_max: int) -> plt.Figure:
    """¿Hay categorías raras? → bins de frecuencia para el desglose de métricas."""
    fig, ax = _fig()
    counts = categories["n_images_positive"].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=SERIES, width=0.8)
    for x, y in counts.items():
        ax.text(x, y + 0.4, str(y), ha="center", color=TEXT_2, fontsize=8)
    ax.axvline(rare_max + 0.5, color=ACCENT, linestyle="--", linewidth=1.5)
    ax.axvline(common_max + 0.5, color=ACCENT, linestyle="--", linewidth=1.5)
    _style(ax, "Imágenes en las que aparece cada categoría", "imágenes", "categorías")
    return fig


def federated_status(images: pd.DataFrame) -> plt.Figure:
    """¿Cuántas categorías por imagen son evaluables? → cuánto pesan los FP."""
    fig, ax = _fig()
    cols = {
        "n_positive": "positivas",
        "n_negative": "negativas",
        "n_not_exhaustive": "no exhaustivas\n(subconj. de positivas)",
        "n_unverified": "no verificadas",
    }
    means = images[list(cols)].mean()
    ax.bar(list(cols.values()), means.values, color=SERIES, width=0.6)
    for i, v in enumerate(means.values):
        ax.text(i, v + 1.5, f"{v:.1f}", ha="center", color=TEXT, fontsize=9)
    _style(ax, "Estado de las 120 categorías por imagen (promedio)", "", "categorías por imagen")
    return fig


def occlusion(overlaps: pd.DataFrame) -> plt.Figure:
    """¿Cuánto se superponen las instancias? → post-proceso de máscaras y NMS."""
    fig, ax = _fig()
    ax.hist(overlaps["max_covered"], bins=20, range=(0, 1), color=SERIES, edgecolor=SURFACE,
            linewidth=2)  # fmt: skip
    _style(ax, "Fracción máxima de la máscara compartida con otra instancia",
           "fracción cubierta", "instancias")  # fmt: skip
    return fig


def position_grid(instances: pd.DataFrame) -> plt.Figure:
    """¿Dónde caen los centroides? → sanidad de la grilla 3×3."""
    share = instances["position"].value_counts(normalize=True).reindex(POSITIONS, fill_value=0)
    grid = share.to_numpy().reshape(3, 3)
    fig, ax = plt.subplots(figsize=(4.2, 3.8), facecolor=SURFACE)
    ax.imshow(grid, cmap=SEQUENTIAL, vmin=0, vmax=grid.max())
    for k, name in enumerate(POSITIONS):
        r, c = divmod(k, 3)
        color = SURFACE if grid[r, c] > 0.6 * grid.max() else TEXT
        ax.text(c, r, f"{name}\n{grid[r, c]:.1%}", ha="center", va="center", color=color,
                fontsize=8)  # fmt: skip
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Centroides por celda 3×3", loc="left", color=TEXT, fontsize=11)
    return fig


def benchmark_types(bench: pd.DataFrame) -> plt.Figure:
    """¿Qué se pregunta en el benchmark? → prioridades de QA."""
    fig, ax = _fig()
    counts = bench["type"].value_counts()
    ax.bar(counts.index, counts.values, color=SERIES, width=0.6)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.3, str(v), ha="center", color=TEXT, fontsize=9)
    _style(ax, "Consignas del benchmark por tipo", "", "consignas")
    return fig


In [ ]:
%%writefile src/eda/stats.py
"""Tablas del EDA calculadas desde el GT (solo análisis, nunca inferencia)."""

from __future__ import annotations

import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils

from eval.annotations import ImageGT
from scene.scene.schema import POSITIONS


def grid_position(cx: float, cy: float, width: int, height: int) -> str:
    """Celda 3×3 (nombre oficial del schema) en la que cae un punto."""
    col = min(int(3 * cx / width), 2)
    row = min(int(3 * cy / height), 2)
    return POSITIONS[3 * row + col]


def instances_table(gt: dict[str, ImageGT]) -> pd.DataFrame:
    """Una fila por instancia: área relativa, caja, centroide de la máscara y celda 3×3."""
    rows = []
    for img in gt.values():
        img_area = img.width * img.height
        for inst in img.instances:
            mask = img.mask(inst)
            ys, xs = np.nonzero(mask)
            cx, cy = (xs.mean(), ys.mean()) if len(xs) else (np.nan, np.nan)
            x0, y0, x1, y1 = inst.bbox_xyxy
            rows.append(
                {
                    "image_id": img.image_id,
                    "instance_id": inst.id,
                    "category_id": inst.category_id,
                    "area": inst.area,
                    "area_ratio": inst.area / img_area,
                    "mask_area_ratio": mask.sum() / img_area,
                    "bbox_w": x1 - x0,
                    "bbox_h": y1 - y0,
                    "centroid_x": cx,
                    "centroid_y": cy,
                    "position": grid_position(cx, cy, img.width, img.height) if len(xs) else None,
                }
            )
    return pd.DataFrame(rows)


def images_table(gt: dict[str, ImageGT], n_categories: int) -> pd.DataFrame:
    """Una fila por imagen: instancias y estado federado de las categorías."""
    rows = []
    for img in gt.values():
        pos = img.positive_category_ids
        rows.append(
            {
                "image_id": img.image_id,
                "width": img.width,
                "height": img.height,
                "n_instances": len(img.instances),
                "n_positive": len(pos),
                "n_negative": len(img.neg_category_ids),
                "n_not_exhaustive": len(img.not_exhaustive_category_ids),
                # Ni positivas ni negativas: sus predicciones se ignoran en la evaluación.
                "n_unverified": n_categories - len(pos | img.neg_category_ids),
            }
        )
    return pd.DataFrame(rows)


def categories_table(gt: dict[str, ImageGT], names: dict[int, str]) -> pd.DataFrame:
    """Una fila por categoría: instancias e imágenes en las que es positiva o negativa."""
    rows = {cid: {"category_id": cid, "category_name": n} for cid, n in names.items()}
    for row in rows.values():
        row.update(n_instances=0, n_images_positive=0, n_images_negative=0, n_images_not_exh=0)
    for img in gt.values():
        for inst in img.instances:
            rows[inst.category_id]["n_instances"] += 1
        for cid in img.positive_category_ids:
            rows[cid]["n_images_positive"] += 1
        for cid in img.neg_category_ids:
            rows[cid]["n_images_negative"] += 1
        for cid in img.not_exhaustive_category_ids:
            rows[cid]["n_images_not_exh"] += 1
    return pd.DataFrame(rows.values()).sort_values("n_instances", ascending=False)


def overlap_table(gt: dict[str, ImageGT], iou_thr: float) -> pd.DataFrame:
    """Por instancia: cuánto la tapa otra instancia y si hay otra de su clase con IoU de caja alto.

    `max_covered` = máxima fracción de su máscara compartida con otra instancia (oclusión).
    `same_class_box_iou` = máximo IoU de caja con otra instancia de la misma categoría.
    """
    rows = []
    for img in gt.values():
        insts = img.instances
        n = len(insts)
        if n == 0:
            continue
        rles = []
        for inst in insts:
            r = mask_utils.frPyObjects(inst.segmentation, img.height, img.width)
            rles.append(mask_utils.merge(r) if isinstance(r, list) else r)
        # Con iscrowd=1, pycocotools devuelve intersección / área de la predicción (fila).
        covered = np.asarray(mask_utils.iou(rles, rles, [1] * n)).reshape(n, n)
        boxes = [[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in (i.bbox_xyxy for i in insts)]
        box_iou = np.asarray(mask_utils.iou(boxes, boxes, [0] * n)).reshape(n, n)
        np.fill_diagonal(covered, 0)
        np.fill_diagonal(box_iou, 0)
        cats = np.array([i.category_id for i in insts])
        for k, inst in enumerate(insts):
            same = cats == cats[k]
            rows.append(
                {
                    "image_id": img.image_id,
                    "instance_id": inst.id,
                    "category_id": inst.category_id,
                    "max_covered": float(covered[k].max()) if n > 1 else 0.0,
                    "same_class_box_iou": float(box_iou[k][same].max()) if same.sum() > 1 else 0.0,
                    "same_class_dup": bool(same.sum() > 1 and box_iou[k][same].max() >= iou_thr),
                }
            )
    return pd.DataFrame(rows)


def benchmark_table(questions: list[dict], gt: dict[str, ImageGT], name_to_id: dict[str, int]):
    """Por pregunta: estado federado y cantidad de instancias GT de cada clase mencionada."""
    rows = []
    for q in questions:
        img = gt[q["image_file"].rsplit(".", 1)[0]]
        row = {"question_id": q["question_id"], "image_id": img.image_id, "type": q["type"]}
        for k, name in enumerate(q["class_en"]):
            cid = name_to_id[name]
            if cid in img.positive_category_ids:
                status = "positive"
            elif cid in img.neg_category_ids:
                status = "negative"
            else:
                status = "unverified"
            row[f"class_{k}"] = name
            row[f"status_{k}"] = status
            row[f"n_gt_{k}"] = sum(i.category_id == cid for i in img.instances)
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
from pathlib import Path

_ = Path('src/eval/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/eval/annotations.py
"""Carga del GT (anotaciones COCO/LVIS). SOLO para evaluación y EDA, nunca para inferencia."""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from pycocotools import mask as mask_utils


@dataclass
class GTInstance:
    """Instancia anotada. `bbox_xyxy` en píxeles de la imagen original."""

    id: int
    category_id: int
    bbox_xyxy: tuple[float, float, float, float]
    segmentation: list | dict
    area: float
    iscrowd: int = 0


@dataclass
class ImageGT:
    """GT de una imagen más su metadata federada LVIS."""

    image_id: str
    file_name: str
    width: int
    height: int
    instances: list[GTInstance] = field(default_factory=list)
    neg_category_ids: set[int] = field(default_factory=set)
    not_exhaustive_category_ids: set[int] = field(default_factory=set)

    @property
    def positive_category_ids(self) -> set[int]:
        """Categorías con al menos una anotación."""
        return {inst.category_id for inst in self.instances}

    def mask(self, inst: GTInstance) -> np.ndarray:
        """Máscara bool HxW de una instancia."""
        return segmentation_to_mask(inst.segmentation, self.height, self.width)


def segmentation_to_mask(segmentation: list | dict, height: int, width: int) -> np.ndarray:
    """Convierte polígonos o RLE (COCO) en una máscara bool HxW."""
    rles = mask_utils.frPyObjects(segmentation, height, width)
    rle = mask_utils.merge(rles) if isinstance(rles, list) else rles
    return mask_utils.decode(rle).astype(bool)


def annotations_path(cfg: dict) -> Path:
    """Ruta del archivo de anotaciones de desarrollo según la config."""
    return Path(cfg["paths"]["data_root"]) / cfg["eval"]["annotations"]


def load_annotations(path: str | Path) -> dict[str, ImageGT]:
    """Lee `annotations.json` y devuelve el GT indexado por image_id (`tpi_0001`, ...)."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    by_num: dict[int, ImageGT] = {}
    for img in data["images"]:
        by_num[img["id"]] = ImageGT(
            image_id=Path(img["file_name"]).stem,
            file_name=img["file_name"],
            width=img["width"],
            height=img["height"],
            neg_category_ids=set(img.get("neg_category_ids", [])),
            not_exhaustive_category_ids=set(img.get("not_exhaustive_category_ids", [])),
        )
    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        by_num[ann["image_id"]].instances.append(
            GTInstance(
                id=ann["id"],
                category_id=ann["category_id"],
                bbox_xyxy=(x, y, x + w, y + h),
                segmentation=ann["segmentation"],
                area=ann["area"],
                iscrowd=ann.get("iscrowd", 0),
            )
        )
    return {gt.image_id: gt for gt in by_num.values()}


In [ ]:
%%writefile src/eval/error_analysis.py
"""Análisis de errores y su propagación por la pipeline (E1-31).

1. **Embudo por instancia GT:** primera etapa en la que se pierde cada objeto anotado. Las etapas
   de detección se recalculan desde la caché de detecciones crudas con el mismo post-proceso que
   la pipeline; en ellas basta con que *alguna* caja de la clase correcta tenga IoU ≥ umbral
   ("cobertura"). Las dos últimas usan el matching uno a uno del evaluador federado.
2. **Tipos de falso positivo** (solo los que el evaluador cuenta como FP).
3. **Galería** de imágenes con mejor y peor F1 y de casos ambiguos (muchas predicciones ignoradas
   por la política federada o confusiones de clase).
"""

from __future__ import annotations

import copy
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from eval.annotations import ImageGT
from eval.lvis_federated import FederatedResult, size_label
from scene.detection.base import Detection
from scene.detection.postprocess import apply_threshold, box_iou_matrix, per_class_nms, postprocess

# Orden del embudo: la primera condición que falla es la etapa donde se pierde la instancia.
FUNNEL = (
    "no_box",  # ninguna caja cruda (de ninguna clase) la cubre
    "wrong_class",  # solo la cubren cajas de otra clase
    "below_threshold",  # la clase correcta la cubre, pero con score bajo el umbral
    "nms",  # la eliminó el NMS
    "max_detections",  # la cortó el tope de detecciones
    "box_match",  # quedó una caja, pero el matching uno a uno la asignó a otra instancia
    "mask",  # caja TP, pero la máscara no llega al IoU
    "ok",  # TP de máscara
)


def _boxes(dets: list[Detection]) -> np.ndarray:
    return np.array([d.box_xyxy for d in dets], dtype=float).reshape(-1, 4)


def _covered(gt_box: np.ndarray, dets: list[Detection], cid: int | None, thr: float) -> bool:
    cand = [d for d in dets if cid is None or d.category_id == cid]
    return bool(cand) and float(box_iou_matrix(gt_box[None], _boxes(cand)).max()) >= thr


def funnel_rows(
    gt: dict[str, ImageGT],
    raw: dict[str, list[Detection]],
    box_res: FederatedResult,
    segm_res: FederatedResult,
    cfg: dict,
) -> pd.DataFrame:
    """Una fila por instancia GT con la etapa en la que se pierde (o `ok`)."""
    det_cfg = cfg["detection"]
    thr = cfg["eval"]["iou_thr"]
    t = cfg["scene"]["size_thresholds"]
    tp_box = {(r.image_id, r.gt_id) for r in box_res.events.itertuples() if r.kind == "tp"}
    tp_segm = {(r.image_id, r.gt_id) for r in segm_res.events.itertuples() if r.kind == "tp"}
    rows = []
    for iid in box_res.events["image_id"].unique():
        img = gt[iid]
        dets_raw = raw[iid]
        dets_thr = apply_threshold(
            dets_raw, det_cfg["score_threshold"], det_cfg.get("per_class_threshold")
        )
        dets_nms = per_class_nms(dets_thr, det_cfg["nms_iou"])
        dets_final = postprocess(dets_raw, det_cfg).detections
        for inst in img.instances:
            box = np.array(inst.bbox_xyxy, dtype=float)
            cid = inst.category_id
            if (iid, inst.id) in tp_segm:
                stage = "ok"
            elif (iid, inst.id) in tp_box:
                stage = "mask"
            elif not _covered(box, dets_raw, None, thr):
                stage = "no_box"
            elif not _covered(box, dets_raw, cid, thr):
                stage = "wrong_class"
            elif not _covered(box, dets_thr, cid, thr):
                stage = "below_threshold"
            elif not _covered(box, dets_nms, cid, thr):
                stage = "nms"
            elif not _covered(box, dets_final, cid, thr):
                stage = "max_detections"
            else:
                stage = "box_match"
            ratio = inst.area / (img.width * img.height)
            rows.append(
                {
                    "image_id": iid,
                    "gt_id": inst.id,
                    "category_id": cid,
                    "area_ratio": ratio,
                    "size": size_label(ratio, t["small_max"], t["medium_max"]),
                    "lost_at": stage,
                }
            )
    return pd.DataFrame(rows)


def funnel_summary(rows: pd.DataFrame) -> dict[str, Any]:
    """Conteos y fracciones por etapa, global y por tamaño, más el embudo acumulado."""

    def counts(df: pd.DataFrame) -> dict[str, Any]:
        c = df["lost_at"].value_counts()
        n = len(df)
        return {
            "n": n,
            **{s: int(c.get(s, 0)) for s in FUNNEL},
            **{f"frac_{s}": float(c.get(s, 0) / n) if n else 0.0 for s in FUNNEL},
        }

    out = {"global": counts(rows)}
    out["by_size"] = {s: counts(rows[rows["size"] == s]) for s in ("small", "medium", "large")}
    # Instancias que siguen "vivas" después de cada etapa (recall acumulado).
    n = len(rows)
    alive, cumulative = n, {}
    for stage in FUNNEL[:-1]:
        alive -= int((rows["lost_at"] == stage).sum())
        cumulative[f"after_{stage}"] = alive / n if n else 0.0
    out["recall_after_stage"] = cumulative
    return out


def fp_types(
    scenes: dict[str, dict], gt: dict[str, ImageGT], res: FederatedResult, cfg: dict
) -> pd.DataFrame:
    """Clasifica cada FP de cajas: duplicado, localización, confusión de clase o fondo."""
    thr = cfg["eval"]["iou_thr"]
    loc_min = cfg["eval"]["error_analysis"]["localization_min_iou"]
    rows = []
    for r in res.events[res.events["kind"] == "fp"].itertuples():
        obj = next(o for o in scenes[r.image_id]["objects"] if o["id"] == r.pred_id)
        img = gt[r.image_id]
        box = np.array(obj["bbox"], dtype=float)[None]
        same = [g for g in img.instances if g.category_id == r.category_id]
        other = [g for g in img.instances if g.category_id != r.category_id]
        iou_same = (
            float(box_iou_matrix(box, np.array([g.bbox_xyxy for g in same])).max()) if same else 0.0
        )
        iou_other, other_cat = 0.0, None
        if other:
            ious = box_iou_matrix(box, np.array([g.bbox_xyxy for g in other]))[0]
            j = int(ious.argmax())
            iou_other, other_cat = float(ious[j]), other[j].category_id
        if r.reason == "unmatched" and iou_same >= thr:
            kind = "duplicate"  # la instancia ya la tomó una predicción de mayor score
        elif r.reason == "unmatched" and iou_same >= loc_min:
            kind = "localization"
        elif iou_other >= thr:
            kind = "class_confusion"
        elif r.reason == "negative_category":
            kind = "negative_category"  # clase verificada como ausente, sin objeto debajo
        else:
            kind = "background"
        rows.append(
            {
                "image_id": r.image_id,
                "pred_id": r.pred_id,
                "category_id": r.category_id,
                "score": r.score,
                "fp_type": kind,
                "iou_same_class": iou_same,
                "iou_other_class": iou_other,
                "confused_with": other_cat if kind == "class_confusion" else None,
            }
        )
    return pd.DataFrame(rows)


def per_image_scores(res: FederatedResult) -> pd.DataFrame:
    """TP/FP/FN/ignorados y F1 por imagen."""
    ev = res.events
    rows = []
    for iid, g in ev.groupby("image_id"):
        k = g["kind"].value_counts()
        tp, fp, fn, ign = (int(k.get(x, 0)) for x in ("tp", "fp", "fn", "ignored"))
        f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
        rows.append({"image_id": iid, "tp": tp, "fp": fp, "fn": fn, "ignored": ign, "f1": f1})
    return pd.DataFrame(rows).sort_values("f1")


def pick_gallery(scores: pd.DataFrame, fps: pd.DataFrame, n: int) -> dict[str, list[str]]:
    """Mejores, peores y ambiguas (más confusiones de clase y más predicciones ignoradas)."""
    confusions = Counter(fps.loc[fps["fp_type"] == "class_confusion", "image_id"])
    amb = scores.assign(
        conf=scores["image_id"].map(confusions).fillna(0),
        ign_frac=scores["ignored"]
        / (scores["tp"] + scores["fp"] + scores["ignored"]).clip(lower=1),
    ).sort_values(["conf", "ign_frac"], ascending=False)
    worst = list(scores["image_id"][:n])
    # Los grupos no se superponen (con pocas imágenes, la peor también podría ser la mejor).
    best = [i for i in scores["image_id"][::-1] if i not in worst][:n]
    taken = set(worst) | set(best)
    ambiguous = [i for i in amb["image_id"] if i not in taken][:n]
    return {"correct": best, "failed": worst, "ambiguous": ambiguous}


def draw_case(
    image: np.ndarray,
    img: ImageGT,
    scene: dict,
    box_res: FederatedResult,
    names: dict[int, str],
    path: str | Path,
) -> Path:
    """GT (verde, FN en amarillo) y predicciones (azul TP, rojo FP, gris ignoradas)."""
    ev = box_res.events[box_res.events["image_id"] == img.image_id]
    kind_by_pred = dict(zip(ev["pred_id"], ev["kind"], strict=True))
    fn_ids = set(ev.loc[ev["kind"] == "fn", "gt_id"])
    canvas = Image.fromarray(image).convert("RGB")
    draw = ImageDraw.Draw(canvas)
    for inst in img.instances:
        color = (255, 210, 0) if inst.id in fn_ids else (0, 200, 0)
        draw.rectangle(inst.bbox_xyxy, outline=color, width=2)
    colors = {"tp": (40, 110, 255), "fp": (255, 40, 40), "ignored": (150, 150, 150)}
    for obj in scene["objects"]:
        kind = kind_by_pred.get(obj["id"], "ignored")
        x1, y1, *_ = obj["bbox"]
        draw.rectangle(obj["bbox"], outline=colors.get(kind, colors["ignored"]), width=1)
        if kind != "ignored":
            draw.text(
                (x1 + 2, y1 + 1), f"{names.get(obj['category_id'], '?')} {kind}", fill=colors[kind]
            )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path)
    return path


def worst_classes(per_category: pd.DataFrame, names: dict[int, str], min_gt: int) -> pd.DataFrame:
    """Clases con peor F1 entre las que tienen al menos `min_gt` instancias GT."""
    df = copy.deepcopy(per_category)
    df["n_gt"] = df["tp"] + df["fn"]
    df["category_name"] = df["category_id"].map(names)
    df = df[df["n_gt"] >= min_gt].sort_values("f1")
    return df[
        ["category_id", "category_name", "n_gt", "tp", "fp", "fn", "precision", "recall", "f1"]
    ]


In [ ]:
%%writefile src/eval/experiments.py
"""Comparación controlada de dos runs de dev (E1-30): métricas en tuning y holdout, costo y
registro en `experiments.csv`."""

from __future__ import annotations

import csv
from pathlib import Path
from typing import Any

from eval.annotations import ImageGT
from eval.lvis_federated import evaluate_from_config, predictions_from_scenes
from eval.scene_consistency import internal_consistency
from eval.tuning import split_tuning_holdout

STAGES_COST = ("detection", "segmentation", "scene")


def run_metrics(
    scenes: dict[str, dict], gt: dict[str, ImageGT], cfg: dict, splits: dict[str, list[str]]
) -> dict[str, Any]:
    """F1 de cajas y máscaras por split (con desglose por tamaño en máscaras) y consistencia."""
    out: dict[str, Any] = {}
    for split, ids in splits.items():
        ids = [i for i in ids if i in scenes]
        preds = predictions_from_scenes({i: scenes[i] for i in ids})
        for iou_type, name in (("bbox", "box"), ("segm", "mask")):
            res = evaluate_from_config(preds, gt, cfg, iou_type=iou_type, image_ids=ids)
            out[f"{split}_f1_{name}"] = res.summary["f1"]
            if iou_type == "segm":
                for size, v in res.by_size.items():
                    out[f"{split}_f1_mask_{size}"] = v["f1"]
    cons = internal_consistency(scenes, cfg)
    out["consistency_issues"] = cons["n_issues"]
    out["mask_outside_bbox_gt10pct"] = cons["mask_outside_bbox"]["frac_objects_gt_10pct"]
    out["objects_per_image"] = cons["n_objects"] / max(cons["n_scenes"], 1)
    return out


def stage_cost_s(trace: list[dict[str, Any]]) -> dict[str, float]:
    """Segundos por etapa en un run (solo lo que se ejecutó, no lo que vino de caché)."""
    cost = dict.fromkeys(STAGES_COST, 0.0)
    for r in trace:
        if r["stage"] in cost and r["status"] == "ok":
            cost[r["stage"]] += r["duration_ms"] / 1000
    return cost


def dev_splits(gt: dict[str, ImageGT], cfg: dict, n_holdout: int = 40) -> dict[str, list[str]]:
    """El mismo split tuning/holdout de E1-29 (misma semilla)."""
    tuning, holdout = split_tuning_holdout(gt, sorted(gt), n_holdout, cfg["seed"])
    return {"tuning": tuning, "holdout": holdout}


def append_row(path: str | Path, row: dict[str, Any]) -> Path:
    """Agrega una fila a `experiments.csv` (crea el archivo con encabezado si no existe)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = not path.is_file()
    with path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row))
        if new:
            writer.writeheader()
        writer.writerow(row)
    return path


In [ ]:
%%writefile src/eval/lvis_federated.py
"""Evaluación federada estilo LVIS: precisión, recall y F1 con IoU ≥ umbral (cajas o máscaras).

Reglas por (imagen, categoría):
- Categoría positiva (tiene GT): matching greedy por score. Match → TP; GT sin match → FN;
  predicción sin match → FP, salvo que la categoría sea no exhaustiva (→ ignorada).
- Categoría negativa (`neg_category_ids`): toda predicción es FP.
- Categoría no verificada (ni positiva ni negativa): las predicciones se ignoran.
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils

from eval.annotations import GTInstance, ImageGT

IOU_TYPES = ("bbox", "segm")
SIZE_LABELS = ("small", "medium", "large")


@dataclass
class Prediction:
    """Predicción a evaluar. `mask` es RLE no comprimido (como en el SceneJSON) o None."""

    category_id: int
    score: float
    bbox_xyxy: tuple[float, float, float, float]
    mask: dict | None = None
    pred_id: str | None = None  # id del objeto en el SceneJSON (para emparejar atributos)


@dataclass
class MatchEvent:
    """Resultado de una predicción o de un GT: tp | fp | fn | ignored."""

    image_id: str
    category_id: int
    kind: str
    score: float | None
    iou: float | None
    area_ratio: float  # del GT para tp/fn, de la predicción para fp/ignored
    reason: str = ""
    pred_id: str | None = None
    gt_id: int | None = None


def predictions_from_scenes(scenes: dict[str, dict]) -> dict[str, list[Prediction]]:
    """Convierte SceneJSON (por image_id) en predicciones."""
    return {
        image_id: [
            Prediction(
                o["category_id"], o["confidence"], tuple(o["bbox"]), o.get("mask"), o.get("id")
            )
            for o in scene["objects"]
        ]
        for image_id, scene in scenes.items()
    }


def size_label(area_ratio: float, small_max: float, medium_max: float) -> str:
    """small si area_ratio < small_max; medium si < medium_max; si no, large."""
    if area_ratio < small_max:
        return "small"
    return "medium" if area_ratio < medium_max else "large"


def prf(tp: int, fp: int, fn: int) -> dict[str, float]:
    """Precisión, recall y F1 (0 cuando el denominador es 0)."""
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": p, "recall": r, "f1": f1}


def _xywh(b: tuple[float, float, float, float]) -> list[float]:
    return [b[0], b[1], b[2] - b[0], b[3] - b[1]]


class _GTCache:
    """RLE comprimido de cada instancia GT (se calcula una sola vez)."""

    def __init__(self) -> None:
        self._rles: dict[int, dict] = {}

    def rle(self, inst: GTInstance, h: int, w: int) -> dict:
        if inst.id not in self._rles:
            rles = mask_utils.frPyObjects(inst.segmentation, h, w)
            self._rles[inst.id] = mask_utils.merge(rles) if isinstance(rles, list) else rles
        return self._rles[inst.id]


def _ious(
    preds: list[Prediction], gts: list[GTInstance], img: ImageGT, iou_type: str, cache: _GTCache
) -> np.ndarray:
    if not preds or not gts:
        return np.zeros((len(preds), len(gts)))
    crowd = [0] * len(gts)
    if iou_type == "bbox":
        dt = [_xywh(p.bbox_xyxy) for p in preds]
        gt = [_xywh(g.bbox_xyxy) for g in gts]
    else:
        if any(p.mask is None for p in preds):
            raise ValueError("iou_type='segm' requiere máscara en todas las predicciones")
        dt = [mask_utils.frPyObjects(p.mask, img.height, img.width) for p in preds]
        gt = [cache.rle(g, img.height, img.width) for g in gts]
    return np.asarray(mask_utils.iou(dt, gt, crowd)).reshape(len(preds), len(gts))


def _pred_area_ratio(p: Prediction, img: ImageGT, iou_type: str) -> float:
    if iou_type == "segm" and p.mask is not None:
        area = sum(p.mask["counts"][1::2])
    else:
        x0, y0, x1, y1 = p.bbox_xyxy
        area = max(0.0, x1 - x0) * max(0.0, y1 - y0)
    return area / (img.width * img.height)


def _pred_event(p: Prediction, img: ImageGT, iou_type: str, kind: str, reason: str):
    """Evento de una predicción sin match (fp o ignorada)."""
    ratio = _pred_area_ratio(p, img, iou_type)
    return MatchEvent(
        img.image_id, p.category_id, kind, p.score, None, ratio, reason, pred_id=p.pred_id
    )


def match_image(
    preds: list[Prediction],
    img: ImageGT,
    iou_type: str = "bbox",
    iou_thr: float = 0.5,
    cache: _GTCache | None = None,
) -> list[MatchEvent]:
    """Eventos TP/FP/FN/ignorados de una imagen según las reglas federadas."""
    if iou_type not in IOU_TYPES:
        raise ValueError(f"iou_type inválido {iou_type!r}; opciones: {IOU_TYPES}")
    cache = cache or _GTCache()
    img_area = img.width * img.height
    events: list[MatchEvent] = []
    categories = {p.category_id for p in preds} | img.positive_category_ids
    for cid in sorted(categories):
        cat_preds = sorted(
            (p for p in preds if p.category_id == cid), key=lambda p: p.score, reverse=True
        )
        gts = [g for g in img.instances if g.category_id == cid]

        if not gts:
            if cid in img.neg_category_ids:
                events += [
                    _pred_event(p, img, iou_type, "fp", "negative_category") for p in cat_preds
                ]
            else:
                events += [
                    _pred_event(p, img, iou_type, "ignored", "unverified_category")
                    for p in cat_preds
                ]
            continue

        ious = _ious(cat_preds, gts, img, iou_type, cache)
        matched = np.zeros(len(gts), dtype=bool)
        for i, p in enumerate(cat_preds):
            # GT libre con mayor IoU (si supera el umbral).
            cand = np.where(~matched & (ious[i] >= iou_thr), ious[i], -1.0)
            j = int(np.argmax(cand))
            if cand[j] >= iou_thr:
                matched[j] = True
                ratio = gts[j].area / img_area
                events.append(
                    MatchEvent(
                        img.image_id,
                        cid,
                        "tp",
                        p.score,
                        float(ious[i, j]),
                        ratio,
                        pred_id=p.pred_id,
                        gt_id=gts[j].id,
                    )
                )
            elif cid in img.not_exhaustive_category_ids:
                events.append(_pred_event(p, img, iou_type, "ignored", "not_exhaustive"))
            else:
                events.append(_pred_event(p, img, iou_type, "fp", "unmatched"))
        for j, g in enumerate(gts):
            if not matched[j]:
                events.append(
                    MatchEvent(
                        img.image_id, cid, "fn", None, None, g.area / img_area, "missed", gt_id=g.id
                    )
                )
    return events


@dataclass
class FederatedResult:
    """Métricas globales, desgloses y eventos individuales."""

    iou_type: str
    iou_thr: float
    summary: dict
    by_size: dict[str, dict]
    by_frequency: dict[str, dict]
    per_category: pd.DataFrame
    events: pd.DataFrame


def _counts(df: pd.DataFrame) -> dict:
    kinds = df["kind"].value_counts() if len(df) else pd.Series(dtype=int)
    tp, fp, fn, ign = (int(kinds.get(k, 0)) for k in ("tp", "fp", "fn", "ignored"))
    n_pred = tp + fp + ign
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "ignored": ign,
        "n_pred": n_pred,
        **prf(tp, fp, fn),
        # Fracción de predicciones que entraron en la cuenta de precisión.
        "evaluable_coverage": (tp + fp) / n_pred if n_pred else 0.0,
    }


def category_frequency(gt: dict[str, ImageGT], rare_max: int, common_max: int) -> dict[int, str]:
    """rare / common / frequent según en cuántas imágenes de GT aparece cada categoría."""
    n_images: dict[int, int] = {}
    for img in gt.values():
        for cid in img.positive_category_ids:
            n_images[cid] = n_images.get(cid, 0) + 1
    return {
        cid: "rare" if n <= rare_max else "common" if n <= common_max else "frequent"
        for cid, n in n_images.items()
    }


def evaluate(
    predictions: dict[str, list[Prediction]],
    gt: dict[str, ImageGT],
    image_ids: list[str] | None = None,
    *,
    iou_type: str,
    iou_thr: float,
    size_thresholds: tuple[float, float],
    freq_bins: tuple[int, int],
) -> FederatedResult:
    """Evalúa las imágenes indicadas (por defecto, las que tienen predicciones).

    `size_thresholds` = (small_max, medium_max) en área relativa; `freq_bins` = (rare_max,
    common_max) en cantidad de imágenes. Una imagen sin predicciones cuenta todo su GT como FN.
    """
    image_ids = sorted(predictions) if image_ids is None else list(image_ids)
    unknown = [i for i in image_ids if i not in gt]
    if unknown:
        raise KeyError(f"Imágenes sin GT: {unknown[:5]}")
    cache = _GTCache()
    events: list[MatchEvent] = []
    for image_id in image_ids:
        events += match_image(predictions.get(image_id, []), gt[image_id], iou_type, iou_thr, cache)

    columns = list(MatchEvent.__dataclass_fields__)
    df = pd.DataFrame([asdict(e) for e in events], columns=columns)
    df["size"] = [size_label(a, *size_thresholds) for a in df["area_ratio"]]
    freq = category_frequency(gt, *freq_bins)
    df["frequency"] = [freq.get(c, "unseen") for c in df["category_id"]]

    by_size = {s: _counts(df[df["size"] == s]) for s in SIZE_LABELS}
    by_freq = {f: _counts(df[df["frequency"] == f]) for f in ("rare", "common", "frequent")}
    rows = [{"category_id": int(c), **_counts(g)} for c, g in df.groupby("category_id")]
    per_cat = pd.DataFrame(rows, columns=["category_id", *_counts(df.iloc[:0])])
    summary = {"n_images": len(image_ids), **_counts(df)}
    # F1 macro: promedio sobre categorías con al menos un TP, FP o FN.
    scored = per_cat[(per_cat["tp"] + per_cat["fp"] + per_cat["fn"]) > 0]
    summary["f1_macro"] = float(scored["f1"].mean()) if len(scored) else 0.0
    return FederatedResult(iou_type, iou_thr, summary, by_size, by_freq, per_cat, df)


def evaluate_from_config(
    predictions: dict[str, list[Prediction]],
    gt: dict[str, ImageGT],
    cfg: dict,
    iou_type: str,
    image_ids: list[str] | None = None,
) -> FederatedResult:
    """`evaluate` con los parámetros de la config (`eval.*` y `scene.size_thresholds`)."""
    sizes = cfg["scene"]["size_thresholds"]
    freq = cfg["eval"]["frequency_bins"]
    return evaluate(
        predictions,
        gt,
        image_ids,
        iou_type=iou_type,
        iou_thr=cfg["eval"]["iou_thr"],
        size_thresholds=(sizes["small_max"], sizes["medium_max"]),
        freq_bins=(freq["rare_max"], freq["common_max"]),
    )


def save_metrics(
    result: FederatedResult,
    out_dir: str | Path,
    name: str = "detection",
    category_names: dict[int, str] | None = None,
) -> Path:
    """Escribe `metrics_<name>.json` y `metrics_<name>_per_category.csv`."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "iou_type": result.iou_type,
        "iou_thr": result.iou_thr,
        "summary": result.summary,
        "by_size": result.by_size,
        "by_frequency": result.by_frequency,
    }
    path = out_dir / f"metrics_{name}.json"
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    per_cat = result.per_category.copy()
    if category_names:
        per_cat.insert(1, "category_name", per_cat["category_id"].map(category_names))
    per_cat.to_csv(out_dir / f"metrics_{name}_per_category.csv", index=False)
    return path


In [ ]:
%%writefile src/eval/oracle.py
"""SceneJSON "oráculo" construido desde el GT (solo evaluación, nunca inferencia).

Cada instancia anotada pasa por el mismo builder que las predicciones (propiedades, color y
relaciones con las mismas reglas), con confianza 1. Correr el mismo ejecutor sobre este JSON y
sobre el predicho permite atribuir un error de QA a la percepción (las respuestas difieren) o al
razonamiento (el oráculo también falla o el DSL interpretado es otro).
"""

from __future__ import annotations

from collections.abc import Mapping
from typing import Any

import numpy as np

from eval.annotations import ImageGT
from scene.detection.base import Detection
from scene.scene.builder import build_scene
from scene.segmentation.sam2 import SegmentedObject


def oracle_scene(
    img: ImageGT, image: np.ndarray, cfg: dict, names: Mapping[int, str]
) -> dict[str, Any]:
    """SceneJSON con una entrada por instancia GT (sin crowd), validado contra el schema."""
    segmented = []
    for inst in img.instances:
        if inst.iscrowd:
            continue
        mask = img.mask(inst)
        det = Detection(
            tuple(float(v) for v in inst.bbox_xyxy),
            1.0,
            inst.category_id,
            names.get(inst.category_id, str(inst.category_id)),
            "gt",
            f"gt_{inst.id}",
        )
        segmented.append(SegmentedObject(det, mask, 1.0, int(mask.sum())))
    scene, _ = build_scene(img.image_id, image, segmented, cfg)
    return scene


In [ ]:
%%writefile src/eval/qa_eval.py
"""Evaluación de QA (E1-24), para el modo determinista (`questions`) y el LLM (`questions_llm`).

- **Fidelidad:** se vuelve a ejecutar el `interpreted_query` de cada respuesta sobre el mismo
  SceneJSON (tiene que dar lo mismo) y se verifica que la evidencia sostenga la respuesta.
- **Exactitud:** el benchmark no trae respuestas, así que la referencia es la respuesta del mismo
  ejecutor, con la consulta estructurada, sobre el SceneJSON oráculo (`eval/oracle.py`). Mide si
  la pipeline llega a la misma respuesta que con una percepción perfecta, bajo nuestras reglas
  de posición, tamaño y relaciones (no las de la cátedra, que no conocemos).
- **Atribución de errores:** en modo determinista la consulta y el ejecutor son los mismos que en
  el oráculo, así que una diferencia es de percepción. En modo LLM, si el DSL interpretado difiere
  del estructurado el error es de interpretación; si coincide, de percepción.
"""

from __future__ import annotations

from collections.abc import Mapping
from typing import Any

import pandas as pd

from scene.qa.dsl import Query
from scene.qa.executor import Executor
from scene.qa.structured_adapter import query_from_benchmark

NON_OK = ("unsupported", "insufficient_information")


def _relations(scene: Mapping[str, Any]) -> set[tuple[str, str, str]]:
    rels = list(scene.get("relations", [])) + list(scene.get("extra_relations", []))
    return {(r["subject"], r["predicate"], r["object"]) for r in rels}


def evidence_supports(item: Mapping[str, Any], scene: Mapping[str, Any]) -> tuple[bool, str]:
    """¿La evidencia sostiene `normalized_answer`? Devuelve (ok, motivo si no)."""
    objs = {o["id"]: o for o in scene.get("objects", [])}
    ev = list(item.get("evidence", []))
    if any(e not in objs for e in ev):
        return False, "evidencia con ids inexistentes"
    if item.get("status") in NON_OK:
        return True, ""
    q = item["interpreted_query"]
    op, ans = q.get("op"), item["normalized_answer"]
    if op == "count" and ans != len(ev):
        return False, f"cuenta {ans} con {len(ev)} objetos de evidencia"
    if op == "exists" and (ans == "yes") != bool(ev):
        return False, f"existencia {ans!r} con {len(ev)} objetos de evidencia"
    if op in ("position", "size"):
        if not ev or objs[ev[0]][op] != ans:
            return False, f"{op} {ans!r} no coincide con el objeto de evidencia"
    if op == "relation":
        if len(ev) < 2:
            return False, "relación con menos de 2 objetos de evidencia"
        holds = (ev[0], q["relation"], ev[1]) in _relations(scene)
        if holds != (ans == "yes"):
            return False, f"relación {ans!r} pero la tripla {'sí' if holds else 'no'} está"
    return True, ""


def reexecutes(item: Mapping[str, Any], scene: Mapping[str, Any], executor: Executor) -> bool:
    """Volver a ejecutar el DSL guardado da la misma respuesta (determinismo y trazabilidad)."""
    query = Query.model_validate(item["interpreted_query"])
    return executor.execute(query, scene).normalized_answer == item["normalized_answer"]


def _dsl_key(q: Query) -> tuple:
    """Lo que tiene que coincidir entre el DSL del LLM y el estructurado."""
    t = [
        (x.category, tuple(sorted(x.filters.active().items()))) for x in (q.target, q.target2) if x
    ]
    return (q.op, tuple(t), q.relation, q.attribute)


def evaluate_mode(
    mode: str,
    benchmark: list[Mapping[str, Any]],
    pred_scenes: Mapping[str, Mapping[str, Any]],
    oracle_scenes: Mapping[str, Mapping[str, Any]],
    executor: Executor,
    phrases: Mapping[str, str],
    resolve: Any,
) -> pd.DataFrame:
    """Una fila por consigna con respuesta, referencia oráculo, fidelidad y atribución.

    `resolve(texto) -> category_id` normaliza las categorías para comparar los DSL.
    """
    field = "questions" if mode == "deterministic" else "questions_llm"
    rows = []
    for q in benchmark:
        iid = q["image_file"].rsplit(".", 1)[0]
        scene = pred_scenes.get(iid)
        items = {it["question_id"]: it for it in (scene or {}).get(field) or []}
        item = items.get(q["question_id"])
        if item is None:
            continue
        structured = query_from_benchmark(q, phrases)
        ref = executor.execute(structured, oracle_scenes[iid])
        row: dict[str, Any] = {
            "mode": mode,
            "image_id": iid,
            "question_id": q["question_id"],
            "type": q["type"],
            "answer": item["normalized_answer"],
            "status": item.get("status", "ok"),
            "oracle_answer": ref.normalized_answer,
            "oracle_status": ref.status,
            "confidence": item["confidence"],
        }
        supported, why = evidence_supports(item, scene)
        row["evidence_supports"] = supported
        row["evidence_issue"] = why
        # La descripción del LLM es texto generado: re-ejecutar su DSL da la plantilla, no aplica.
        llm_text = mode == "llm" and q["type"] == "description"
        row["reexecutes"] = None if llm_text else reexecutes(item, scene, executor)
        llm_q = Query.model_validate(item["interpreted_query"])
        if mode == "llm":
            row["interpreter_status"] = item.get("interpreter", {}).get("status")
            # E1-27: estado del descriptor (ok = fiel; unfaithful = se usó la plantilla).
            row["describer_status"] = item.get("interpreter", {}).get("describer_status")

            def canon(query: Query) -> tuple:
                key = _dsl_key(query)
                cats = tuple((resolve(c), f) for c, f in key[1])
                return (key[0], cats, key[2], key[3])

            row["dsl_match"] = canon(llm_q) == canon(structured)
        if q["type"] == "description":
            row["correct"] = None  # texto libre: se mide con la fidelidad del descriptor
            row["error_source"] = None
        else:
            row["correct"] = item["normalized_answer"] == ref.normalized_answer
            if row["correct"]:
                row["error_source"] = None
            elif ref.status != "ok":
                row["error_source"] = "oracle_not_ok"  # ni con percepción perfecta se responde
            elif mode == "llm" and not row["dsl_match"]:
                row["error_source"] = "interpretation"
            else:
                row["error_source"] = "perception"
        rows.append(row)
    return pd.DataFrame(rows)


def _rate(s: pd.Series) -> float | None:
    """Proporción de True ignorando los casos donde no aplica (None)."""
    s = s.dropna()
    return float(s.astype(bool).mean()) if len(s) else None


def summarize(df: pd.DataFrame) -> dict[str, Any]:
    """Exactitud por modo y tipo, estados, fidelidad, atribución y métricas del intérprete."""
    out: dict[str, Any] = {}
    for mode, g in df.groupby("mode"):
        closed = g[g["type"] != "description"]
        m: dict[str, Any] = {
            "n": len(g),
            "accuracy_vs_oracle": _rate(closed["correct"]),
            "by_type": {
                t: {
                    "n": len(x),
                    "accuracy_vs_oracle": _rate(x["correct"]),
                    "status": x["status"].value_counts().to_dict(),
                }
                for t, x in closed.groupby("type")
            },
            "unsupported_rate": float((g["status"] == "unsupported").mean()),
            "evidence_supports_rate": _rate(g["evidence_supports"]),
            "reexecutes_rate": _rate(g["reexecutes"]),
            "error_source": closed["error_source"].value_counts().to_dict(),
        }
        if mode == "llm":
            m["json_valid_rate"] = float((g["interpreter_status"] == "ok").mean())
            m["dsl_match_rate"] = _rate(g["dsl_match"])
            desc = g[g["type"] == "description"]
            m["descriptions"] = {
                "n": len(desc),
                "faithful_rate": _rate(desc["describer_status"].eq("ok")) if len(desc) else None,
                "template_fallback": int(desc["describer_status"].eq("unfaithful").sum()),
                "not_interpreted_as_describe": int(desc["describer_status"].isna().sum()),
            }
        out[mode] = m
    return out


In [ ]:
%%writefile src/eval/scene_consistency.py
"""Consistencia de la representación estructurada (E1-19).

1. **Interna** (sin GT): recalcula desde la máscara decodificada el área, el centroide, la
   posición y el tamaño, y revisa relaciones (ids, enum, inversa presente, coherencia con los
   centroides) y evidencias. Se recalcula con código propio, no con el de `scene.scene`, para que
   el chequeo no herede los mismos errores que el builder.
2. **Contra GT** (solo dev): en los pares predicción–GT emparejados por el evaluador federado,
   exactitud de posición, tamaño y color (el color "real" es el dominante dentro de la máscara GT).
"""

from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
from pycocotools import mask as mask_utils

from eval.annotations import ImageGT
from eval.lvis_federated import evaluate_from_config, predictions_from_scenes

GRID = (
    ("top_left", "top_center", "top_right"),
    ("middle_left", "center", "middle_right"),
    ("bottom_left", "bottom_center", "bottom_right"),
)
INVERSE = {"left_of": "right_of", "right_of": "left_of", "above": "below", "below": "above"}
SCHEMA_PREDICATES = {"left_of", "right_of", "above", "below", "overlaps"}


def decode_rle(rle: dict) -> np.ndarray:
    """RLE no comprimido column-major → bool HxW (vía pycocotools, independiente de scene)."""
    h, w = rle["size"]
    return mask_utils.decode(mask_utils.frPyObjects(rle, h, w)).astype(bool)


def grid_cell(cx: float, cy: float, width: int, height: int) -> str:
    col = min(int(3 * cx / width), 2) if cx >= 0 else 0
    row = min(int(3 * cy / height), 2) if cy >= 0 else 0
    return GRID[row][col]


def size_class(area_ratio: float, small_max: float, medium_max: float) -> str:
    return "small" if area_ratio < small_max else "medium" if area_ratio < medium_max else "large"


def object_issues(
    obj: dict, width: int, height: int, sizes: tuple[float, float], tol_px: float
) -> tuple[list[str], dict[str, float]]:
    """Inconsistencias de un objeto y medidas auxiliares (por ejemplo, máscara fuera de la caja)."""
    issues: list[str] = []
    mask = decode_rle(obj["mask"])
    if list(mask.shape) != [height, width]:
        return ["mask_size"], {}
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return ["mask_empty"], {}
    area_ratio = len(xs) / (width * height)
    cx, cy = float(xs.mean()), float(ys.mean())
    if not np.isclose(obj["area_ratio"], area_ratio, rtol=1e-6, atol=1e-9):
        issues.append("area_ratio")
    if abs(obj["centroid"][0] - cx) > tol_px or abs(obj["centroid"][1] - cy) > tol_px:
        issues.append("centroid")
    if obj["position"] != grid_cell(cx, cy, width, height):
        issues.append("position")
    if obj["size"] != size_class(area_ratio, *sizes):
        issues.append("size")
    x1, y1, x2, y2 = obj["bbox"]
    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
        issues.append("bbox_out_of_image")
    inside = mask[
        max(0, int(np.floor(y1))) : int(np.ceil(y2)), max(0, int(np.floor(x1))) : int(np.ceil(x2))
    ].sum()
    return issues, {"mask_outside_bbox": 1 - inside / len(xs)}


def relation_issues(scene: dict) -> list[str]:
    """Ids inexistentes, predicados fuera del enum, inversas faltantes y dirección incoherente."""
    issues: list[str] = []
    objs = {o["id"]: o for o in scene["objects"]}
    triples = {(r["subject"], r["predicate"], r["object"]) for r in scene["relations"]}
    for s, p, o in triples:
        if s not in objs or o not in objs:
            issues.append("relation_unknown_id")
            continue
        if s == o:
            issues.append("relation_self")
        if p not in SCHEMA_PREDICATES:
            issues.append("relation_predicate")
            continue
        inverse = INVERSE.get(p, p)  # overlaps es simétrica
        if (o, inverse, s) not in triples:
            issues.append("relation_missing_inverse")
        (sx, sy), (ox, oy) = objs[s]["centroid"], objs[o]["centroid"]
        wrong = {"left_of": sx >= ox, "right_of": sx <= ox, "above": sy >= oy, "below": sy <= oy}
        if wrong.get(p, False):
            issues.append("relation_direction")
    for field in ("questions", "questions_llm"):
        for qa in scene.get(field) or []:
            if any(ev not in objs for ev in qa.get("evidence", [])):
                issues.append("evidence_unknown_id")
    return issues


def internal_consistency(scenes: dict[str, dict], cfg: dict) -> dict[str, Any]:
    """Resumen de inconsistencias internas sobre todas las escenas."""
    t = cfg["scene"]["size_thresholds"]
    sizes = (t["small_max"], t["medium_max"])
    tol = 10 ** -cfg["eval"]["centroid_decimals"]
    counts: Counter[str] = Counter()
    outside: list[float] = []
    scenes_with_issues: list[str] = []
    n_objects = 0
    for iid, scene in scenes.items():
        w, h = scene["image"]["width"], scene["image"]["height"]
        before = sum(counts.values())
        for obj in scene["objects"]:
            n_objects += 1
            issues, extra = object_issues(obj, w, h, sizes, tol)
            counts.update(issues)
            if "mask_outside_bbox" in extra:
                outside.append(extra["mask_outside_bbox"])
        counts.update(relation_issues(scene))
        if sum(counts.values()) > before:
            scenes_with_issues.append(iid)
    out = np.array(outside) if outside else np.zeros(1)
    return {
        "n_scenes": len(scenes),
        "n_objects": n_objects,
        "n_relations": sum(len(s["relations"]) for s in scenes.values()),
        "n_issues": int(sum(counts.values())),
        "issues": dict(counts),
        "scenes_with_issues": scenes_with_issues,
        # Fracción de la máscara que cae fuera de la caja entregada (no es error del schema,
        # pero la caja y la máscara deberían describir el mismo objeto).
        "mask_outside_bbox": {
            "mean": float(out.mean()),
            "p95": float(np.percentile(out, 95)),
            "frac_objects_gt_10pct": float((out > 0.10).mean()),
        },
    }


def attribute_accuracy(
    scenes: dict[str, dict],
    gt: dict[str, ImageGT],
    cfg: dict,
    images: dict[str, np.ndarray] | None = None,
) -> dict[str, Any]:
    """Exactitud de posición, tamaño (y color, si se pasan las imágenes) en pares TP de máscaras."""
    from scene.scene.color import dominant_color_from_config

    ids = [i for i in scenes if i in gt]
    preds = predictions_from_scenes({i: scenes[i] for i in ids})
    result = evaluate_from_config(preds, gt, cfg, iou_type="segm", image_ids=ids)
    t = cfg["scene"]["size_thresholds"]
    sizes = (t["small_max"], t["medium_max"])
    hits: Counter[str] = Counter()
    confusion: dict[str, Counter] = {"size": Counter(), "color": Counter()}
    n = n_color = 0
    tp = result.events[result.events["kind"] == "tp"]
    for row in tp.itertuples():
        img = gt[row.image_id]
        obj = next(o for o in scenes[row.image_id]["objects"] if o["id"] == row.pred_id)
        inst = next(g for g in img.instances if g.id == row.gt_id)
        gmask = img.mask(inst)
        ys, xs = np.nonzero(gmask)
        if len(xs) == 0:
            continue
        n += 1
        gt_pos = grid_cell(float(xs.mean()), float(ys.mean()), img.width, img.height)
        gt_size = size_class(len(xs) / (img.width * img.height), *sizes)
        hits["position"] += obj["position"] == gt_pos
        hits["size"] += obj["size"] == gt_size
        confusion["size"][f"{gt_size}->{obj['size']}"] += 1
        if images is not None and row.image_id in images:
            ref = dominant_color_from_config(images[row.image_id], gmask, cfg)
            pred = obj.get("attributes", {}).get("color")
            if ref is not None and pred is not None:
                n_color += 1
                hits["color"] += pred == ref.name
                confusion["color"][f"{ref.name}->{pred}"] += 1
    return {
        "n_pairs": n,
        "position_acc": hits["position"] / n if n else None,
        "size_acc": hits["size"] / n if n else None,
        "n_color_pairs": n_color,
        "color_acc": hits["color"] / n_color if n_color else None,
        "size_confusion": dict(confusion["size"]),
        "color_confusion": dict(confusion["color"].most_common(20)),
    }


def save_scene_metrics(metrics: dict[str, Any], out_dir: str | Path) -> Path:
    path = Path(out_dir) / "metrics_scene.json"
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return path


In [ ]:
%%writefile src/eval/segmentation.py
"""Evaluación de la segmentación con oráculo (E1-12).

Se corre SAM2 con las **cajas GT** como prompt (solo en evaluación, nunca en la inferencia).
Como cada máscara sale de la caja de una instancia conocida, se compara directo con su máscara
GT, sin matching: eso mide el error propio de SAM2 y del post-proceso, separado del detector.
Con una sola pasada de SAM2 se evalúan la máscara cruda y la post-procesada.
"""

from __future__ import annotations

from typing import Any

import numpy as np

from eval.annotations import ImageGT
from eval.lvis_federated import FederatedResult, size_label
from scene.detection.base import Detection
from scene.segmentation.postprocess import postprocess_mask


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def gt_detections(img: ImageGT) -> list[Detection]:
    """Una detección por instancia GT (score 1), con `raw_query` = id de la instancia."""
    return [
        Detection(inst.bbox_xyxy, 1.0, inst.category_id, str(inst.category_id), "gt", str(inst.id))
        for inst in img.instances
        if not inst.iscrowd
    ]


def oracle_rows(
    img: ImageGT,
    segmenter: Any,
    image: np.ndarray,
    pp_cfg: dict[str, Any],
    small_max: float | None = None,
) -> list[dict[str, Any]]:
    """IoU con GT de cada instancia, para la máscara cruda de SAM2 y la post-procesada.

    Con `small_max` agrega la variante de E1-30 (`post_small`): post-proceso solo si la máscara
    cruda tiene área relativa menor a `small_max`. `segmenter` debe estar construido con el
    post-proceso apagado (así devuelve la cruda).
    """
    pp_all = {**pp_cfg, "apply_below_area_ratio": None}
    dets = gt_detections(img)
    by_id = {str(inst.id): inst for inst in img.instances}
    rows = []
    for obj in segmenter.segment(image, dets):
        inst = by_id[obj.detection.raw_query]
        gmask = img.mask(inst)
        post = postprocess_mask(obj.mask, obj.detection.box_xyxy, pp_all)
        row = {
            "image_id": img.image_id,
            "gt_id": inst.id,
            "category_id": inst.category_id,
            "area_ratio": float(gmask.sum()) / (img.width * img.height),
            "mask_score": obj.mask_score,
            "iou_raw": mask_iou(obj.mask, gmask),
            "iou_post": mask_iou(post, gmask),
        }
        if small_max is not None:
            small = obj.mask.mean() < small_max
            row["iou_post_small"] = row["iou_post"] if small else row["iou_raw"]
        rows.append(row)
    return rows


def summarize_oracle(rows: list[dict[str, Any]], cfg: dict) -> dict[str, Any]:
    """mIoU y fracción con IoU ≥ umbral, global y por tamaño, para cruda y post-procesada."""
    thr = cfg["eval"]["iou_thr"]
    t = cfg["scene"]["size_thresholds"]

    def stats(sub: list[dict[str, Any]]) -> dict[str, Any]:
        out: dict[str, Any] = {"n": len(sub)}
        kinds = [k for k in ("raw", "post", "post_small") if sub and f"iou_{k}" in sub[0]]
        for kind in kinds or ("raw", "post"):
            ious = np.array([r[f"iou_{kind}"] for r in sub]) if sub else np.zeros(0)
            out[f"miou_{kind}"] = float(ious.mean()) if len(ious) else None
            out[f"frac_iou_ge_thr_{kind}"] = float((ious >= thr).mean()) if len(ious) else None
        return out

    by_size = {
        s: stats(
            [r for r in rows if size_label(r["area_ratio"], t["small_max"], t["medium_max"]) == s]
        )
        for s in ("small", "medium", "large")
    }
    # Correlación entre el IoU que predice SAM2 y el real: ¿sirve mask_score como confianza?
    if len(rows) > 2:
        corr = float(
            np.corrcoef([r["mask_score"] for r in rows], [r["iou_post"] for r in rows])[0, 1]
        )
    else:
        corr = None
    return {"iou_thr": thr, **stats(rows), "by_size": by_size, "corr_mask_score_iou": corr}


def predicted_pairs_summary(box: FederatedResult, segm: FederatedResult) -> dict[str, Any]:
    """Con cajas predichas: mIoU de máscaras en TP y cuántos TP de cajas se pierden en máscaras."""
    tp_box = box.events[box.events["kind"] == "tp"]
    tp_segm = segm.events[segm.events["kind"] == "tp"]
    box_ids = set(zip(tp_box["image_id"], tp_box["pred_id"], strict=True))
    segm_ids = set(zip(tp_segm["image_id"], tp_segm["pred_id"], strict=True))
    return {
        "f1_box": box.summary["f1"],
        "f1_mask": segm.summary["f1"],
        "tp_box": len(box_ids),
        "tp_mask": len(segm_ids),
        "box_tp_lost_in_mask": len(box_ids - segm_ids),
        "miou_mask_tp": float(tp_segm["iou"].mean()) if len(tp_segm) else None,
    }


In [ ]:
%%writefile src/eval/tuning.py
"""Ajuste del post-proceso de detección (E1-29) sobre la caché de detecciones crudas.

El umbral se elige en un split de **tuning** y se reporta en un **holdout** que no participa de
la elección. Ajustar hiperparámetros con GT es legítimo (no asiste a la inferencia de ninguna
imagen); lo que no se puede es reportar el F1 del mismo conjunto con el que se eligió.
"""

from __future__ import annotations

import copy
from typing import Any

import numpy as np
import pandas as pd

from eval.annotations import ImageGT
from eval.lvis_federated import Prediction, evaluate_from_config
from scene.detection.base import Detection
from scene.detection.postprocess import postprocess


def split_tuning_holdout(
    gt: dict[str, ImageGT], image_ids: list[str], n_holdout: int, seed: int
) -> tuple[list[str], list[str]]:
    """Split estratificado por cantidad de instancias GT.

    Se ordenan las imágenes por cantidad de instancias y se cortan en grupos consecutivos de
    `len / n_holdout`; de cada grupo va una imagen al azar (con semilla) al holdout.
    """
    if not 0 < n_holdout < len(image_ids):
        raise ValueError("n_holdout tiene que estar entre 1 y la cantidad de imágenes - 1")
    rng = np.random.default_rng(seed)
    ordered = sorted(image_ids, key=lambda i: (len(gt[i].instances), i))
    groups = np.array_split(np.array(ordered), n_holdout)
    holdout = sorted(str(g[rng.integers(len(g))]) for g in groups)
    tuning = sorted(set(image_ids) - set(holdout))
    return tuning, holdout


def to_predictions(dets: list[Detection]) -> list[Prediction]:
    return [Prediction(d.category_id, d.score, d.box_xyxy) for d in dets]


def evaluate_postprocess(
    raw: dict[str, list[Detection]],
    gt: dict[str, ImageGT],
    image_ids: list[str],
    cfg: dict,
    det_overrides: dict[str, Any],
) -> dict[str, float]:
    """F1 de cajas de aplicar `postprocess` con `cfg['detection']` + overrides a la caché."""
    det_cfg = copy.deepcopy(cfg["detection"])
    det_cfg.update(det_overrides)
    preds = {i: to_predictions(postprocess(raw[i], det_cfg).detections) for i in image_ids}
    res = evaluate_from_config(preds, gt, cfg, iou_type="bbox", image_ids=image_ids)
    s = res.summary
    return {
        "precision": s["precision"],
        "recall": s["recall"],
        "f1": s["f1"],
        "evaluable_coverage": s["evaluable_coverage"],
        "preds_per_image": s["n_pred"] / len(image_ids),
        "f1_small": res.by_size["small"]["f1"],
    }


def sweep(
    raw: dict[str, list[Detection]],
    gt: dict[str, ImageGT],
    splits: dict[str, list[str]],
    cfg: dict,
    thresholds: list[float],
    variants: dict[str, dict[str, Any]],
) -> pd.DataFrame:
    """Una fila por (variante, umbral, split). `variants` = nombre → overrides de detección."""
    rows = []
    for name, overrides in variants.items():
        for thr in thresholds:
            for split, ids in splits.items():
                m = evaluate_postprocess(
                    raw, gt, ids, cfg, {**overrides, "score_threshold": float(thr)}
                )
                rows.append({"variant": name, "threshold": float(thr), "split": split, **m})
    return pd.DataFrame(rows)


def choose(curve: pd.DataFrame, variant: str, split: str = "tuning") -> pd.Series:
    """Fila de máximo F1 en `split`; ante empates, el umbral más alto (menos predicciones)."""
    sub = curve[(curve["variant"] == variant) & (curve["split"] == split)]
    best = sub["f1"].max()
    return sub[np.isclose(sub["f1"], best)].sort_values("threshold").iloc[-1]


In [ ]:
from pathlib import Path

_ = Path('src/scene/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/config.py
"""Carga de configuración, semillas, run_id y manifest de cada corrida."""

from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from importlib import metadata
from pathlib import Path
from typing import Any

import numpy as np
import yaml

# Librerías cuya versión se registra en el manifest (si están instaladas).
TRACKED_PACKAGES = (
    "numpy",
    "torch",
    "transformers",
    "pycocotools",
    "opencv-python",
    "pydantic",
    "jsonschema",
    "scikit-learn",
    "pandas",
    "pillow",
)


def deep_merge(base: dict, override: dict) -> dict:
    """Combina dos dicts: los anidados se mezclan y el resto (listas incluidas) se reemplaza."""
    out = copy.deepcopy(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = copy.deepcopy(value)
    return out


def apply_dotlist(cfg: dict, dotlist: list[str]) -> dict:
    """Aplica overrides tipo `detection.threshold=0.3`. El valor se parsea como YAML."""
    out = copy.deepcopy(cfg)
    for item in dotlist:
        if "=" not in item:
            raise ValueError(f"Override inválido (se espera clave=valor): {item!r}")
        dotted, raw = item.split("=", 1)
        keys = dotted.strip().split(".")
        node = out
        for key in keys[:-1]:
            node = node.setdefault(key, {})
            if not isinstance(node, dict):
                raise ValueError(f"No se puede asignar dentro de un valor no-dict: {dotted!r}")
        node[keys[-1]] = yaml.safe_load(raw)
    return out


def load_config(*paths: str | Path, dotlist: list[str] | None = None) -> dict:
    """Carga uno o más YAML y los combina de izquierda a derecha (base primero)."""
    if not paths:
        raise ValueError("Se necesita al menos un archivo de configuración")
    cfg: dict = {}
    for path in paths:
        with open(path, encoding="utf-8") as f:
            data = yaml.safe_load(f) or {}
        if not isinstance(data, dict):
            raise ValueError(f"{path}: la raíz del YAML debe ser un mapeo")
        cfg = deep_merge(cfg, data)
    return apply_dotlist(cfg, dotlist or [])


def config_hash(cfg: dict) -> str:
    """Hash SHA-256 de la config canónica (claves ordenadas): misma config → mismo hash."""
    canonical = json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def make_run_id(cfg: dict, now: datetime | None = None) -> str:
    """run_id = timestamp + hash corto de la config."""
    now = now or datetime.now()
    return f"{now:%Y%m%d-%H%M%S}-{config_hash(cfg)[:8]}"


def set_seed(seed: int) -> None:
    """Fija las semillas de random, numpy y torch (si está instalado)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
    except ImportError:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name: str = "auto") -> str:
    """Traduce `auto` a `cuda` si hay GPU disponible; si no, `cpu`."""
    if name != "auto":
        return name
    try:
        import torch
    except ImportError:
        return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"


def data_path(cfg: dict, key: str) -> Path:
    """Ruta absoluta de `paths.<key>` resuelta contra `paths.data_root`."""
    paths = cfg["paths"]
    return Path(paths["data_root"]) / paths[key]


def _package_versions() -> dict[str, str | None]:
    versions: dict[str, str | None] = {}
    for name in TRACKED_PACKAGES:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def _hardware_info(device: str) -> dict[str, Any]:
    info: dict[str, Any] = {
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "cpu_count": os.cpu_count(),
        "device": device,
        "gpu": None,
    }
    if device.startswith("cuda"):
        import torch

        props = torch.cuda.get_device_properties(0)
        info["gpu"] = {"name": props.name, "total_memory_mb": props.total_memory // 2**20}
    return info


def _git_commit() -> str | None:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5
        )
    except (OSError, subprocess.SubprocessError):
        return None
    return out.stdout.strip() or None


@dataclass
class RunContext:
    """Datos de una corrida: id, config y carpeta de salida."""

    run_id: str
    config: dict
    config_hash: str
    run_dir: Path
    device: str

    @property
    def manifest_path(self) -> Path:
        return self.run_dir / self.config["tracing"]["manifest_file"]

    @property
    def trace_path(self) -> Path:
        return self.run_dir / self.config["tracing"]["trace_file"]

    def update_manifest(self, key: str, value: Any) -> None:
        """Agrega o reemplaza una clave del manifest (por ejemplo, los modelos usados)."""
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        manifest[key] = value
        _write_json(self.manifest_path, manifest)


def _write_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def start_run(cfg: dict, run_id: str | None = None) -> RunContext:
    """Fija la semilla, crea `outputs/runs/<run_id>/` y escribe `run_manifest.json`."""
    set_seed(cfg["seed"])
    if run_id is None:
        # Dos runs con la misma config en el mismo segundo: se agrega un sufijo.
        base_id = run_id = make_run_id(cfg)
        n = 1
        while (Path(cfg["paths"]["outputs"]) / run_id).exists():
            run_id, n = f"{base_id}-{n}", n + 1
    run_dir = Path(cfg["paths"]["outputs"]) / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    device = resolve_device(cfg.get("device", "auto"))
    ctx = RunContext(
        run_id=run_id, config=cfg, config_hash=config_hash(cfg), run_dir=run_dir, device=device
    )
    manifest = {
        "run_id": run_id,
        "config_hash": ctx.config_hash,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "seed": cfg["seed"],
        "config": cfg,
        "python": sys.version,
        "packages": _package_versions(),
        "hardware": _hardware_info(device),
        "git_commit": _git_commit(),
        "models": {},
    }
    _write_json(ctx.manifest_path, manifest)
    return ctx


In [ ]:
from pathlib import Path

_ = Path('src/scene/detection/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/detection/base.py
"""Contrato de los detectores: `Detection` y la interfaz `Detector`."""

from __future__ import annotations

import json
from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np


@dataclass(frozen=True)
class Detection:
    """Una caja predicha, en píxeles de la imagen original."""

    box_xyxy: tuple[float, float, float, float]
    score: float
    category_id: int
    category_name: str
    source: str  # "owlv2", "yolo_world", ...
    raw_query: str  # texto/sinónimo que la disparó

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> Detection:
        return cls(**{**data, "box_xyxy": tuple(data["box_xyxy"])})


class Detector(ABC):
    """Detector open-vocabulary. El vocabulario se fija al construirlo."""

    source: str
    model_id: str

    @abstractmethod
    def detect(self, image: np.ndarray) -> list[Detection]:
        """Detecta sobre una imagen RGB uint8 HxWx3. Devuelve detecciones crudas (sin NMS)."""

    def params(self) -> dict[str, Any]:
        """Parámetros que se registran en la traza."""
        return {}


def save_detections(dets: list[Detection], path: str | Path) -> None:
    """Guarda detecciones en JSON (caché por imagen)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps([d.to_dict() for d in dets], ensure_ascii=False), encoding="utf-8")


def load_detections(path: str | Path) -> list[Detection]:
    return [Detection.from_dict(d) for d in json.loads(Path(path).read_text(encoding="utf-8"))]


In [ ]:
%%writefile src/scene/detection/debug.py
"""Imágenes de debug con las cajas dibujadas."""

from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

from scene.detection.base import Detection


def draw_detections(
    image: np.ndarray, dets: list[Detection], path: str | Path, max_boxes: int = 50
) -> Path:
    """Dibuja las `max_boxes` cajas de mayor score con su clase y score, y guarda un PNG."""
    img = Image.fromarray(image).convert("RGB")
    draw = ImageDraw.Draw(img)
    for d in sorted(dets, key=lambda d: -d.score)[:max_boxes]:
        x1, y1, x2, y2 = d.box_xyxy
        draw.rectangle([x1, y1, x2, y2], outline=(255, 40, 40), width=2)
        label = f"{d.category_name} {d.score:.2f}"
        tx, ty = x1 + 2, max(0, y1 - 12)
        draw.rectangle(draw.textbbox((tx, ty), label), fill=(255, 40, 40))
        draw.text((tx, ty), label, fill=(255, 255, 255))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)
    return path


In [ ]:
%%writefile src/scene/detection/owlv2.py
"""Detector OWLv2 (línea base) vía `transformers`.

OWLv2 rellena la imagen hasta hacerla cuadrada (abajo y a la derecha) y la reescala a 960 px.
Sus cajas salen normalizadas respecto de ese cuadrado, así que para volver a píxeles de la imagen
original se multiplica por el lado mayor (no por ancho y alto por separado).
"""

from __future__ import annotations

from typing import Any

import numpy as np

from scene.detection.base import Detection, Detector
from scene.detection.vocabulary import Vocabulary


def owl_boxes_to_xyxy(boxes_cxcywh: np.ndarray, width: int, height: int) -> np.ndarray:
    """Cajas (cx, cy, w, h) normalizadas al cuadrado con padding → xyxy en píxeles originales."""
    side = max(width, height)
    cx, cy, w, h = (boxes_cxcywh[:, i] * side for i in range(4))
    xyxy = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)
    xyxy[:, [0, 2]] = xyxy[:, [0, 2]].clip(0, width)
    xyxy[:, [1, 3]] = xyxy[:, [1, 3]].clip(0, height)
    return xyxy


def decode_detections(
    scores: np.ndarray,
    boxes_xyxy: np.ndarray,
    vocab: Vocabulary,
    score_floor: float,
    raw_topk: int,
    labels_per_box: int,
    source: str = "owlv2",
) -> list[Detection]:
    """Scores por (caja, query) → detecciones crudas.

    1. Por cada caja y categoría queda la query (nombre o sinónimo) con mayor score.
    2. Por cada caja se conservan las `labels_per_box` categorías con mayor score.
    3. Se descarta lo que está bajo `score_floor` y se ordena; quedan como máximo `raw_topk`.
    """
    by_cat = vocab.by_category()
    cat_ids = [cid for cid, qs in by_cat.items() if qs]
    query_index = {id(q): i for i, q in enumerate(vocab.queries)}
    n_boxes = scores.shape[0]
    cat_scores = np.empty((n_boxes, len(cat_ids)), dtype=np.float32)
    best_query = np.empty((n_boxes, len(cat_ids)), dtype=np.int64)
    for j, cid in enumerate(cat_ids):
        qidx = np.array([query_index[id(q)] for q in by_cat[cid]])
        sub = scores[:, qidx]
        cat_scores[:, j] = sub.max(axis=1)
        best_query[:, j] = qidx[sub.argmax(axis=1)]

    k = min(labels_per_box, len(cat_ids))
    top = np.argsort(-cat_scores, axis=1, kind="stable")[:, :k]
    rows = np.repeat(np.arange(n_boxes), k)
    cols = top.reshape(-1)
    sel_scores = cat_scores[rows, cols]
    valid = (sel_scores >= score_floor) & (boxes_xyxy[rows, 2] > boxes_xyxy[rows, 0])
    valid &= boxes_xyxy[rows, 3] > boxes_xyxy[rows, 1]
    rows, cols, sel_scores = rows[valid], cols[valid], sel_scores[valid]
    order = np.argsort(-sel_scores, kind="stable")[:raw_topk]

    names = {c.id: c.name for c in vocab.categories}
    dets: list[Detection] = []
    for r, c in zip(rows[order], cols[order], strict=True):
        q = vocab.queries[best_query[r, c]]
        dets.append(
            Detection(
                box_xyxy=tuple(float(v) for v in boxes_xyxy[r]),
                score=float(cat_scores[r, c]),
                category_id=q.category_id,
                category_name=names[q.category_id],
                source=source,
                raw_query=q.text,
            )
        )
    return dets


class Owlv2Detector(Detector):
    """OWLv2 con el vocabulario fijo. Devuelve detecciones crudas (el NMS va en postprocess)."""

    source = "owlv2"

    def __init__(
        self,
        vocab: Vocabulary,
        model_id: str,
        device: str = "cpu",
        score_floor: float = 0.01,
        raw_topk: int = 300,
        labels_per_box: int = 1,
        query_batch_size: int | None = None,
    ) -> None:
        import torch
        from transformers import Owlv2ForObjectDetection, Owlv2Processor

        self._torch = torch
        self.vocab = vocab
        self.model_id = model_id
        self.device = device
        self.score_floor = score_floor
        self.raw_topk = raw_topk
        self.labels_per_box = labels_per_box
        self.query_batch_size = query_batch_size
        self.processor = Owlv2Processor.from_pretrained(model_id)
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self.model = Owlv2ForObjectDetection.from_pretrained(model_id, torch_dtype=dtype)
        self.model.to(device).eval()
        self.dtype = dtype

    def params(self) -> dict[str, Any]:
        return {
            "model_id": self.model_id,
            "n_queries": len(self.vocab.queries),
            "template": self.vocab.template,
            "score_floor": self.score_floor,
            "raw_topk": self.raw_topk,
            "labels_per_box": self.labels_per_box,
            "query_batch_size": self.query_batch_size,
        }

    def _scores_and_boxes(self, image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Corre el modelo: scores (cajas × queries) y cajas cxcywh normalizadas."""
        torch = self._torch
        prompts = self.vocab.prompts()
        step = self.query_batch_size or len(prompts)
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"]
        pixel_values = pixel_values.to(self.device, self.dtype)
        chunks, boxes = [], None
        for start in range(0, len(prompts), step):
            text = self.processor(text=[prompts[start : start + step]], return_tensors="pt")
            with torch.inference_mode():
                out = self.model(
                    input_ids=text["input_ids"].to(self.device),
                    attention_mask=text["attention_mask"].to(self.device),
                    pixel_values=pixel_values,
                )
            chunks.append(out.logits[0].sigmoid().float().cpu().numpy())
            boxes = out.pred_boxes[0].float().cpu().numpy()
        return np.concatenate(chunks, axis=1), boxes

    def detect(self, image: np.ndarray) -> list[Detection]:
        height, width = image.shape[:2]
        scores, boxes = self._scores_and_boxes(image)
        return decode_detections(
            scores,
            owl_boxes_to_xyxy(boxes, width, height),
            self.vocab,
            self.score_floor,
            self.raw_topk,
            self.labels_per_box,
            self.source,
        )


def build_detector(cfg: dict, vocab: Vocabulary, device: str) -> Detector:
    """Detector según `cfg['detection']['detector']`."""
    det_cfg = cfg["detection"]
    name = det_cfg["detector"]
    if name == "owlv2":
        o = det_cfg["owlv2"]
        return Owlv2Detector(
            vocab,
            model_id=o["model_id"],
            device=device,
            score_floor=o["score_floor"],
            raw_topk=o["raw_topk"],
            labels_per_box=o["labels_per_box"],
            query_batch_size=o.get("query_batch_size"),
        )
    raise ValueError(f"detection.detector desconocido: {name!r}")


In [ ]:
%%writefile src/scene/detection/postprocess.py
"""Post-proceso de detecciones: umbral, NMS por clase, NMS entre clases y tope por imagen.

Cada filtro informa cuántas detecciones elimina, para el embudo de la traza (E1-31).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

import numpy as np

from scene.detection.base import Detection


def box_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """IoU entre dos conjuntos de cajas xyxy (N×4 y M×4) → N×M."""
    a = boxes_a[:, None, :]
    b = boxes_b[None, :, :]
    iw = np.clip(np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]), 0, None)
    ih = np.clip(np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]), 0, None)
    inter = iw * ih
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    union = area_a + area_b - inter
    return np.where(union > 0, inter / np.where(union > 0, union, 1), 0.0)


def nms(boxes: np.ndarray, scores: np.ndarray, iou_thr: float) -> list[int]:
    """NMS greedy: índices que sobreviven, ordenados por score descendente."""
    order = np.argsort(-scores, kind="stable")
    keep: list[int] = []
    suppressed = np.zeros(len(boxes), dtype=bool)
    iou = box_iou_matrix(boxes, boxes)
    for i in order:
        if suppressed[i]:
            continue
        keep.append(int(i))
        suppressed |= iou[i] > iou_thr
    return keep


def _boxes(dets: list[Detection]) -> np.ndarray:
    return np.array([d.box_xyxy for d in dets], dtype=np.float64).reshape(-1, 4)


def _scores(dets: list[Detection]) -> np.ndarray:
    return np.array([d.score for d in dets], dtype=np.float64)


def apply_threshold(
    dets: list[Detection], threshold: float, per_class: dict[str, float] | None = None
) -> list[Detection]:
    """Umbral global; `per_class` (nombre de categoría → umbral) lo reemplaza para esa clase."""
    per_class = per_class or {}
    return [d for d in dets if d.score >= per_class.get(d.category_name, threshold)]


def per_class_nms(dets: list[Detection], iou_thr: float) -> list[Detection]:
    """NMS dentro de cada categoría. También une las cajas de sinónimos de una misma clase."""
    by_cat: dict[int, list[Detection]] = {}
    for d in dets:
        by_cat.setdefault(d.category_id, []).append(d)
    out: list[Detection] = []
    for group in by_cat.values():
        out.extend(group[i] for i in nms(_boxes(group), _scores(group), iou_thr))
    return sorted(out, key=lambda d: -d.score)


def cross_class_nms(dets: list[Detection], iou_thr: float) -> list[Detection]:
    """NMS ignorando la clase: ante cajas casi iguales queda la de mayor score."""
    if not dets:
        return []
    return [dets[i] for i in nms(_boxes(dets), _scores(dets), iou_thr)]


@dataclass
class PostprocessResult:
    detections: list[Detection]
    # Cantidad que sobrevive a cada paso, en orden: {"raw": 812, "threshold": 95, ...}.
    counts: dict[str, int] = field(default_factory=dict)


def postprocess(dets: list[Detection], det_cfg: dict[str, Any]) -> PostprocessResult:
    """Aplica los filtros de `cfg['detection']` en orden y registra cuántas quedan tras cada uno."""
    counts = {"raw": len(dets)}
    out = apply_threshold(dets, det_cfg["score_threshold"], det_cfg.get("per_class_threshold"))
    counts["threshold"] = len(out)
    out = per_class_nms(out, det_cfg["nms_iou"])
    counts["nms_per_class"] = len(out)
    cross = det_cfg.get("cross_class_nms") or {}
    if cross.get("enabled"):
        out = cross_class_nms(out, cross["iou"])
        counts["nms_cross_class"] = len(out)
    out = sorted(out, key=lambda d: -d.score)[: det_cfg["max_detections"]]
    counts["max_detections"] = len(out)
    return PostprocessResult(detections=out, counts=counts)


In [ ]:
%%writefile src/scene/detection/vocabulary.py
"""Vocabulario de detección: nombres limpios, sinónimos LVIS opcionales y mapeo query → categoría.

Solo usa el vocabulario de categorías (nombres y sinónimos públicos de LVIS), nunca anotaciones.
"""

from __future__ import annotations

import ast
import hashlib
import json
import re
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

from scene.io.dataset import Category

VOCAB_MODES = ("clean", "synonyms")


def clean_name(name: str) -> str:
    """`glass_(drink_container)` → `glass`: sin desambiguación, `_` → espacio, minúsculas."""
    name = re.sub(r"\([^)]*\)", " ", name.replace("_", " "))
    return " ".join(name.lower().split())


@dataclass(frozen=True)
class Query:
    """Texto que se le pasa al detector y la categoría a la que pertenece."""

    text: str
    category_id: int
    category_name: str
    is_synonym: bool = False


@dataclass
class Vocabulary:
    """Lista ordenada de queries. El índice de la query es el que devuelve el detector."""

    categories: list[Category]
    queries: list[Query]
    template: str = "{}"
    # Sinónimos descartados por colisionar entre categorías (para trazabilidad).
    dropped: dict[str, list[int]] = field(default_factory=dict)

    def prompts(self) -> list[str]:
        """Textos finales para el detector (query dentro del template)."""
        return [self.template.format(q.text) for q in self.queries]

    def query_to_category(self) -> list[int]:
        """category_id de cada query, en el mismo orden que `prompts()`."""
        return [q.category_id for q in self.queries]

    def by_category(self) -> dict[int, list[Query]]:
        out: dict[int, list[Query]] = {c.id: [] for c in self.categories}
        for q in self.queries:
            out[q.category_id].append(q)
        return out


def build_vocabulary(
    categories: list[Category],
    mode: str = "clean",
    synonyms: dict[int, list[str]] | None = None,
    template: str = "{}",
    max_synonyms: int | None = None,
) -> Vocabulary:
    """Arma las queries. En modo `synonyms` agrega sinónimos sin repetir textos entre categorías.

    Si un sinónimo coincide con el nombre de otra categoría o con un sinónimo de otra, se descarta
    en todas (así cada query mapea a una sola categoría).
    """
    if mode not in VOCAB_MODES:
        raise ValueError(f"vocab.mode inválido {mode!r}; opciones: {VOCAB_MODES}")
    main_names = {c.id: clean_name(c.name) for c in categories}
    if len(set(main_names.values())) != len(main_names):
        raise ValueError("Hay categorías con el mismo nombre limpio")

    extra: dict[int, list[str]] = {c.id: [] for c in categories}
    if mode == "synonyms":
        for cid in extra:
            seen = {main_names[cid]}
            for syn in (synonyms or {}).get(cid, []):
                text = clean_name(syn)
                if text and text not in seen:
                    seen.add(text)
                    extra[cid].append(text)
            if max_synonyms is not None:
                extra[cid] = extra[cid][:max_synonyms]

    # Quién usa cada texto: si lo usa más de una categoría, se descarta como sinónimo.
    owners: dict[str, set[int]] = {}
    for cid, name in main_names.items():
        owners.setdefault(name, set()).add(cid)
    for cid, texts in extra.items():
        for text in texts:
            owners.setdefault(text, set()).add(cid)
    dropped = {t: sorted(ids) for t, ids in owners.items() if len(ids) > 1}

    queries: list[Query] = []
    for cat in categories:
        queries.append(Query(main_names[cat.id], cat.id, cat.name))
        for text in extra[cat.id]:
            if text not in dropped:
                queries.append(Query(text, cat.id, cat.name, is_synonym=True))
    return Vocabulary(categories=categories, queries=queries, template=template, dropped=dropped)


def parse_lvis_categories(source: str) -> list[dict]:
    """Extrae `LVIS_CATEGORIES` de `lvis_v1_categories.py` (detectron2) sin ejecutar código."""
    for node in ast.parse(source).body:
        if isinstance(node, ast.Assign) and any(
            isinstance(t, ast.Name) and t.id == "LVIS_CATEGORIES" for t in node.targets
        ):
            return ast.literal_eval(node.value)
    raise ValueError("No se encontró LVIS_CATEGORIES en la fuente")


def match_lvis_synonyms(
    categories: list[Category], lvis_categories: list[dict], aliases: dict[str, str] | None = None
) -> dict[int, list[str]]:
    """Sinónimos LVIS de cada categoría de la competencia.

    Busca por: alias explícito (nombre LVIS crudo) → nombre LVIS limpio → algún sinónimo limpio.
    """
    aliases = aliases or {}
    by_raw = {c["name"]: c for c in lvis_categories}
    by_clean = {clean_name(c["name"]): c for c in lvis_categories}
    out: dict[int, list[str]] = {}
    missing: list[str] = []
    for cat in categories:
        name = clean_name(cat.name)
        entry = by_raw.get(aliases[name]) if name in aliases else by_clean.get(name)
        if entry is None:
            entry = next(
                (c for c in lvis_categories if name in {clean_name(s) for s in c["synonyms"]}),
                None,
            )
        if entry is None:
            missing.append(cat.name)
            continue
        out[cat.id] = list(entry["synonyms"])
    if missing:
        raise ValueError(f"Categorías sin entrada LVIS (agregar en vocab.lvis_aliases): {missing}")
    return out


def load_lvis_synonyms(vocab_cfg: dict, categories: list[Category]) -> dict[int, list[str]]:
    """Descarga (una vez) los sinónimos LVIS públicos y los cachea en JSON."""
    cache = Path(vocab_cfg["synonyms_cache"])
    if cache.is_file():
        data = json.loads(cache.read_text(encoding="utf-8"))
        return {int(k): v for k, v in data["synonyms"].items()}
    url = vocab_cfg["lvis_categories_url"]
    with urllib.request.urlopen(url, timeout=60) as resp:
        source = resp.read().decode("utf-8")
    synonyms = match_lvis_synonyms(
        categories, parse_lvis_categories(source), vocab_cfg.get("lvis_aliases")
    )
    cache.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "source_url": url,
        "source_sha256": hashlib.sha256(source.encode("utf-8")).hexdigest(),
        "synonyms": {str(k): v for k, v in synonyms.items()},
    }
    cache.write_text(json.dumps(payload, indent=1, ensure_ascii=False), encoding="utf-8")
    return synonyms


def vocabulary_from_config(cfg: dict, categories: list[Category]) -> Vocabulary:
    """Vocabulario según `cfg['vocab']` (descarga sinónimos solo en modo `synonyms`)."""
    vcfg = cfg["vocab"]
    synonyms = load_lvis_synonyms(vcfg, categories) if vcfg["mode"] == "synonyms" else None
    return build_vocabulary(
        categories,
        mode=vcfg["mode"],
        synonyms=synonyms,
        template=vcfg["prompt_template"],
        max_synonyms=vcfg.get("max_synonyms"),
    )


In [ ]:
from pathlib import Path

_ = Path('src/scene/io/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/io/dataset.py
"""Listado de imágenes, vocabulario y preguntas del benchmark.

Este módulo es parte de la inferencia: NO lee anotaciones de instancias. Las dimensiones
de cada imagen salen del propio archivo (no de la metadata de GT), así la pipeline puede
procesar una imagen arbitraria.
"""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from PIL import Image

from scene.config import data_path

SPLITS = ("dev", "test", "benchmark")


@dataclass(frozen=True)
class ImageRecord:
    """Imagen a procesar. `image_id` es el nombre del archivo sin extensión (p. ej. `tpi_0001`)."""

    image_id: str
    path: Path
    split: str
    width: int
    height: int


@dataclass(frozen=True)
class Category:
    """Categoría del vocabulario evaluable. Los sinónimos se agregan en E1-08."""

    id: int
    name: str
    synonyms: tuple[str, ...] = ()


def image_record_from_path(path: str | Path, split: str = "external") -> ImageRecord:
    """Arma un `ImageRecord` para cualquier imagen leyendo solo su cabecera."""
    path = Path(path)
    with Image.open(path) as img:
        width, height = img.size
    return ImageRecord(image_id=path.stem, path=path, split=split, width=width, height=height)


def load_image(record: ImageRecord) -> np.ndarray:
    """Carga la imagen como array RGB uint8 HxWx3 (sin rotar por EXIF, igual que COCO)."""
    with Image.open(record.path) as img:
        return np.asarray(img.convert("RGB"))


def load_vocabulary(cfg: dict) -> list[Category]:
    """Lee `categories.csv` (category_id, category_name) de la competencia."""
    with data_path(cfg, "categories").open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    return [Category(id=int(r["category_id"]), name=r["category_name"].strip()) for r in rows]


def load_questions(cfg: dict) -> list[dict]:
    """Lee el benchmark de preguntas (`questions_dev.jsonl`). No contiene respuestas."""
    with data_path(cfg, "questions").open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def _dev_images(cfg: dict) -> list[ImageRecord]:
    exts = {e.lower() for e in cfg["dataset"]["image_extensions"]}
    folder = data_path(cfg, "dev_images")
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in exts)
    return [image_record_from_path(p, split="dev") for p in files]


def _test_images(cfg: dict) -> list[ImageRecord]:
    folder = data_path(cfg, "test_images")
    with data_path(cfg, "test_index").open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    return [image_record_from_path(folder / r["file_name"], split="test") for r in rows]


def _benchmark_images(cfg: dict) -> list[ImageRecord]:
    """Imágenes de desarrollo referenciadas por el benchmark, en orden de aparición."""
    folder = data_path(cfg, "dev_images")
    seen: dict[str, None] = {}
    for q in load_questions(cfg):
        seen.setdefault(q["image_file"], None)
    return [image_record_from_path(folder / name, split="benchmark") for name in seen]


def list_images(cfg: dict, split: str) -> list[ImageRecord]:
    """Lista las imágenes de un split: `dev` (120), `test` (60) o `benchmark` (20 de dev)."""
    loaders = {"dev": _dev_images, "test": _test_images, "benchmark": _benchmark_images}
    if split not in loaders:
        raise ValueError(f"Split desconocido {split!r}; opciones: {SPLITS}")
    return loaders[split](cfg)


In [ ]:
%%writefile src/scene/io/submission.py
"""Armado y validación de `submission.csv`: columnas `image_id,SceneJSON`, mismo orden e ids
que `sample_submission.csv` (60 de test + 20 del benchmark)."""

from __future__ import annotations

import csv
import json
import sys
from pathlib import Path
from typing import Any

from scene.config import data_path
from scene.scene.schema import validate_scene

COLUMNS = ["image_id", "SceneJSON"]


def sample_ids(cfg: dict) -> list[str]:
    """Ids de `sample_submission.csv`, en orden."""
    with data_path(cfg, "sample_submission").open(encoding="utf-8", newline="") as f:
        return [row["image_id"] for row in csv.DictReader(f)]


def build_submission(
    scenes: dict[str, dict[str, Any]], cfg: dict
) -> tuple[list[tuple[str, str]], dict[str, list[str]]]:
    """Filas del CSV y errores por image_id (falta la escena o no valida)."""
    schema = data_path(cfg, "scene_schema")
    rows: list[tuple[str, str]] = []
    errors: dict[str, list[str]] = {}
    for iid in sample_ids(cfg):
        scene = scenes.get(iid)
        if scene is None:
            errors[iid] = ["falta el SceneJSON de esta imagen"]
            continue
        if scene["image"]["id"] != iid:
            errors[iid] = [f"image.id {scene['image']['id']!r} no coincide con la fila"]
            continue
        errs = validate_scene(scene, schema)
        if errs:
            errors[iid] = errs
            continue
        rows.append((iid, json.dumps(scene, separators=(",", ":"), ensure_ascii=False)))
    return rows, errors


def write_submission(rows: list[tuple[str, str]], path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Las celdas pueden pasar el límite por defecto del módulo csv al leerlas.
    csv.field_size_limit(sys.maxsize if sys.maxsize < 2**31 else 2**31 - 1)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(COLUMNS)
        writer.writerows(rows)
    return path


def read_submission(path: str | Path) -> dict[str, dict[str, Any]]:
    """Lee un submission y parsea cada SceneJSON (para re-validarlo)."""
    csv.field_size_limit(sys.maxsize if sys.maxsize < 2**31 else 2**31 - 1)
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != COLUMNS:
            raise ValueError(f"Columnas {reader.fieldnames}, se esperaba {COLUMNS}")
        return {row["image_id"]: json.loads(row["SceneJSON"]) for row in reader}


In [ ]:
%%writefile src/scene/llm/__init__.py
"""Modelos de lenguaje: carga, interpretación y descripción fiel."""

from scene.llm.describer import DescriptionResult, FidelityReport, SceneDescriber
from scene.llm.interpreter import InterpretationResult, Interpreter
from scene.llm.model import TransformersLLM, build_llm

__all__ = [
    "DescriptionResult",
    "FidelityReport",
    "InterpretationResult",
    "Interpreter",
    "SceneDescriber",
    "TransformersLLM",
    "build_llm",
]


In [ ]:
%%writefile src/scene/llm/describer.py
"""Descripción textual de un SceneJSON y verificación de fidelidad."""

from __future__ import annotations

import re
import unicodedata
from collections import Counter
from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from importlib import resources
from typing import Any

from scene.llm.model import TextGenerator

NUMBER_WORDS = {
    "cero": 0,
    "uno": 1,
    "una": 1,
    "un": 1,
    "dos": 2,
    "tres": 3,
    "cuatro": 4,
    "cinco": 5,
    "seis": 6,
    "siete": 7,
    "ocho": 8,
    "nueve": 9,
    "diez": 10,
    "once": 11,
    "doce": 12,
    "trece": 13,
    "catorce": 14,
    "quince": 15,
    "dieciseis": 16,
    "diecisiete": 17,
    "dieciocho": 18,
    "diecinueve": 19,
    "veinte": 20,
}


@dataclass(frozen=True)
class FidelityReport:
    """Afirmaciones comprobables encontradas en una descripción."""

    faithful: bool
    mentioned_categories: tuple[str, ...]
    unsupported_categories: tuple[str, ...]
    mentioned_numbers: tuple[int | float, ...]
    unsupported_numbers: tuple[int | float, ...]
    reasons: tuple[str, ...]


@dataclass(frozen=True)
class DescriptionResult:
    """Texto generado junto con el resumen fuente y su verificación."""

    text: str
    summary: str
    fidelity: FidelityReport
    attempts: int
    status: str


def _normalized(text: str) -> str:
    decomposed = unicodedata.normalize("NFKD", text.casefold())
    return "".join(char for char in decomposed if not unicodedata.combining(char))


def _object_category(obj: Mapping[str, Any], names: Mapping[int, str]) -> str:
    category = obj.get("category_name")
    if isinstance(category, str) and category:
        return category
    category_id = obj.get("category_id")
    return names.get(category_id, f"category_{category_id}")


def _object_color(obj: Mapping[str, Any]) -> str | None:
    for key in ("color", "dominant_color"):
        value = obj.get(key)
        if isinstance(value, str) and value:
            return value
        if isinstance(value, dict) and isinstance(value.get("value"), str):
            return value["value"]
    attributes = obj.get("attributes")
    if isinstance(attributes, dict):
        color = attributes.get("color")
        if isinstance(color, str):
            return color
        if isinstance(color, dict) and isinstance(color.get("value"), str):
            return color["value"]
    return None


def build_scene_summary(
    scene: Mapping[str, Any],
    *,
    category_names: Mapping[int, str],
    max_objects: int,
    max_relations: int,
) -> str:
    """Construye el único contenido de escena que recibe el LLM."""
    objects = list(scene.get("objects", []))
    relations = list(scene.get("relations", []))
    limited_objects = objects[:max_objects]
    all_names = [_object_category(obj, category_names) for obj in objects]
    object_names = {
        str(obj.get("id")): _object_category(obj, category_names) for obj in limited_objects
    }
    counts = Counter(all_names)
    lines = [f"Total de objetos: {len(objects)}", "Conteos por categoría:"]
    if counts:
        lines.extend(f"- {name}: {count}" for name, count in sorted(counts.items()))
    else:
        lines.append("- ninguno")

    lines.append("Objetos:")
    for obj in limited_objects:
        obj_id = str(obj.get("id"))
        fields = [
            f"id={obj_id}",
            f"categoría={object_names[obj_id]}",
            f"posición={obj.get('position', 'desconocida')}",
            f"tamaño={obj.get('size', 'desconocido')}",
        ]
        color = _object_color(obj)
        if color:
            fields.append(f"color={color}")
        lines.append("- " + "; ".join(fields))

    lines.append("Relaciones:")
    relation_count = 0
    for relation in relations:
        subject = str(relation.get("subject"))
        target = str(relation.get("object"))
        if subject not in object_names or target not in object_names:
            continue
        lines.append(
            f"- {object_names[subject]} ({subject}) {relation.get('predicate')} "
            f"{object_names[target]} ({target})"
        )
        relation_count += 1
        if relation_count >= max_relations:
            break
    if relation_count == 0:
        lines.append("- ninguna")
    return "\n".join(lines)


def _category_variants(name: str) -> tuple[str, ...]:
    normalized = _normalized(name)
    variants = {normalized, f"{normalized}s", f"{normalized}es"}
    return tuple(sorted(variants, key=len, reverse=True))


def _mentioned_categories(text: str, vocabulary: Iterable[str]) -> set[str]:
    normalized = _normalized(text)
    found: set[str] = set()
    for name in sorted(set(vocabulary), key=len, reverse=True):
        for variant in _category_variants(name):
            if re.search(rf"(?<!\w){re.escape(variant)}(?!\w)", normalized):
                found.add(name)
                break
    return found


def _mentioned_numbers(text: str) -> list[int | float]:
    normalized = _normalized(text)
    values: list[int | float] = []
    for raw in re.findall(r"(?<!\w)\d+(?:[.,]\d+)?(?!\w)", normalized):
        value = float(raw.replace(",", "."))
        values.append(int(value) if value.is_integer() else value)
    for word, value in NUMBER_WORDS.items():
        if re.search(rf"(?<!\w){word}(?!\w)", normalized):
            values.append(value)
    return values


def verify_description(
    description: str,
    scene: Mapping[str, Any],
    *,
    category_names: Mapping[int, str],
) -> FidelityReport:
    """Comprueba categorías del vocabulario y números contra el SceneJSON."""
    scene_categories = [_object_category(obj, category_names) for obj in scene.get("objects", [])]
    allowed_categories = set(scene_categories)
    vocabulary = set(category_names.values()) | allowed_categories
    mentioned_categories = _mentioned_categories(description, vocabulary)
    unsupported_categories = mentioned_categories - allowed_categories

    counts = Counter(scene_categories)
    allowed_numbers = {len(scene_categories), *counts.values()}
    mentioned_numbers = _mentioned_numbers(description)
    unsupported_numbers = [value for value in mentioned_numbers if value not in allowed_numbers]

    reasons: list[str] = []
    if not description.strip():
        reasons.append("descripción vacía")
    if unsupported_categories:
        reasons.append("categorías no presentes: " + ", ".join(sorted(unsupported_categories)))
    if unsupported_numbers:
        reasons.append("números no sustentados: " + ", ".join(map(str, unsupported_numbers)))
    return FidelityReport(
        faithful=not reasons,
        mentioned_categories=tuple(sorted(mentioned_categories)),
        unsupported_categories=tuple(sorted(unsupported_categories)),
        mentioned_numbers=tuple(mentioned_numbers),
        unsupported_numbers=tuple(unsupported_numbers),
        reasons=tuple(reasons),
    )


class SceneDescriber:
    """Genera una descripción sin exponer la imagen al modelo de lenguaje."""

    def __init__(
        self,
        generator: TextGenerator,
        *,
        category_names: Mapping[int, str],
        prompt_version: str = "describer_v1.txt",
        max_new_tokens: int,
        max_objects: int,
        max_relations: int,
        retry_on_unfaithful: int,
    ) -> None:
        self.generator = generator
        self.category_names = dict(category_names)
        self.prompt_template = (
            resources.files("scene.llm")
            .joinpath("prompts", prompt_version)
            .read_text(encoding="utf-8")
        )
        self.max_new_tokens = max_new_tokens
        self.max_objects = max_objects
        self.max_relations = max_relations
        self.retry_on_unfaithful = retry_on_unfaithful

    def describe(self, scene: Mapping[str, Any]) -> DescriptionResult:
        """Genera y, si hace falta, reintenta indicando las afirmaciones inválidas."""
        summary = build_scene_summary(
            scene,
            category_names=self.category_names,
            max_objects=self.max_objects,
            max_relations=self.max_relations,
        )
        base_prompt = self.prompt_template.replace("{{SCENE_SUMMARY}}", summary)
        prompt = base_prompt
        text = ""
        report = verify_description(text, scene, category_names=self.category_names)
        attempts = 1 + self.retry_on_unfaithful
        for attempt in range(1, attempts + 1):
            text = self.generator.generate(prompt, max_new_tokens=self.max_new_tokens).strip()
            report = verify_description(text, scene, category_names=self.category_names)
            if report.faithful:
                return DescriptionResult(text, summary, report, attempt, "ok")
            prompt = (
                base_prompt
                + "\n\nLa descripción anterior no fue fiel: "
                + "; ".join(report.reasons)
                + f"\nDescripción anterior: {text}\nCorregila usando solo el resumen."
            )
        return DescriptionResult(text, summary, report, attempts, "unfaithful")


def build_describer(
    cfg: dict,
    generator: TextGenerator,
    *,
    category_names: Mapping[int, str],
) -> SceneDescriber:
    """Construye el descriptor desde `llm.describer`."""
    describer_cfg = cfg["llm"]["describer"]
    return SceneDescriber(
        generator,
        category_names=category_names,
        prompt_version=describer_cfg["prompt_version"],
        max_new_tokens=describer_cfg["max_new_tokens"],
        max_objects=describer_cfg["max_objects"],
        max_relations=describer_cfg["max_relations"],
        retry_on_unfaithful=describer_cfg["retry_on_unfaithful"],
    )


In [ ]:
%%writefile src/scene/llm/interpreter.py
"""Interpretación de preguntas en texto al DSL validado de consultas."""

from __future__ import annotations

import json
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from importlib import resources
from typing import Any

from pydantic import ValidationError

from scene.llm.model import TextGenerator

CategoryMatcher = Callable[[str], int | None]


@dataclass(frozen=True)
class InterpretationResult:
    """Resultado del intérprete; `query` solo existe cuando el estado es `ok`."""

    status: str
    query: Any | None
    attempts: int
    raw_output: str
    errors: tuple[str, ...] = ()

    @property
    def interpreted_query(self) -> dict[str, Any]:
        """Versión serializable para `questions_llm[].interpreted_query`."""
        if self.query is None:
            return {"status": self.status, "errors": list(self.errors)}
        if hasattr(self.query, "model_dump"):
            return self.query.model_dump(mode="json")
        if isinstance(self.query, dict):
            return self.query
        raise TypeError("El modelo DSL no se puede serializar")


def _load_prompt(name: str) -> str:
    prompt = resources.files("scene.llm").joinpath("prompts", name)
    return prompt.read_text(encoding="utf-8")


def _first_json_object(text: str) -> dict[str, Any]:
    """Extrae el primer objeto JSON aun si el modelo agregó fences o texto."""
    decoder = json.JSONDecoder()
    for index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    raise ValueError("la salida no contiene un objeto JSON válido")


def _default_query_model() -> type:
    try:
        from scene.qa.dsl import Query
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            "E1-20 todavía no está disponible: pase query_model al Interpreter"
        ) from exc
    return Query


class Interpreter:
    """LLM -> JSON -> resolución de categorías -> validación pydantic."""

    def __init__(
        self,
        generator: TextGenerator,
        *,
        category_matcher: CategoryMatcher,
        category_names: Mapping[int, str],
        query_model: type | None = None,
        prompt_version: str = "interpreter_v1.txt",
        max_attempts: int = 2,
    ) -> None:
        if max_attempts < 1:
            raise ValueError("max_attempts debe ser al menos 1")
        self.generator = generator
        self.category_matcher = category_matcher
        self.category_names = dict(category_names)
        self.query_model = query_model or _default_query_model()
        self.prompt_template = _load_prompt(prompt_version)
        self.max_attempts = max_attempts

    def _prompt(self, question: str, previous: str | None, error: str | None) -> str:
        prompt = self.prompt_template.replace("{{QUESTION}}", question)
        if previous is not None:
            prompt += (
                "\n\nLa salida anterior fue inválida. Corregila y devolvé solo JSON."
                f"\nError de validación: {error}\nSalida anterior: {previous}"
            )
        return prompt

    def _canonical_category(self, mention: Any) -> str:
        if not isinstance(mention, str) or not mention.strip():
            raise ValueError("target.category debe ser texto no vacío")
        category_id = self.category_matcher(mention)
        if category_id is None:
            raise ValueError(f"categoría fuera del vocabulario: {mention!r}")
        if category_id not in self.category_names:
            raise ValueError(f"category matcher devolvió un id desconocido: {category_id}")
        return self.category_names[category_id]

    def _resolve_categories(self, payload: dict[str, Any]) -> dict[str, Any]:
        """E1-21 decide el id; el LLM solamente copia la mención de la pregunta."""
        resolved = json.loads(json.dumps(payload))
        for key in ("target", "target2"):
            target = resolved.get(key)
            if target is None:
                continue
            if not isinstance(target, dict):
                raise ValueError(f"{key} debe ser un objeto o null")
            if target.get("category") is not None:
                target["category"] = self._canonical_category(target["category"])
        return resolved

    def _validate(self, raw: str) -> Any:
        payload = _first_json_object(raw)
        if "unsupported_reason" in payload:
            raise UnsupportedQuery(str(payload["unsupported_reason"]))
        payload = self._resolve_categories(payload)
        validator = getattr(self.query_model, "model_validate", None)
        if validator is None:
            raise TypeError("query_model debe ser un modelo pydantic v2")
        return validator(payload)

    def interpret(self, question: str) -> InterpretationResult:
        """Interpreta una pregunta; reintenta una vez por defecto ante salida inválida."""
        if not question.strip():
            return InterpretationResult("unsupported", None, 0, "", ("pregunta vacía",))

        errors: list[str] = []
        previous: str | None = None
        for attempt in range(1, self.max_attempts + 1):
            raw = self.generator.generate(
                self._prompt(question, previous, errors[-1] if errors else None)
            )
            try:
                query = self._validate(raw)
            except UnsupportedQuery as exc:
                return InterpretationResult("unsupported", None, attempt, raw, (str(exc),))
            except (TypeError, ValueError, ValidationError) as exc:
                errors.append(str(exc))
                previous = raw
                continue
            return InterpretationResult("ok", query, attempt, raw, tuple(errors))
        return InterpretationResult(
            "unsupported", None, self.max_attempts, previous or "", tuple(errors)
        )


class UnsupportedQuery(ValueError):
    """El propio modelo marcó la consulta como no soportada."""


def build_interpreter(
    cfg: dict,
    generator: TextGenerator,
    *,
    category_matcher: CategoryMatcher,
    category_names: Mapping[int, str],
    query_model: type | None = None,
) -> Interpreter:
    """Construye el intérprete desde `llm.interpreter`."""
    interpreter_cfg = cfg["llm"]["interpreter"]
    return Interpreter(
        generator,
        category_matcher=category_matcher,
        category_names=category_names,
        query_model=query_model,
        prompt_version=interpreter_cfg["prompt_version"],
        max_attempts=interpreter_cfg["max_attempts"],
    )


In [ ]:
%%writefile src/scene/llm/model.py
"""Carga y medición reproducible de modelos de lenguaje de Hugging Face."""

from __future__ import annotations

import math
import statistics
import time
from dataclasses import asdict, dataclass
from typing import Any, Protocol

from scene.config import resolve_device, set_seed


class TextGenerator(Protocol):
    """Contrato mínimo usado por el intérprete y el descriptor."""

    def generate(self, prompt: str, *, max_new_tokens: int | None = None) -> str:
        """Genera texto de manera determinista para un prompt."""


@dataclass(frozen=True)
class GenerationMetrics:
    """Mediciones de una generación individual."""

    latency_ms: float
    input_tokens: int
    output_tokens: int
    peak_memory_mb: float | None


@dataclass(frozen=True)
class BenchmarkResult:
    """Resumen serializable del benchmark de un candidato."""

    candidate: str
    model_id: str
    precision: str
    device: str
    load_peak_memory_mb: float | None
    latency_mean_ms: float
    latency_p95_ms: float
    output_tokens_mean: float
    generation_peak_memory_mb: float | None
    samples: int

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


class TransformersLLM:
    """Wrapper lazy de un modelo causal con chat template."""

    def __init__(
        self,
        model_id: str,
        *,
        device: str,
        precision: str,
        seed: int,
        max_new_tokens: int,
        trust_remote_code: bool = False,
        chat_template_kwargs: dict[str, Any] | None = None,
    ) -> None:
        if precision not in {"fp16", "4bit"}:
            raise ValueError("llm.precision debe ser 'fp16' o '4bit'")
        self.model_id = model_id
        self.device = device
        self.precision = precision
        self.seed = seed
        self.max_new_tokens = max_new_tokens
        self.trust_remote_code = trust_remote_code
        self.chat_template_kwargs = chat_template_kwargs or {}
        self.model: Any | None = None
        self.tokenizer: Any | None = None
        self.last_metrics: GenerationMetrics | None = None
        self.load_peak_memory_mb: float | None = None
        self._torch: Any | None = None

    def load(self) -> TransformersLLM:
        """Descarga/carga el modelo. En 4 bits requiere CUDA y `bitsandbytes`."""
        if self.model is not None:
            return self

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        if self.precision == "4bit" and not self.device.startswith("cuda"):
            raise RuntimeError("La cuantización 4-bit requiere un dispositivo CUDA")

        set_seed(self.seed)
        self._torch = torch
        if self.device.startswith("cuda"):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        model_kwargs: dict[str, Any] = {"trust_remote_code": self.trust_remote_code}
        if self.precision == "4bit":
            try:
                from transformers import BitsAndBytesConfig
            except ImportError as exc:  # pragma: no cover - depende del entorno Kaggle
                raise RuntimeError("4-bit requiere transformers con BitsAndBytesConfig") from exc
            model_kwargs.update(
                {
                    "quantization_config": BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16,
                        bnb_4bit_use_double_quant=True,
                    ),
                    "device_map": {"": self.device},
                }
            )
        else:
            model_kwargs["torch_dtype"] = torch.float16

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id, trust_remote_code=self.trust_remote_code
        )
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, **model_kwargs)
        if self.precision == "fp16":
            self.model.to(self.device)
        self.model.eval()
        if self.device.startswith("cuda"):
            torch.cuda.synchronize()
            self.load_peak_memory_mb = torch.cuda.max_memory_allocated() / 2**20
        return self

    def _formatted_prompt(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            **self.chat_template_kwargs,
        )

    def generate(self, prompt: str, *, max_new_tokens: int | None = None) -> str:
        """Genera con greedy decoding (`do_sample=False`) y registra costo."""
        self.load()
        torch = self._torch
        set_seed(self.seed)
        formatted = self._formatted_prompt(prompt)
        inputs = self.tokenizer(formatted, return_tensors="pt")
        inputs = {name: tensor.to(self.device) for name, tensor in inputs.items()}
        if self.device.startswith("cuda"):
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens or self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        if self.device.startswith("cuda"):
            torch.cuda.synchronize()
        latency_ms = (time.perf_counter() - started) * 1000
        input_tokens = int(inputs["input_ids"].shape[-1])
        generated = output[0, input_tokens:]
        peak = torch.cuda.max_memory_allocated() / 2**20 if self.device.startswith("cuda") else None
        self.last_metrics = GenerationMetrics(
            latency_ms=latency_ms,
            input_tokens=input_tokens,
            output_tokens=int(generated.shape[-1]),
            peak_memory_mb=peak,
        )
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def close(self) -> None:
        """Libera referencias y la caché CUDA entre candidatos del benchmark."""
        self.model = None
        self.tokenizer = None
        if self._torch is not None and self.device.startswith("cuda"):
            self._torch.cuda.empty_cache()


def build_llm(
    cfg: dict, *, device: str | None = None, candidate: str | None = None
) -> TransformersLLM:
    """Construye el candidato configurado sin cargar todavía sus pesos."""
    llm_cfg = cfg["llm"]
    candidate = candidate or llm_cfg["selected"]
    if candidate not in llm_cfg["candidates"]:
        raise ValueError(f"Candidato LLM desconocido: {candidate!r}")
    candidate_cfg = llm_cfg["candidates"][candidate]
    return TransformersLLM(
        candidate_cfg["model_id"],
        device=device or resolve_device(cfg.get("device", "auto")),
        precision=candidate_cfg["precision"],
        seed=llm_cfg.get("seed", cfg["seed"]),
        max_new_tokens=llm_cfg["max_new_tokens"],
        trust_remote_code=llm_cfg["trust_remote_code"],
        chat_template_kwargs=candidate_cfg.get("chat_template_kwargs"),
    )


def _percentile_95(values: list[float]) -> float:
    ordered = sorted(values)
    index = max(0, math.ceil(0.95 * len(ordered)) - 1)
    return ordered[index]


def benchmark_llm(
    model: TransformersLLM,
    prompts: list[str],
    *,
    candidate: str,
    warmup: int,
    repeats: int,
) -> BenchmarkResult:
    """Mide latencia y memoria. El llamador debe ejecutarlo en una T4."""
    if not prompts:
        raise ValueError("El benchmark necesita al menos un prompt")
    if warmup < 0 or repeats < 1:
        raise ValueError("warmup debe ser >= 0 y repeats >= 1")
    model.load()
    for index in range(warmup):
        model.generate(prompts[index % len(prompts)])

    samples: list[GenerationMetrics] = []
    for index in range(repeats):
        model.generate(prompts[index % len(prompts)])
        if model.last_metrics is None:  # pragma: no cover - protege el contrato interno
            raise RuntimeError("El generador no registró las métricas")
        samples.append(model.last_metrics)

    peaks = [sample.peak_memory_mb for sample in samples if sample.peak_memory_mb is not None]
    latencies = [sample.latency_ms for sample in samples]
    return BenchmarkResult(
        candidate=candidate,
        model_id=model.model_id,
        precision=model.precision,
        device=model.device,
        load_peak_memory_mb=model.load_peak_memory_mb,
        latency_mean_ms=statistics.fmean(latencies),
        latency_p95_ms=_percentile_95(latencies),
        output_tokens_mean=statistics.fmean(sample.output_tokens for sample in samples),
        generation_peak_memory_mb=max(peaks) if peaks else None,
        samples=len(samples),
    )


In [ ]:
%%writefile src/scene/llm/prompts/describer_v1.txt
Redactá una descripción breve en español usando exclusivamente el resumen estructurado.
No agregues objetos, cantidades, atributos, relaciones, intenciones ni contexto.
Conservá literalmente los nombres de categoría del resumen, aunque estén en inglés.
No menciones ids, coordenadas, porcentajes ni confianzas.
Si no hay objetos, indicá solamente que no se detectaron objetos.
Devolvé solo la descripción, sin títulos ni explicaciones.

Resumen estructurado:
{{SCENE_SUMMARY}}

Descripción:


In [ ]:
%%writefile src/scene/llm/prompts/interpreter_v1.txt
Convertí la pregunta a un único objeto JSON del DSL. No respondas la pregunta.
Recibís solamente texto: no supongas qué hay en una imagen.
Copiá las categorías tal como aparecen en la pregunta; otro componente las normaliza.
Usá null cuando un campo no aplique. No agregues claves.

Esquema:
{"op":"count|exists|position|size|relation|attribute|describe","target":{"category":"texto|null","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int|yes_no|region|size_label|text"}

Ejemplos:
Pregunta: ¿Cuántas tazas aparecen?
JSON: {"op":"count","target":{"category":"tazas","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int"}

Pregunta: ¿Hay un reloj?
JSON: {"op":"exists","target":{"category":"reloj","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"yes_no"}

Pregunta: ¿En qué zona está la mochila?
JSON: {"op":"position","target":{"category":"mochila","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"region"}

Pregunta: ¿Qué tamaño relativo tiene el collar?
JSON: {"op":"size","target":{"category":"collar","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"size_label"}

Pregunta: ¿El reloj está por encima del anillo?
JSON: {"op":"relation","target":{"category":"reloj","filters":{"color":null,"position":null,"size":null}},"target2":{"category":"anillo","filters":{"color":null,"position":null,"size":null}},"relation":"above","attribute":null,"output_type":"yes_no"}

Pregunta: ¿De qué color es la taza?
JSON: {"op":"attribute","target":{"category":"taza","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":"color","output_type":"text"}

Pregunta: ¿Dónde está la taza?
JSON: {"op":"position","target":{"category":"taza","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"region"}
Nota: si hubiera varias tazas, el ejecutor marcará la referencia como ambigua.

Pregunta: ¿Por qué la persona está contenta?
JSON: {"unsupported_reason":"requiere inferir una causa no representada"}

Pregunta: {{QUESTION}}
JSON:


In [ ]:
%%writefile src/scene/llm/prompts/interpreter_v2.txt
Convertí la pregunta a un único objeto JSON del DSL. No respondas la pregunta.
Recibís solamente texto: no supongas qué hay en una imagen.
Copiá las categorías tal como aparecen en la pregunta; otro componente las normaliza.
Usá null cuando un campo no aplique. No agregues claves.

Esquema:
{"op":"count|exists|position|size|relation|attribute|describe","target":{"category":"texto|null","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int|yes_no|region|size_label|text"}
Relaciones posibles: left_of, right_of, above, below, overlaps, near, on_top_of.

Ejemplos:
Pregunta: ¿Cuántas tazas aparecen?
JSON: {"op":"count","target":{"category":"tazas","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int"}

Pregunta: ¿Cuántas instancias de la categoría "libro" aparecen en la imagen?
JSON: {"op":"count","target":{"category":"libro","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int"}

Pregunta: ¿Hay algún paraguas?
JSON: {"op":"exists","target":{"category":"paraguas","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"yes_no"}

Pregunta: ¿Hay al menos una instancia de la categoría "gato" en la imagen?
JSON: {"op":"exists","target":{"category":"gato","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"yes_no"}

Pregunta: ¿En qué zona de la imagen está la instancia de "lámpara"?
JSON: {"op":"position","target":{"category":"lámpara","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"region"}

Pregunta: ¿Qué tamaño relativo tiene la silla?
JSON: {"op":"size","target":{"category":"silla","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"size_label"}

Pregunta: ¿La instancia de "lámpara" está por encima de la instancia de "silla"?
JSON: {"op":"relation","target":{"category":"lámpara","filters":{"color":null,"position":null,"size":null}},"target2":{"category":"silla","filters":{"color":null,"position":null,"size":null}},"relation":"above","attribute":null,"output_type":"yes_no"}

Pregunta: ¿El auto está a la izquierda del gato?
JSON: {"op":"relation","target":{"category":"auto","filters":{"color":null,"position":null,"size":null}},"target2":{"category":"gato","filters":{"color":null,"position":null,"size":null}},"relation":"left_of","attribute":null,"output_type":"yes_no"}

Pregunta: ¿Cuántas tazas rojas hay?
JSON: {"op":"count","target":{"category":"tazas","filters":{"color":"rojo","position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"int"}

Pregunta: ¿De qué color es el paraguas?
JSON: {"op":"attribute","target":{"category":"paraguas","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":"color","output_type":"text"}

Pregunta: Hacé un resumen de lo que se ve.
JSON: {"op":"describe","target":null,"target2":null,"relation":null,"attribute":null,"output_type":"text"}

Pregunta: ¿Dónde está la taza?
JSON: {"op":"position","target":{"category":"taza","filters":{"color":null,"position":null,"size":null}},"target2":null,"relation":null,"attribute":null,"output_type":"region"}
Nota: si hubiera varias tazas, el ejecutor marcará la referencia como ambigua.

Pregunta: ¿Por qué la persona está contenta?
JSON: {"unsupported_reason":"requiere inferir una causa no representada"}

Pregunta: {{QUESTION}}
JSON:


In [ ]:
from pathlib import Path

_ = Path('src/scene/masks/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/masks/rle.py
"""RLE no comprimido en orden column-major (formato COCO), tal como lo pide el schema.

Formato: {"size": [H, W], "counts": [c0, c1, ...]}. Se recorre la máscara por columnas
(`flatten(order="F")`) y las corridas alternan ceros y unos, empezando siempre por ceros
(la primera corrida puede valer 0 si el primer píxel es 1).
"""

from __future__ import annotations

import numpy as np


def encode(mask: np.ndarray) -> dict:
    """Codifica una máscara binaria HxW en RLE no comprimido column-major."""
    mask = np.asarray(mask)
    if mask.ndim != 2 or mask.size == 0:
        raise ValueError(f"Se espera una máscara 2D no vacía, llegó shape={mask.shape}")
    h, w = mask.shape
    flat = mask.astype(bool).ravel(order="F")
    # Posiciones donde cambia el valor → límites de cada corrida.
    changes = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    bounds = np.concatenate(([0], changes, [flat.size]))
    counts = np.diff(bounds).tolist()
    if flat[0]:
        counts = [0, *counts]
    return {"size": [int(h), int(w)], "counts": [int(c) for c in counts]}


def decode(rle: dict) -> np.ndarray:
    """Decodifica un RLE no comprimido a una máscara bool HxW."""
    h, w = (int(v) for v in rle["size"])
    counts = np.asarray(rle["counts"], dtype=np.int64)
    if counts.ndim != 1 or counts.size == 0 or (counts < 0).any():
        raise ValueError("counts debe ser una lista no vacía de enteros >= 0")
    if counts.sum() != h * w:
        raise ValueError(f"La suma de counts ({counts.sum()}) no coincide con H*W ({h * w})")
    values = np.arange(counts.size) % 2 == 1  # corridas pares = 0, impares = 1
    flat = np.repeat(values, counts)
    return flat.reshape((h, w), order="F")


def area(rle: dict) -> int:
    """Cantidad de píxeles en 1 sin decodificar (suma de las corridas impares)."""
    return int(sum(rle["counts"][1::2]))


In [ ]:
%%writefile src/scene/pipeline.py
"""Orquestador end-to-end: detección → post-proceso → SAM2 → SceneJSON, con traza y caché.

Caché por etapa en `paths.cache`, con una clave que es el hash de la parte de la config que
afecta a esa etapa (y a las anteriores). Así, cambiar el umbral reutiliza las detecciones
crudas, y cambiar las relaciones reutiliza también las máscaras.
"""

from __future__ import annotations

import json
from collections.abc import Callable
from pathlib import Path
from typing import Any

from scene.config import RunContext, config_hash
from scene.detection.base import load_detections, save_detections
from scene.detection.debug import draw_detections
from scene.detection.postprocess import postprocess
from scene.detection.vocabulary import Vocabulary
from scene.io.dataset import ImageRecord, load_image
from scene.scene.builder import build_scene
from scene.segmentation.debug import draw_masks
from scene.segmentation.sam2 import load_segmented, save_segmented
from scene.tracing.tracer import Tracer

STAGES = ("detection", "segmentation", "scene")


def stage_keys(cfg: dict) -> dict[str, str]:
    """Clave de caché de cada etapa: hash de su config y la de las etapas previas."""
    det = cfg["detection"]
    raw_part = {"detector": det["detector"], det["detector"]: det[det["detector"]]}
    raw_part["vocab"] = cfg["vocab"]
    pp_part = {k: v for k, v in det.items() if k not in ("detector", det["detector"])}
    seg_part = {"raw": raw_part, "pp": pp_part, "segmentation": cfg["segmentation"]}
    return {
        "detection": config_hash(raw_part)[:12],
        "segmentation": config_hash(seg_part)[:12],
    }


class Pipeline:
    """Corre la pipeline por imagen. Los modelos se cargan recién cuando hacen falta."""

    def __init__(
        self,
        cfg: dict,
        ctx: RunContext,
        tracer: Tracer,
        vocab: Vocabulary,
        make_detector: Callable[[], Any],
        make_segmenter: Callable[[], Any],
        from_stage: str | None = None,
        debug: bool = False,
    ) -> None:
        if from_stage is not None and from_stage not in STAGES:
            raise ValueError(f"--from-stage inválido {from_stage!r}; opciones: {STAGES}")
        self.cfg = cfg
        self.ctx = ctx
        self.tracer = tracer
        self.vocab = vocab
        self._make_detector, self._make_segmenter = make_detector, make_segmenter
        self._detector = self._segmenter = None
        # Las etapas desde `from_stage` en adelante ignoran la caché.
        self.force = set(STAGES[STAGES.index(from_stage) :]) if from_stage else set()
        self.debug = debug
        keys = stage_keys(cfg)
        cache = Path(cfg["paths"]["cache"])
        self.raw_dir = cache / "detections_raw" / keys["detection"]
        self.mask_dir = cache / "masks" / keys["segmentation"]
        self.scene_dir = ctx.run_dir / "scenes"
        self.models: dict[str, Any] = {}

    @property
    def detector(self):
        if self._detector is None:
            self._detector = self._make_detector()
            self.models["detector"] = self._detector.params()
            self.ctx.update_manifest("models", self.models)
        return self._detector

    @property
    def segmenter(self):
        if self._segmenter is None:
            self._segmenter = self._make_segmenter()
            self.models["segmenter"] = self._segmenter.params()
            self.ctx.update_manifest("models", self.models)
        return self._segmenter

    def _use_cache(self, stage: str, path: Path) -> bool:
        return stage not in self.force and path.is_file()

    def process(self, rec: ImageRecord) -> dict[str, Any]:
        """SceneJSON de una imagen (lo guarda en `scenes/<image_id>.json`)."""
        iid = rec.image_id
        image = load_image(rec)
        det_cfg = self.cfg["detection"]

        raw_path = self.raw_dir / f"{iid}.json"
        cached = self._use_cache("detection", raw_path)
        model_id = None if cached else self.detector.model_id
        params = {} if cached else self.detector.params()
        with self.tracer.stage(iid, "detection", params, model_id, n_in=1) as s:
            if cached:
                raw = load_detections(raw_path)
                s.status = "cached"
            else:
                raw = self.detector.detect(image)
                save_detections(raw, raw_path)
            s.n_out, s.artifact_path = len(raw), str(raw_path)

        pp_params = {k: v for k, v in det_cfg.items() if k not in ("detector", det_cfg["detector"])}
        with self.tracer.stage(iid, "detection_postprocess", pp_params, n_in=len(raw)) as s:
            res = postprocess(raw, det_cfg)
            s.n_out = len(res.detections)
            s.extra["counts"] = res.counts
        dets = res.detections

        mask_path = self.mask_dir / f"{iid}.json"
        cached = self._use_cache("segmentation", mask_path)
        model_id = None if cached else self.segmenter.model_id
        params = {} if cached else self.segmenter.params()
        with self.tracer.stage(iid, "segmentation", params, model_id, n_in=len(dets)) as s:
            if cached:
                objs = load_segmented(mask_path)
                s.status = "cached"
            else:
                objs = self.segmenter.segment(image, dets)
                save_segmented(objs, mask_path)
            s.n_out, s.artifact_path = len(objs), str(mask_path)
            s.extra["empty_masks"] = sum(1 for o in objs if not o.mask.any())

        scene_path = self.scene_dir / f"{iid}.json"
        with self.tracer.stage(iid, "scene", self.cfg["scene"]["relations"], n_in=len(objs)) as s:
            scene, stats = build_scene(iid, image, objs, self.cfg)
            scene_path.parent.mkdir(parents=True, exist_ok=True)
            scene_path.write_text(json.dumps(scene, separators=(",", ":")), encoding="utf-8")
            s.n_out, s.artifact_path = len(scene["objects"]), str(scene_path)
            s.extra.update(stats)

        if self.debug:
            draw_detections(image, dets, self.ctx.run_dir / "debug" / f"{iid}_boxes.png")
            draw_masks(image, objs, self.ctx.run_dir / "debug" / f"{iid}_masks.png")
        return scene

    def run(self, images: list[ImageRecord]) -> dict[str, dict[str, Any]]:
        """Procesa todas las imágenes; una que falla queda en la traza y el run sigue."""
        scenes: dict[str, dict[str, Any]] = {}
        for rec in images:
            with self.tracer.image_guard(rec.image_id):
                scenes[rec.image_id] = self.process(rec)
        return scenes


def load_scenes(run_dir: str | Path) -> dict[str, dict[str, Any]]:
    """Lee los SceneJSON de un run (`scenes/*.json`)."""
    return {
        p.stem: json.loads(p.read_text(encoding="utf-8"))
        for p in sorted(Path(run_dir, "scenes").glob("*.json"))
    }


In [ ]:
%%writefile src/scene/qa/__init__.py
"""Consultas y respuestas sobre SceneJSON."""

from scene.qa.verbalize import verbalize

__all__ = ["verbalize"]


In [ ]:
%%writefile src/scene/qa/category_matcher.py
"""Category matcher: texto (inglés o español) → category_id del vocabulario, o `None`.

Orden de búsqueda (PLAN §4.5): para cada forma del texto (tal cual y en singular) se prueba
nombre del vocabulario → léxico español → sinónimos LVIS. Si nada coincide, como último recurso,
similitud de embeddings con umbral. Sin match → `None` (la pregunta queda `unsupported`).
Solo usa el vocabulario y recursos públicos, nunca anotaciones.
"""

from __future__ import annotations

import itertools
import re
import unicodedata
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import yaml

from scene.detection.vocabulary import clean_name, load_lvis_synonyms
from scene.io.dataset import Category

# Una función que recibe textos y devuelve sus embeddings normalizados (una fila por texto).
Embedder = Callable[[list[str]], np.ndarray]

LEADING_ARTICLES = {"the", "a", "an", "el", "la", "los", "las", "un", "una", "unos", "unas"}
IRREGULAR_PLURALS = {
    "people": "person",
    "men": "man",
    "women": "woman",
    "children": "child",
    "mice": "mouse",
    "feet": "foot",
    "teeth": "tooth",
}
# Sufijo de plural → reemplazos posibles (inglés y español). Se prueban todos; solo vale el
# que coincide con alguna entrada del índice, así que generar formas de más no rompe nada.
PLURAL_SUFFIXES = (
    ("ies", ("y",)),
    ("ves", ("f", "fe")),
    ("ces", ("z",)),
    ("es", ("",)),
    ("s", ("",)),
)
MIN_WORD_FOR_SINGULAR = 4
MAX_CANDIDATES = 64


def normalize_text(text: str) -> str:
    """Minúsculas, sin acentos (ñ → n) ni signos, espacios simples y sin artículo inicial."""
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    words = re.sub(r"[^a-z0-9]+", " ", text).split()
    if len(words) > 1 and words[0] in LEADING_ARTICLES:
        words = words[1:]
    return " ".join(words)


def _singular_options(word: str) -> list[str]:
    """La palabra tal cual y sus posibles singulares."""
    options = [word]
    if word in IRREGULAR_PLURALS:
        options.append(IRREGULAR_PLURALS[word])
    elif len(word) >= MIN_WORD_FOR_SINGULAR:
        for suffix, replacements in PLURAL_SUFFIXES:
            if word.endswith(suffix):
                options += [word[: -len(suffix)] + r for r in replacements]
    return list(dict.fromkeys(options))


def singular_candidates(text: str) -> list[str]:
    """Formas a buscar: primero el texto tal cual, después combinaciones con palabras en singular.

    Se combinan todas las palabras porque en español el plural puede ir en cualquiera
    ("raquetas de tenis", "teléfonos celulares").
    """
    words = normalize_text(text).split()
    if not words:
        return []
    combos = itertools.product(*(_singular_options(w) for w in words))
    return [" ".join(c) for c in itertools.islice(combos, MAX_CANDIDATES)]


@dataclass(frozen=True)
class CategoryMatch:
    category_id: int
    category_name: str
    method: str  # name | lexicon_es | lvis_synonym | embedding
    score: float  # 1.0 en los matches exactos; similitud coseno en `embedding`
    matched_text: str  # entrada del índice que coincidió


@dataclass
class CategoryMatcher:
    """Índices por nivel de confianza; dentro de un nivel, un texto ambiguo se descarta."""

    categories: list[Category]
    lexicon: dict[str, list[str]] = field(default_factory=dict)  # nombre en inglés → términos
    synonyms: dict[int, list[str]] = field(default_factory=dict)  # category_id → sinónimos LVIS
    embedder: Embedder | None = None
    embedding_threshold: float = 1.0
    # Textos que apuntaban a más de una categoría dentro de un nivel (para trazabilidad).
    dropped: dict[str, list[int]] = field(default_factory=dict, init=False)

    def __post_init__(self) -> None:
        self._names = {c.id: c.name for c in self.categories}
        by_name = {normalize_text(clean_name(c.name)): c.id for c in self.categories}
        unknown = [n for n in self.lexicon if normalize_text(n) not in by_name]
        if unknown:
            raise ValueError(
                f"El léxico tiene categorías que no están en el vocabulario: {unknown}"
            )

        raw_tiers = {
            "name": [(c.name, c.id) for c in self.categories],
            "lexicon_es": [
                (term, by_name[normalize_text(name)])
                for name, terms in self.lexicon.items()
                for term in terms
            ],
            "lvis_synonym": [(syn, cid) for cid, syns in self.synonyms.items() for syn in syns],
        }
        self._tiers: list[tuple[str, dict[str, int]]] = []
        self._display: dict[str, str] = {}  # clave normalizada → texto original (embeddings)
        for method, entries in raw_tiers.items():
            owners: dict[str, set[int]] = {}
            for text, cid in entries:
                key = normalize_text(clean_name(text))
                if key:
                    owners.setdefault(key, set()).add(cid)
                    self._display.setdefault(key, clean_name(text))
            index = {k: next(iter(ids)) for k, ids in owners.items() if len(ids) == 1}
            for k, ids in owners.items():
                if len(ids) > 1:
                    self.dropped[f"{method}:{k}"] = sorted(ids)
            self._tiers.append((method, index))
        self._emb_keys: list[tuple[str, int]] | None = None
        self._emb_matrix: np.ndarray | None = None

    def match(self, text: str) -> CategoryMatch | None:
        """Categoría para `text`, o `None` si no hay match confiable."""
        for candidate in singular_candidates(text):
            for method, index in self._tiers:
                if candidate in index:
                    cid = index[candidate]
                    return CategoryMatch(cid, self._names[cid], method, 1.0, candidate)
        return self._match_embedding(text)

    def match_id(self, text: str) -> int | None:
        found = self.match(text)
        return found.category_id if found else None

    def _match_embedding(self, text: str) -> CategoryMatch | None:
        if self.embedder is None or not normalize_text(text):
            return None
        if self._emb_matrix is None:
            # Una entrada por clave: la del nivel más confiable donde aparece.
            seen: dict[str, int] = {}
            for _, index in self._tiers:
                for key, cid in index.items():
                    seen.setdefault(key, cid)
            self._emb_keys = list(seen.items())
            self._emb_matrix = self.embedder([self._display[k] for k, _ in self._emb_keys])
        query = self.embedder([text.strip().lower()])[0]
        sims = self._emb_matrix @ query
        best = int(np.argmax(sims))
        if float(sims[best]) < self.embedding_threshold:
            return None
        key, cid = self._emb_keys[best]
        return CategoryMatch(cid, self._names[cid], "embedding", float(sims[best]), key)


class HFTextEmbedder:
    """Embeddings de oraciones con `transformers` (mean pooling + normalización L2).

    El modelo se carga en el primer uso: si el léxico resuelve todo, no se descarga.
    """

    def __init__(self, model_id: str, device: str = "cpu", batch_size: int = 64) -> None:
        self.model_id = model_id
        self.device = device
        self.batch_size = batch_size
        self.model = None

    def _load(self) -> None:
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModel.from_pretrained(self.model_id).to(self.device).eval()

    def __call__(self, texts: list[str]) -> np.ndarray:
        import torch

        if self.model is None:
            self._load()
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch = self.tokenizer(
                    texts[i : i + self.batch_size],
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                ).to(self.device)
                hidden = self.model(**batch).last_hidden_state
                mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
                emb = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
                out.append(torch.nn.functional.normalize(emb, dim=1).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, 0), dtype=np.float32)


def load_lexicon(path: str | Path) -> dict[str, list[str]]:
    with Path(path).open(encoding="utf-8") as f:
        return {str(k): [str(t) for t in v] for k, v in (yaml.safe_load(f) or {}).items()}


def spanish_names(categories: list[Category], lexicon: dict[str, list[str]]) -> dict[int, str]:
    """Nombre en español de cada categoría (primer término del léxico; si no hay, el original)."""
    return {c.id: (lexicon.get(c.name) or [c.name])[0] for c in categories}


def matcher_from_config(
    cfg: dict, categories: list[Category], device: str = "cpu"
) -> CategoryMatcher:
    """Matcher según `cfg['qa']['matcher']` (sinónimos LVIS vía `cfg['vocab']`)."""
    mcfg = cfg["qa"]["matcher"]
    lexicon = load_lexicon(mcfg["lexicon_es"]) if mcfg.get("lexicon_es") else {}
    synonyms = load_lvis_synonyms(cfg["vocab"], categories) if mcfg["lvis_synonyms"] else {}
    ecfg = mcfg["embeddings"]
    embedder = (
        HFTextEmbedder(ecfg["model_id"], device=device, batch_size=ecfg["batch_size"])
        if ecfg["enabled"]
        else None
    )
    return CategoryMatcher(
        categories,
        lexicon=lexicon,
        synonyms=synonyms,
        embedder=embedder,
        embedding_threshold=ecfg["threshold"],
    )


In [ ]:
%%writefile src/scene/qa/dsl.py
"""DSL de consultas (PLAN §3) y resultado del ejecutor.

Lo usan los dos modos de QA: el adapter determinista (campos estructurados) y el intérprete LLM
(solo texto). Las categorías van como texto; las resuelve `qa.category_matcher` en código.
"""

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

from scene.scene.schema import POSITIONS, SIZES

# `Literal[tupla]` equivale a `Literal["a", "b", ...]`: así los enums salen del schema.
Position = Literal[POSITIONS]
Size = Literal[SIZES]
Op = Literal[
    "count", "exists", "position", "size", "relation", "attribute", "describe", "unsupported"
]
Relation = Literal["left_of", "right_of", "above", "below", "near", "overlaps", "on_top_of"]
Attribute = Literal["color"]
OutputType = Literal["int", "yes_no", "region", "size_label", "text"]
Status = Literal["ok", "unsupported", "insufficient_information", "ambiguous"]

# Tipo de salida de cada op (el `output_type` de una consulta tiene que coincidir).
OUTPUT_TYPE_BY_OP: dict[str, str] = {
    "count": "int",
    "exists": "yes_no",
    "position": "region",
    "size": "size_label",
    "relation": "yes_no",
    "attribute": "text",
    "describe": "text",
    "unsupported": "text",
}
# Tipo de pregunta del benchmark → op, y `answer_format` → output_type.
QUESTION_TYPE_TO_OP: dict[str, str] = {
    "count": "count",
    "existence": "exists",
    "position": "position",
    "size": "size",
    "relation": "relation",
    "description": "describe",
}
ANSWER_FORMAT_TO_OUTPUT: dict[str, str] = {
    "integer": "int",
    "boolean": "yes_no",
    "position_3x3": "region",
    "size_class": "size_label",
    "free_text": "text",
}


class Filters(BaseModel):
    """Filtros opcionales sobre las instancias de la categoría (`null` = sin filtro)."""

    model_config = ConfigDict(extra="forbid")

    color: str | None = None
    position: Position | None = None
    size: Size | None = None

    def active(self) -> dict[str, str]:
        return {k: v for k, v in self.model_dump().items() if v is not None}


class Target(BaseModel):
    """Objeto al que se refiere la pregunta: texto de la categoría más filtros."""

    model_config = ConfigDict(extra="forbid")

    category: str = Field(min_length=1)
    filters: Filters = Field(default_factory=Filters)


class Query(BaseModel):
    """Consulta ejecutable sobre un SceneJSON.

    `relation` y `target2` solo van con `op="relation"`; `attribute` solo con `op="attribute"`.
    `unsupported` lo usa el intérprete cuando la pregunta no entra en ninguna op (con `reason`).
    """

    model_config = ConfigDict(extra="forbid")

    op: Op
    target: Target | None = None
    target2: Target | None = None
    relation: Relation | None = None
    attribute: Attribute | None = None
    output_type: OutputType | None = None
    reason: str | None = None

    @model_validator(mode="after")
    def _check_op(self) -> Query:
        if self.op in ("describe", "unsupported"):
            if self.target is not None or self.target2 is not None:
                raise ValueError(f"op={self.op!r} no lleva target")
        elif self.target is None:
            raise ValueError(f"op={self.op!r} necesita target")
        if (self.op == "relation") != (self.relation is not None):
            raise ValueError("relation va si y solo si op='relation'")
        if (self.op == "relation") != (self.target2 is not None):
            raise ValueError("target2 va si y solo si op='relation'")
        if (self.op == "attribute") != (self.attribute is not None):
            raise ValueError("attribute va si y solo si op='attribute'")
        expected = OUTPUT_TYPE_BY_OP[self.op]
        if self.output_type is None:
            self.output_type = expected
        elif self.output_type != expected:
            raise ValueError(f"output_type {self.output_type!r} no corresponde a op={self.op!r}")
        return self

    def categories(self) -> list[str]:
        """Textos de categoría en orden [target, target2]."""
        return [t.category for t in (self.target, self.target2) if t is not None]


class QueryResult(BaseModel):
    """Salida del ejecutor. `evidence` son ids de objetos del mismo SceneJSON."""

    model_config = ConfigDict(extra="forbid")

    answer: str = Field(min_length=1)
    normalized_answer: int | str
    evidence: list[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    status: Status = "ok"
    detail: str | None = None  # motivo de un estado distinto de `ok`

    # `before`: si no, pydantic convierte True → 1 antes de llegar acá.
    @field_validator("normalized_answer", mode="before")
    @classmethod
    def _check_normalized(cls, v: int | str) -> int | str:
        if isinstance(v, bool) or (isinstance(v, int) and v < 0):
            raise ValueError("normalized_answer entero tiene que ser ≥ 0")
        if isinstance(v, str) and not v:
            raise ValueError("normalized_answer string no puede ser vacío")
        return v

    @field_validator("evidence")
    @classmethod
    def _check_evidence(cls, v: list[str]) -> list[str]:
        if len(v) != len(set(v)):
            raise ValueError("evidence tiene ids repetidos")
        return v

    def to_question_answer(
        self, question_id: str, qtype: str, question: str, query: Query | None = None
    ) -> dict:
        """Item de `questions` (o de `questions_llm` si se pasa `query`) según el schema."""
        item = {
            "question_id": question_id,
            "type": qtype,
            "question": question,
            "answer": self.answer,
            "normalized_answer": self.normalized_answer,
            "evidence": list(self.evidence),
            "confidence": self.confidence,
            "status": self.status,
        }
        if self.detail is not None:
            item["status_detail"] = self.detail
        if query is not None:
            item["interpreted_query"] = query.model_dump(mode="json")
        return item


In [ ]:
%%writefile src/scene/qa/executor.py
"""Ejecutor determinista del DSL sobre un SceneJSON (E1-22).

Lo usan los dos modos de QA. Todo se calcula con código a partir del JSON; nada se adivina:
- categoría fuera del vocabulario, filtro de color desconocido u op no soportada → `unsupported`;
- pregunta singular sin instancias (posición, tamaño, color, relación) → `insufficient_information`;
- pregunta singular con varias instancias → `ambiguous` (D-10): se responde con la instancia de
  mayor confianza, la evidencia lleva todas y la confianza se reparte entre ellas.

Confianza (PLAN §4.5): `exists` → máx. de las instancias, o 1 − máx. de las descartadas por
filtros (sin ninguna: 1 − umbral del detector); `count` → promedio de las contadas (si son 0,
igual que `exists`); relaciones → producto de las confianzas de los dos objetos.
"""

from __future__ import annotations

from collections import Counter
from collections.abc import Callable, Mapping
from dataclasses import dataclass, field
from typing import Any

from scene.qa.dsl import Query, QueryResult, Target
from scene.qa.verbalize import POSITION_LABELS, SIZE_LABELS, verbalize

# Texto → category_id o None (p. ej. `CategoryMatcher.match_id`).
CategoryResolver = Callable[[str], int | None]
EXTRA_PREDICATES = ("near", "on_top_of")  # D-04: van en `extra_relations`


class _NotOk(Exception):
    """Corta la ejecución con un estado distinto de `ok`."""

    def __init__(self, status: str, detail: str, evidence: list[str] | None = None) -> None:
        super().__init__(detail)
        self.status, self.detail, self.evidence = status, detail, evidence or []


@dataclass
class Executor:
    resolve_category: CategoryResolver
    category_names: Mapping[int, str]
    palette: tuple[str, ...]  # nombres de color válidos (scene.color.palette)
    absent_confidence: float  # confianza de "no hay" cuando no queda ninguna instancia
    color_aliases: Mapping[str, str] = field(default_factory=dict)  # "rojo" → "red"
    display_names: Mapping[int, str] = field(default_factory=dict)  # nombre para describir
    describe_max_categories: int = 5

    def execute(self, query: Query, scene: Mapping[str, Any]) -> QueryResult:
        """Resultado validado; la evidencia siempre son ids de `scene['objects']`."""
        handler = getattr(self, f"_op_{query.op}", None)
        try:
            if handler is None or query.op == "unsupported":
                raise _NotOk("unsupported", query.reason or f"op {query.op!r} no soportada")
            normalized, evidence, confidence, status, detail = handler(query, scene)
        except _NotOk as stop:
            normalized, evidence, confidence = stop.status, stop.evidence, 0.0
            status, detail = stop.status, stop.detail
        result = QueryResult(
            answer=status,  # se reemplaza abajo por el texto en lenguaje natural
            normalized_answer=normalized,
            evidence=evidence,
            confidence=round(min(max(confidence, 0.0), 1.0), 4),
            status=status,
            detail=detail,
        )
        return result.model_copy(update={"answer": self._answer(result, query)})

    # --- selección de instancias ---------------------------------------------------------

    def _instances(self, target: Target, scene: Mapping[str, Any]) -> tuple[list[dict], list[dict]]:
        """(las que cumplen los filtros, las de la categoría que no), por confianza."""
        cid = self.resolve_category(target.category)
        if cid is None:
            raise _NotOk("unsupported", f"categoría fuera del vocabulario: {target.category!r}")
        color = target.filters.color
        if color is not None:
            color = self.color_aliases.get(color.lower(), color.lower())
            if color not in self.palette:
                raise _NotOk("unsupported", f"color no reconocido: {target.filters.color!r}")
        same = [o for o in scene.get("objects", []) if o.get("category_id") == cid]
        same.sort(key=lambda o: -o["confidence"])
        f = target.filters
        keep = [
            o
            for o in same
            if (f.position is None or o.get("position") == f.position)
            and (f.size is None or o.get("size") == f.size)
            and (color is None or (o.get("attributes") or {}).get("color") == color)
        ]
        kept = {o["id"] for o in keep}
        return keep, [o for o in same if o["id"] not in kept]

    def _single(self, target: Target, scene: Mapping[str, Any], exclude: str | None = None):
        """La instancia a la que se refiere una pregunta singular, y el estado/detalle."""
        found, _ = self._instances(target, scene)
        found = [o for o in found if o["id"] != exclude]
        if not found:
            raise _NotOk("insufficient_information", f"no se detectó {target.category!r}")
        top = found[0]
        if len(found) == 1:
            return top, [top["id"]], top["confidence"], "ok", None
        # D-10: la de mayor confianza, con su parte de la confianza total.
        share = top["confidence"] / sum(o["confidence"] for o in found)
        detail = f"{len(found)} instancias de {target.category!r}; se usó la de mayor confianza"
        return top, [o["id"] for o in found], top["confidence"] * share, "ambiguous", detail

    def _absent(self, rejected: list[dict]) -> float:
        return 1.0 - rejected[0]["confidence"] if rejected else self.absent_confidence

    # --- ops ------------------------------------------------------------------------------

    def _op_exists(self, query: Query, scene: Mapping[str, Any]):
        found, rejected = self._instances(query.target, scene)
        if found:
            return "yes", [o["id"] for o in found], found[0]["confidence"], "ok", None
        return "no", [], self._absent(rejected), "ok", None

    def _op_count(self, query: Query, scene: Mapping[str, Any]):
        found, rejected = self._instances(query.target, scene)
        if not found:
            return 0, [], self._absent(rejected), "ok", None
        mean = sum(o["confidence"] for o in found) / len(found)
        return len(found), [o["id"] for o in found], mean, "ok", None

    def _op_position(self, query: Query, scene: Mapping[str, Any]):
        top, evidence, conf, status, detail = self._single(query.target, scene)
        return top["position"], evidence, conf, status, detail

    def _op_size(self, query: Query, scene: Mapping[str, Any]):
        top, evidence, conf, status, detail = self._single(query.target, scene)
        return top["size"], evidence, conf, status, detail

    def _op_attribute(self, query: Query, scene: Mapping[str, Any]):
        top, evidence, conf, status, detail = self._single(query.target, scene)
        attrs = top.get("attributes") or {}
        if attrs.get("color") is None:
            raise _NotOk("insufficient_information", "color desconocido (máscara muy chica)")
        return (
            attrs["color"],
            evidence,
            conf * (attrs.get("color_confidence") or 1.0),
            status,
            detail,
        )

    def _op_relation(self, query: Query, scene: Mapping[str, Any]):
        subj, ev1, c1, st1, d1 = self._single(query.target, scene)
        obj, ev2, c2, st2, d2 = self._single(query.target2, scene, exclude=subj["id"])
        field_name = "extra_relations" if query.relation in EXTRA_PREDICATES else "relations"
        holds = any(
            r.get("subject") == subj["id"]
            and r.get("object") == obj["id"]
            and r.get("predicate") == query.relation
            for r in scene.get(field_name, [])
        )
        evidence = [subj["id"], obj["id"]] + [
            i for i in ev1 + ev2 if i not in (subj["id"], obj["id"])
        ]
        ambiguous = "ambiguous" in (st1, st2)
        detail = "; ".join(d for d in (d1, d2) if d) or None
        return (
            "yes" if holds else "no",
            evidence,
            c1 * c2,
            "ambiguous" if ambiguous else "ok",
            detail,
        )

    def _op_describe(self, query: Query, scene: Mapping[str, Any]):
        objects = sorted(scene.get("objects", []), key=lambda o: -o["confidence"])
        if not objects:
            return "No se detectaron objetos en la imagen.", [], self.absent_confidence, "ok", None
        counts = Counter(o["category_id"] for o in objects)
        top_cats = [cid for cid, _ in counts.most_common(self.describe_max_categories)]
        parts = [f"{self._display(cid)} ({counts[cid]})" for cid in top_cats]
        main = objects[0]
        n_obj = f"{len(objects)} objeto" + ("s" if len(objects) > 1 else "")
        n_cat = f"{len(counts)} categoría" + ("s" if len(counts) > 1 else "")
        text = (
            f"Se {'detectaron' if len(objects) > 1 else 'detectó'} {n_obj} de {n_cat}. "
            f"Las más frecuentes: {', '.join(parts)}. "
            f"El objeto más confiable es {self._display(main['category_id'])}, "
            f"{POSITION_LABELS[main['position']]} y de tamaño {SIZE_LABELS[main['size']]}."
        )
        used = [o for o in objects if o["category_id"] in top_cats]
        mean = sum(o["confidence"] for o in used) / len(used)
        return text, [o["id"] for o in used], mean, "ok", None

    # --- texto ----------------------------------------------------------------------------

    def _display(self, cid: int) -> str:
        return self.display_names.get(cid) or self.category_names.get(cid, str(cid))

    def _answer(self, result: QueryResult, query: Query) -> str:
        if query.op == "describe" and result.status == "ok":
            return str(result.normalized_answer)
        # Las plantillas nombran la categoría: se usa el nombre para mostrar (en español).
        cid = self.resolve_category(query.target.category) if query.target else None
        if cid is not None:
            target = query.target.model_copy(update={"category": self._display(cid)})
            query = query.model_copy(update={"target": target})
        if result.status == "ambiguous":
            text = verbalize(result.model_copy(update={"status": "ok"}), query)
            return f"{text} Hay varias instancias; se usó la de mayor confianza."
        return verbalize(result, query)


def executor_from_config(
    cfg: dict,
    resolve_category: CategoryResolver,
    category_names: Mapping[int, str],
    display_names: Mapping[int, str] | None = None,
) -> Executor:
    """Ejecutor según `cfg['qa']['executor']`, la paleta de color y el umbral del detector."""
    ecfg = cfg["qa"]["executor"]
    return Executor(
        resolve_category=resolve_category,
        category_names=category_names,
        palette=tuple(cfg["scene"]["color"]["palette"]),
        absent_confidence=1.0 - cfg["detection"]["score_threshold"],
        color_aliases=ecfg.get("color_aliases") or {},
        display_names=display_names or {},
        describe_max_categories=ecfg["describe_max_categories"],
    )


In [ ]:
%%writefile src/scene/qa/llm_mode.py
"""Modo con LLM (E1-28): texto de la pregunta → LLM → DSL validado → mismo ejecutor determinista.

El LLM recibe **solo** el texto de la pregunta (nunca `type`, `class_en` ni la imagen) y su única
salida es el JSON intermedio. Los conteos, filtros y comparaciones los resuelve el ejecutor.
Las consignas de descripción (`op="describe"`) las redacta el descriptor a partir de un resumen
del SceneJSON y se verifican; si la descripción no es fiel, se usa la plantilla determinista
(una respuesta incompleta pero sustentada es preferible a una fluida inventada).
"""

from __future__ import annotations

from collections.abc import Mapping
from typing import Any

from scene.llm.describer import SceneDescriber
from scene.llm.interpreter import InterpretationResult, Interpreter
from scene.qa.dsl import Query, QueryResult
from scene.qa.executor import Executor


def interpretation_to_query(interp: InterpretationResult) -> Query:
    """DSL que se ejecuta: el del LLM, o `op="unsupported"` con el motivo si no lo produjo."""
    if interp.status == "ok" and interp.query is not None:
        return interp.query
    reason = "; ".join(interp.errors) or "el intérprete no produjo una consulta válida"
    return Query(op="unsupported", reason=f"intérprete LLM: {reason}"[:500])


def _description_evidence(
    scene: Mapping[str, Any], mentioned: tuple[str, ...], names: Mapping[int, str]
) -> list[str]:
    """Ids de los objetos de las categorías que menciona la descripción (de mayor confianza)."""
    wanted = {m.lower() for m in mentioned}
    objs = sorted(scene.get("objects", []), key=lambda o: -o["confidence"])
    return [
        o["id"]
        for o in objs
        if (o.get("category_name") or names.get(o["category_id"], "")).lower() in wanted
    ]


def describe_result(
    scene: Mapping[str, Any],
    describer: SceneDescriber,
    executor: Executor,
    query: Query,
    names: Mapping[int, str],
) -> tuple[QueryResult, dict[str, Any]]:
    """Descripción con LLM verificada; si no es fiel, la plantilla determinista del ejecutor."""
    desc = describer.describe(scene)
    trace = {
        "describer_status": desc.status,
        "describer_attempts": desc.attempts,
        "fidelity_reasons": list(desc.fidelity.reasons),
    }
    if desc.status != "ok" or not desc.text:
        fallback = executor.execute(query, scene)
        detail = "descripción LLM no fiel; se usó la plantilla determinista"
        return fallback.model_copy(update={"detail": detail}), trace
    evidence = _description_evidence(scene, desc.fidelity.mentioned_categories, names)
    objs = {o["id"]: o for o in scene.get("objects", [])}
    conf = sum(objs[i]["confidence"] for i in evidence) / len(evidence) if evidence else 0.0
    result = QueryResult(
        answer=desc.text,
        normalized_answer=desc.text,
        evidence=evidence,
        confidence=round(conf, 4),
        status="ok",
    )
    return result, trace


def answer_questions_llm(
    questions: list[Mapping[str, Any]],
    scene: Mapping[str, Any],
    interpreter: Interpreter,
    executor: Executor,
    describer: SceneDescriber,
    category_names: Mapping[int, str],
) -> list[dict[str, Any]]:
    """Items de `questions_llm`, con `interpreted_query` y la traza del intérprete."""
    items = []
    for q in questions:
        interp = interpreter.interpret(q["question"])  # solo el texto
        query = interpretation_to_query(interp)
        extra: dict[str, Any] = {}
        if query.op == "describe":
            result, extra = describe_result(scene, describer, executor, query, category_names)
        else:
            result = executor.execute(query, scene)
        item = result.to_question_answer(q["question_id"], q["type"], q["question"], query)
        item["interpreter"] = {
            "status": interp.status,
            "attempts": interp.attempts,
            "errors": list(interp.errors),
            "raw_output": interp.raw_output,
            **extra,
        }
        items.append(item)
    return items


In [ ]:
%%writefile src/scene/qa/structured_adapter.py
"""Modo determinista (E1-23): campos estructurados del benchmark → DSL, sin LLM.

Usa `type` y `class_en` de cada consigna; el predicado de las relaciones no viene estructurado,
así que se busca en el texto con las frases de `qa.adapter.relation_phrases`. Las respuestas
van a `SceneJSON.questions` conservando `question_id`, `type` y `question`.
"""

from __future__ import annotations

from collections.abc import Mapping
from pathlib import Path
from typing import Any

from scene.qa.category_matcher import normalize_text
from scene.qa.dsl import QUESTION_TYPE_TO_OP, Query
from scene.qa.executor import Executor


def relation_from_text(question: str, phrases: Mapping[str, str]) -> str | None:
    """Predicado de la frase más larga que aparece en la pregunta (sin acentos ni mayúsculas)."""
    text = f" {normalize_text(question)} "
    for phrase in sorted(phrases, key=len, reverse=True):
        if f" {normalize_text(phrase)} " in text:
            return phrases[phrase]
    return None


def query_from_benchmark(question: Mapping[str, Any], phrases: Mapping[str, str]) -> Query:
    """DSL de una consigna; si no se puede traducir, `op="unsupported"` con el motivo."""
    op = QUESTION_TYPE_TO_OP.get(question["type"])
    classes = list(question.get("class_en") or [])
    if op is None:
        return Query(op="unsupported", reason=f"tipo de pregunta desconocido {question['type']!r}")
    if op == "describe":
        return Query(op="describe")
    if op == "relation":
        relation = relation_from_text(question["question"], phrases)
        if relation is None or len(classes) != 2:
            return Query(op="unsupported", reason="no se reconoce la relación o faltan clases")
        return Query(
            op="relation",
            target={"category": classes[0]},
            target2={"category": classes[1]},
            relation=relation,
        )
    if len(classes) != 1:
        return Query(op="unsupported", reason=f"se esperaba 1 clase y hay {len(classes)}")
    return Query(op=op, target={"category": classes[0]})


def benchmark_image_id(question: Mapping[str, Any]) -> str:
    """`tpi_0001.jpg` → `tpi_0001` (el `image_id` de la fila del submission)."""
    return Path(question["image_file"]).stem


def answer_questions(
    questions: list[Mapping[str, Any]],
    scene: Mapping[str, Any],
    executor: Executor,
    phrases: Mapping[str, str],
) -> list[dict[str, Any]]:
    """Items de `questions` (con `interpreted_query` propio para trazabilidad)."""
    items = []
    for q in questions:
        query = query_from_benchmark(q, phrases)
        result = executor.execute(query, scene)
        item = result.to_question_answer(q["question_id"], q["type"], q["question"])
        item["interpreted_query"] = query.model_dump(mode="json")
        items.append(item)
    return items


def group_by_image(questions: list[Mapping[str, Any]]) -> dict[str, list[Mapping[str, Any]]]:
    """Consignas por `image_id`, en el orden del benchmark."""
    out: dict[str, list[Mapping[str, Any]]] = {}
    for q in questions:
        out.setdefault(benchmark_image_id(q), []).append(q)
    return out


In [ ]:
%%writefile src/scene/qa/verbalize.py
"""Plantillas deterministas para verbalizar resultados del ejecutor."""

from __future__ import annotations

from collections.abc import Mapping
from typing import Any

POSITION_LABELS = {
    "top_left": "arriba a la izquierda",
    "top_center": "arriba en el centro",
    "top_right": "arriba a la derecha",
    "middle_left": "en el medio a la izquierda",
    "center": "en el centro",
    "middle_right": "en el medio a la derecha",
    "bottom_left": "abajo a la izquierda",
    "bottom_center": "abajo en el centro",
    "bottom_right": "abajo a la derecha",
}
SIZE_LABELS = {"small": "pequeño", "medium": "mediano", "large": "grande"}


def _value(source: Any, key: str, default: Any = None) -> Any:
    if isinstance(source, Mapping):
        return source.get(key, default)
    return getattr(source, key, default)


def _target_name(query: Any | None) -> str:
    target = _value(query, "target") if query is not None else None
    category = _value(target, "category") if target is not None else None
    return str(category) if category else "esa categoría"


def verbalize(result: Any, query: Any | None = None) -> str:
    """Convierte un resultado estructurado en una respuesta breve en español."""
    status = _value(result, "status", "ok")
    if status == "unsupported":
        return "La pregunta no está soportada."
    if status == "insufficient_information":
        return "No hay información suficiente para responder."
    if status == "ambiguous":
        return "La referencia es ambigua porque hay varias instancias posibles."

    normalized = _value(result, "normalized_answer", _value(result, "answer"))
    operation = _value(query, "op") if query is not None else None
    category = _target_name(query)
    if operation == "count" or isinstance(normalized, int):
        noun = "instancia" if normalized == 1 else "instancias"
        return f"Hay {normalized} {noun} de {category}."
    if operation in {"exists", "relation"} or normalized in {"yes", "no"}:
        return "Sí." if normalized == "yes" else "No."
    if operation == "position" or normalized in POSITION_LABELS:
        label = POSITION_LABELS.get(normalized, str(normalized))
        return f"La instancia está {label}."
    if operation == "size" or normalized in SIZE_LABELS:
        label = SIZE_LABELS.get(normalized, str(normalized))
        return f"La instancia tiene tamaño {label}."
    answer = _value(result, "answer")
    if isinstance(answer, str) and answer.strip():
        return answer.strip()
    return str(normalized)


In [ ]:
from pathlib import Path

_ = Path('src/scene/scene/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/scene/builder.py
"""Builder del SceneJSON: une detecciones, máscaras, propiedades, color y relaciones, y valida.

Cada objeto lleva `provenance` con el origen de sus campos:
- `observed`: salida directa de un modelo (caja y confianza del detector, máscara de SAM2);
- `derived`: calculado con código a partir de lo observado (centroide, posición, color, ...);
- `unknown`: no se pudo determinar (el valor queda en `null`).
"""

from __future__ import annotations

from typing import Any

import numpy as np

from scene.config import data_path
from scene.masks import rle
from scene.scene.color import dominant_color_from_config
from scene.scene.properties import ensure_nonempty, properties_from_config
from scene.scene.relations import RelObject, compute_relations
from scene.scene.schema import assert_valid_scene
from scene.segmentation.sam2 import SegmentedObject

DECIMALS = 2  # redondeo de coordenadas en píxeles (cajas y centroides)


def _round(values, ndigits: int = DECIMALS) -> list[float]:
    return [round(float(v), ndigits) for v in values]


def build_object(
    oid: str, seg: SegmentedObject, image: np.ndarray, cfg: dict
) -> tuple[dict[str, Any], RelObject]:
    """Un objeto del SceneJSON y su vista para el cálculo de relaciones."""
    det = seg.detection
    mask, mask_source = ensure_nonempty(seg.mask, det.box_xyxy)
    props = properties_from_config(mask, cfg, mask_source)
    color = dominant_color_from_config(image, mask, cfg)
    obj = {
        "id": oid,
        "category_id": det.category_id,
        "category_name": det.category_name,
        "confidence": round(det.score, 4),
        "bbox": _round(det.box_xyxy),
        "mask": rle.encode(mask),
        "centroid": _round(props.centroid),
        "area_ratio": props.area_ratio,
        "position": props.position,
        "size": props.size,
        "area_px": props.area,
        "mask_bbox": list(props.mask_bbox),
        "mask_score": round(seg.mask_score, 4),
        "attributes": {
            "color": color.name if color else None,
            "color_confidence": round(color.confidence, 3) if color else None,
            "color_rgb": list(color.rgb) if color else None,
        },
        "provenance": {
            "category_id": "observed",
            "confidence": "observed",
            "bbox": "observed",
            "mask": "observed" if mask_source == "mask" else "derived",
            "centroid": "derived",
            "area_ratio": "derived",
            "position": "derived",
            "size": "derived",
            "color": "derived" if color else "unknown",
        },
        "source": {"detector": det.source, "raw_query": det.raw_query, "mask": mask_source},
    }
    view = RelObject(oid, props.centroid, props.mask_bbox, mask, det.score)
    return obj, view


def build_scene(
    image_id: str,
    image: np.ndarray,
    segmented: list[SegmentedObject],
    cfg: dict,
    validate: bool = True,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """SceneJSON de una imagen y estadísticas para la traza. Ids `obj_<n>` por confianza."""
    height, width = image.shape[:2]
    ordered = sorted(segmented, key=lambda s: -s.detection.score)
    objects, views = [], []
    for n, seg in enumerate(ordered):
        obj, view = build_object(f"obj_{n}", seg, image, cfg)
        objects.append(obj)
        views.append(view)
    rel = compute_relations(views, width, height, cfg["scene"]["relations"])
    scene = {
        "image": {"id": image_id, "width": int(width), "height": int(height)},
        "objects": objects,
        "relations": [r.to_dict() for r in rel.relations],
        "extra_relations": [r.to_dict() for r in rel.extra_relations],
    }
    if validate:
        assert_valid_scene(scene, data_path(cfg, "scene_schema"))
    stats = {
        **rel.stats,
        "box_fallback_masks": sum(o["source"]["mask"] == "box_fallback" for o in objects),
        "unknown_color": sum(o["attributes"]["color"] is None for o in objects),
    }
    return scene, stats


In [ ]:
%%writefile src/scene/scene/color.py
"""Color dominante dentro de la máscara: k-means en Lab y mapeo a una paleta con nombre.

La confianza es la proporción de píxeles del cluster dominante. Si la máscara tiene pocos píxeles
el color queda desconocido (`None`) en lugar de inventarlo.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np
from sklearn.cluster import KMeans


@dataclass(frozen=True)
class DominantColor:
    name: str  # nombre de la paleta más cercano al centro del cluster dominante
    rgb: tuple[int, int, int]  # centro del cluster dominante
    confidence: float  # proporción de píxeles del cluster dominante
    lab_distance: float  # distancia (CIE76) entre el cluster y el color de la paleta

    def to_dict(self) -> dict:
        d = asdict(self)
        d["rgb"] = list(self.rgb)
        return d


def rgb_to_lab(rgb: np.ndarray) -> np.ndarray:
    """RGB uint8 (N×3) → Lab float (L en [0, 100])."""
    arr = (rgb.reshape(-1, 1, 3).astype(np.float32)) / 255.0
    return cv2.cvtColor(arr, cv2.COLOR_RGB2LAB).reshape(-1, 3)


def nearest_palette_color(lab: np.ndarray, palette: dict[str, list[int]]) -> tuple[str, float]:
    names = list(palette)
    ref = rgb_to_lab(np.array([palette[n] for n in names], dtype=np.uint8))
    dist = np.linalg.norm(ref - lab[None, :], axis=1)
    i = int(dist.argmin())
    return names[i], float(dist[i])


def dominant_color(
    image: np.ndarray,
    mask: np.ndarray,
    palette: dict[str, list[int]],
    k: int = 3,
    max_pixels: int = 5000,
    min_pixels: int = 20,
    seed: int = 0,
) -> DominantColor | None:
    """Color dominante de los píxeles de `mask` en `image` (RGB uint8 HxWx3)."""
    pixels = image[mask]
    if len(pixels) < min_pixels:
        return None
    rng = np.random.default_rng(seed)
    if len(pixels) > max_pixels:
        pixels = pixels[rng.choice(len(pixels), max_pixels, replace=False)]
    lab = rgb_to_lab(pixels)
    n_clusters = min(k, len(np.unique(pixels, axis=0)))
    km = KMeans(n_clusters=n_clusters, n_init=3, random_state=seed).fit(lab)
    counts = np.bincount(km.labels_, minlength=n_clusters)
    top = int(counts.argmax())
    members = pixels[km.labels_ == top]
    name, dist = nearest_palette_color(km.cluster_centers_[top], palette)
    return DominantColor(
        name=name,
        rgb=tuple(int(v) for v in members.mean(axis=0).round()),
        confidence=float(counts[top] / counts.sum()),
        lab_distance=dist,
    )


def dominant_color_from_config(
    image: np.ndarray, mask: np.ndarray, cfg: dict
) -> DominantColor | None:
    c = cfg["scene"]["color"]
    return dominant_color(
        image,
        mask,
        c["palette"],
        k=c["k"],
        max_pixels=c["max_pixels"],
        min_pixels=c["min_pixels"],
        seed=cfg["seed"],
    )


In [ ]:
%%writefile src/scene/scene/properties.py
"""Propiedades geométricas derivadas de la máscara: área, área relativa, centroide, posición 3×3,
tamaño y caja de la máscara. Convenciones en `docs/schema_notes.md` §6.

Se calculan siempre sobre la máscara que va al SceneJSON (la decodificada del RLE), así lo que se
entrega es consistente por construcción. Si la máscara está vacía se usa la caja rasterizada
como máscara y se marca `source="box_fallback"`.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np

from scene.masks import rle
from scene.scene.schema import POSITIONS


@dataclass(frozen=True)
class GeometricProperties:
    area: int  # píxeles de la máscara
    area_ratio: float  # área / (H·W), en (0, 1]
    centroid: tuple[float, float]  # [x, y] en píxeles: media de los índices de la máscara
    position: str  # celda de la grilla 3×3 que contiene el centroide
    size: str  # small | medium | large según area_ratio
    mask_bbox: tuple[int, int, int, int]  # xyxy con borde exclusivo
    source: str  # mask | box_fallback

    def to_dict(self) -> dict:
        d = asdict(self)
        d["centroid"] = list(self.centroid)
        d["mask_bbox"] = list(self.mask_bbox)
        return d


def position_3x3(cx: float, cy: float, width: int, height: int) -> str:
    """Celda por tercios iguales; un punto sobre un borde va a la celda de la derecha/abajo."""
    col = min(max(int(cx * 3 // width), 0), 2)
    row = min(max(int(cy * 3 // height), 0), 2)
    return POSITIONS[row * 3 + col]


def size_label(area_ratio: float, small_max: float, medium_max: float) -> str:
    if area_ratio < small_max:
        return "small"
    if area_ratio < medium_max:
        return "medium"
    return "large"


def box_to_mask(box_xyxy: tuple[float, float, float, float], height: int, width: int) -> np.ndarray:
    """Rasteriza una caja (al menos 1 píxel, dentro de la imagen)."""
    x1, y1, x2, y2 = box_xyxy
    c1 = min(max(int(np.floor(x1)), 0), width - 1)
    r1 = min(max(int(np.floor(y1)), 0), height - 1)
    c2 = min(max(int(np.ceil(x2)), c1 + 1), width)
    r2 = min(max(int(np.ceil(y2)), r1 + 1), height)
    mask = np.zeros((height, width), dtype=bool)
    mask[r1:r2, c1:c2] = True
    return mask


def ensure_nonempty(
    mask: np.ndarray, box_xyxy: tuple[float, float, float, float]
) -> tuple[np.ndarray, str]:
    """Devuelve la máscara y su origen; si está vacía, la reemplaza por la caja rasterizada."""
    if mask.any():
        return mask, "mask"
    return box_to_mask(box_xyxy, *mask.shape), "box_fallback"


def properties_from_mask(
    mask: np.ndarray, small_max: float, medium_max: float, source: str = "mask"
) -> GeometricProperties:
    """Propiedades de una máscara no vacía (bool HxW)."""
    height, width = mask.shape
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        raise ValueError("Máscara vacía: usar ensure_nonempty antes")
    area = int(len(xs))
    area_ratio = area / (height * width)
    cx, cy = float(xs.mean()), float(ys.mean())
    return GeometricProperties(
        area=area,
        area_ratio=area_ratio,
        centroid=(cx, cy),
        position=position_3x3(cx, cy, width, height),
        size=size_label(area_ratio, small_max, medium_max),
        mask_bbox=(int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1),
        source=source,
    )


def properties_from_rle(
    mask_rle: dict, small_max: float, medium_max: float, source: str = "mask"
) -> GeometricProperties:
    """Igual que `properties_from_mask`, decodificando antes el RLE del SceneJSON."""
    return properties_from_mask(rle.decode(mask_rle), small_max, medium_max, source)


def properties_from_config(
    mask: np.ndarray, cfg: dict, source: str = "mask"
) -> GeometricProperties:
    t = cfg["scene"]["size_thresholds"]
    return properties_from_mask(mask, t["small_max"], t["medium_max"], source)


In [ ]:
%%writefile src/scene/scene/relations.py
"""Relaciones espaciales entre objetos, derivadas de centroides, cajas y máscaras.

- `relations` (enum del schema): `left_of`, `right_of`, `above`, `below`, `overlaps`.
- `extra_relations` (D-04, fuera del enum): `near` y `on_top_of`.

Las relaciones direccionales se emiten en los dos sentidos (A left_of B ⇔ B right_of A) y
`overlaps`/`near` son simétricas. Las operaciones con máscaras solo se hacen sobre pares cuyas
cajas se tocan, y se recortan a la unión de las cajas.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field
from itertools import combinations
from typing import Any

import cv2
import numpy as np


@dataclass(frozen=True)
class RelObject:
    id: str
    centroid: tuple[float, float]  # [x, y]
    box: tuple[int, int, int, int]  # caja de la máscara, xyxy exclusivo
    mask: np.ndarray = field(repr=False, compare=False)
    confidence: float = 1.0


@dataclass(frozen=True)
class Relation:
    subject: str
    predicate: str
    object: str

    def to_dict(self) -> dict[str, str]:
        return asdict(self)


@dataclass
class RelationsResult:
    relations: list[Relation]
    extra_relations: list[Relation]
    stats: dict[str, Any]


def directional(
    a: RelObject, b: RelObject, width: int, height: int, h_margin: float, v_margin: float
) -> list[str]:
    """Predicados direccionales de `a` respecto de `b` por centroides (con margen mínimo)."""
    preds = []
    dx = b.centroid[0] - a.centroid[0]
    dy = b.centroid[1] - a.centroid[1]
    if dx > h_margin * width:
        preds.append("left_of")
    elif dx < -h_margin * width:
        preds.append("right_of")
    if dy > v_margin * height:
        preds.append("above")
    elif dy < -v_margin * height:
        preds.append("below")
    return preds


def _inverse(pred: str) -> str:
    return {"left_of": "right_of", "right_of": "left_of", "above": "below", "below": "above"}[pred]


def box_gap(a: tuple[int, ...], b: tuple[int, ...]) -> float:
    """Distancia euclídea entre dos cajas (0 si se tocan o se solapan)."""
    gx = max(0, max(a[0], b[0]) - min(a[2], b[2]))
    gy = max(0, max(a[1], b[1]) - min(a[3], b[3]))
    return float(np.hypot(gx, gy))


def _union_crop(a: RelObject, b: RelObject, pad: int) -> tuple[slice, slice]:
    h, w = a.mask.shape
    x1, y1 = max(0, min(a.box[0], b.box[0]) - pad), max(0, min(a.box[1], b.box[1]) - pad)
    x2, y2 = min(w, max(a.box[2], b.box[2]) + pad), min(h, max(a.box[3], b.box[3]) + pad)
    return slice(y1, y2), slice(x1, x2)


def mask_iou(a: RelObject, b: RelObject) -> float:
    rows, cols = _union_crop(a, b, 0)
    ma, mb = a.mask[rows, cols], b.mask[rows, cols]
    union = np.logical_or(ma, mb).sum()
    return float(np.logical_and(ma, mb).sum() / union) if union else 0.0


def masks_touch(a: RelObject, b: RelObject, contact_px: int) -> bool:
    """True si la máscara de `a` dilatada `contact_px` toca la de `b`."""
    rows, cols = _union_crop(a, b, contact_px)
    ma = a.mask[rows, cols].astype(np.uint8)
    kernel = np.ones((2 * contact_px + 1, 2 * contact_px + 1), np.uint8)
    return bool(np.logical_and(cv2.dilate(ma, kernel) > 0, b.mask[rows, cols]).any())


def on_top_of(a: RelObject, b: RelObject, min_x_overlap: float, contact_px: int) -> bool:
    """`a` apoyado sobre `b`: más arriba, con solape horizontal y contacto entre máscaras."""
    if not (a.centroid[1] < b.centroid[1] and a.box[3] <= b.box[3]):
        return False
    x_overlap = min(a.box[2], b.box[2]) - max(a.box[0], b.box[0])
    if x_overlap < min_x_overlap * max(a.box[2] - a.box[0], 1):
        return False
    return masks_touch(a, b, contact_px)


def _pair_relations(
    a: RelObject, b: RelObject, width: int, height: int, r: dict[str, Any], stats: dict
) -> tuple[list[Relation], list[Relation]]:
    """Todas las relaciones del par no ordenado {a, b}, en los dos sentidos."""
    rels = []
    for p in directional(a, b, width, height, r["h_margin"], r["v_margin"]):
        rels += [Relation(a.id, p, b.id), Relation(b.id, _inverse(p), a.id)]
    extra = []
    gap = box_gap(a.box, b.box)
    if gap <= r["contact_px"]:
        stats["n_mask_ops"] += 1
        if gap == 0 and mask_iou(a, b) >= r["overlap_min_iou"]:
            rels += [Relation(a.id, "overlaps", b.id), Relation(b.id, "overlaps", a.id)]
        for s, o in ((a, b), (b, a)):
            if on_top_of(s, o, r["on_top_min_x_overlap"], r["contact_px"]):
                extra.append(Relation(s.id, "on_top_of", o.id))
    if gap <= r["near_max_gap"] * np.hypot(width, height):
        extra += [Relation(a.id, "near", b.id), Relation(b.id, "near", a.id)]
    return rels, extra


def compute_relations(
    objs: list[RelObject], width: int, height: int, rel_cfg: dict[str, Any]
) -> RelationsResult:
    """Relaciones de todos los pares, priorizando los de mayor confianza hasta `max_relations`.

    Un par entra completo o no entra, así nunca queda una relación sin su inversa.
    """
    pairs = sorted(
        combinations(range(len(objs)), 2),
        key=lambda ij: (-min(objs[ij[0]].confidence, objs[ij[1]].confidence), ij),
    )
    stats: dict[str, Any] = {"n_objects": len(objs), "n_pairs": len(pairs), "n_mask_ops": 0}
    relations: list[Relation] = []
    extra: list[Relation] = []
    truncated = 0
    for i, j in pairs:
        rels, ext = _pair_relations(objs[i], objs[j], width, height, rel_cfg, stats)
        if len(relations) + len(rels) > rel_cfg["max_relations"]:
            truncated += 1
            continue
        relations += rels
        extra += ext
    stats["pairs_truncated"] = truncated
    stats["n_relations"] = len(relations)
    stats["n_extra_relations"] = len(extra)
    return RelationsResult(relations, extra, stats)


In [ ]:
%%writefile src/scene/scene/schema.py
"""Validación del SceneJSON contra `scene_schema.json` (normativo) y chequeos estructurales."""

from __future__ import annotations

import json
from functools import lru_cache
from pathlib import Path

from jsonschema import Draft202012Validator

# Enums del schema. Un test verifica que coincidan con `scene_schema.json`.
POSITIONS = (
    "top_left",
    "top_center",
    "top_right",
    "middle_left",
    "center",
    "middle_right",
    "bottom_left",
    "bottom_center",
    "bottom_right",
)
SIZES = ("small", "medium", "large")
RELATION_PREDICATES = ("left_of", "right_of", "above", "below", "overlaps")
QUESTION_TYPES = ("description", "count", "existence", "position", "size", "relation")


class SceneValidationError(ValueError):
    """El SceneJSON no cumple el schema o tiene inconsistencias estructurales."""

    def __init__(self, errors: list[str]) -> None:
        super().__init__("; ".join(errors[:5]) + (" ..." if len(errors) > 5 else ""))
        self.errors = errors


@lru_cache(maxsize=4)
def _validator(schema_path: str) -> Draft202012Validator:
    with open(schema_path, encoding="utf-8") as f:
        schema = json.load(f)
    Draft202012Validator.check_schema(schema)
    return Draft202012Validator(schema)


def schema_errors(scene: dict, schema_path: str | Path) -> list[str]:
    """Errores de validación jsonschema, con la ruta del campo."""
    validator = _validator(str(schema_path))
    errors = sorted(validator.iter_errors(scene), key=lambda e: list(e.absolute_path))
    return [f"{'/'.join(map(str, e.absolute_path)) or '<raíz>'}: {e.message}" for e in errors]


def structural_errors(scene: dict) -> list[str]:
    """Chequeos que el schema no expresa: ids únicos, tamaño de máscara, referencias válidas."""
    errors: list[str] = []
    image = scene.get("image", {})
    hw = [image.get("height"), image.get("width")]
    objects = scene.get("objects", [])
    ids = [o.get("id") for o in objects]
    if len(ids) != len(set(ids)):
        errors.append("objects: hay ids repetidos")
    valid_ids = set(ids)
    for i, obj in enumerate(objects):
        mask = obj.get("mask", {})
        if mask.get("size") != hw:
            errors.append(f"objects/{i}/mask/size: {mask.get('size')} != [alto, ancho] {hw}")
        elif sum(mask.get("counts", [])) != hw[0] * hw[1]:
            errors.append(f"objects/{i}/mask/counts: la suma no coincide con alto*ancho")
    for i, rel in enumerate(scene.get("relations", [])):
        for key in ("subject", "object"):
            if rel.get(key) not in valid_ids:
                errors.append(f"relations/{i}/{key}: id inexistente {rel.get(key)!r}")
        if rel.get("subject") == rel.get("object"):
            errors.append(f"relations/{i}: sujeto y objeto son el mismo id")
    for field in ("questions", "questions_llm"):
        for i, qa in enumerate(scene.get(field, [])):
            for ev in qa.get("evidence", []):
                if ev not in valid_ids:
                    errors.append(f"{field}/{i}/evidence: id inexistente {ev!r}")
    return errors


def validate_scene(scene: dict, schema_path: str | Path, structural: bool = True) -> list[str]:
    """Todos los errores (schema y, opcionalmente, estructurales). Lista vacía = válido."""
    errors = schema_errors(scene, schema_path)
    if structural:
        errors += structural_errors(scene)
    return errors


def assert_valid_scene(scene: dict, schema_path: str | Path, structural: bool = True) -> None:
    """Igual que `validate_scene` pero lanza `SceneValidationError` si hay errores."""
    errors = validate_scene(scene, schema_path, structural=structural)
    if errors:
        raise SceneValidationError(errors)


In [ ]:
from pathlib import Path

_ = Path('src/scene/segmentation/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/segmentation/debug.py
"""Imágenes de debug con las máscaras superpuestas."""

from __future__ import annotations

from pathlib import Path

import numpy as np

from scene.detection.debug import draw_detections
from scene.segmentation.sam2 import SegmentedObject


def draw_masks(
    image: np.ndarray, objs: list[SegmentedObject], path: str | Path, alpha: float = 0.45
) -> Path:
    """Pinta cada máscara con un color fijo por índice y dibuja su caja encima."""
    out = image.astype(np.float32).copy()
    rng = np.random.default_rng(0)
    colors = rng.integers(40, 255, size=(max(len(objs), 1), 3))
    for o, color in zip(objs, colors, strict=False):
        out[o.mask] = (1 - alpha) * out[o.mask] + alpha * color
    blended = out.clip(0, 255).astype(np.uint8)
    return draw_detections(blended, [o.detection for o in objs], Path(path))


In [ ]:
%%writefile src/scene/segmentation/postprocess.py
"""Post-proceso de máscaras: recorte a la caja con margen, componente principal y relleno de huecos.

Cada paso puede apagarse desde `cfg['segmentation']['postprocess']` para medir su aporte (E1-12).
"""

from __future__ import annotations

from typing import Any

import cv2
import numpy as np


def crop_to_box(
    mask: np.ndarray, box_xyxy: tuple[float, float, float, float], margin: float
) -> np.ndarray:
    """Anula lo que cae fuera de la caja agrandada `margin` (fracción del ancho y alto)."""
    h, w = mask.shape
    x1, y1, x2, y2 = box_xyxy
    mx, my = (x2 - x1) * margin, (y2 - y1) * margin
    c1, r1 = max(0, int(np.floor(x1 - mx))), max(0, int(np.floor(y1 - my)))
    c2, r2 = min(w, int(np.ceil(x2 + mx))), min(h, int(np.ceil(y2 + my)))
    out = np.zeros_like(mask)
    out[r1:r2, c1:c2] = mask[r1:r2, c1:c2]
    return out


def largest_component(mask: np.ndarray) -> np.ndarray:
    """Se queda con la componente conexa (8-vecinos) de mayor área."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if n <= 2:  # fondo + a lo sumo una componente
        return mask
    biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return labels == biggest


def fill_holes(mask: np.ndarray, max_hole_ratio: float) -> np.ndarray:
    """Rellena huecos (fondo que no toca el borde) de área ≤ `max_hole_ratio` × área de máscara."""
    area = int(mask.sum())
    if area == 0 or max_hole_ratio <= 0:
        return mask
    inv = (~mask).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(inv, connectivity=4)
    h, w = mask.shape
    out = mask.copy()
    for i in range(1, n):
        x, y, bw, bh, hole_area = stats[i]
        touches_border = x == 0 or y == 0 or x + bw == w or y + bh == h
        if not touches_border and hole_area <= max_hole_ratio * area:
            out[labels == i] = True
    return out


def postprocess_mask(
    mask: np.ndarray, box_xyxy: tuple[float, float, float, float], pp_cfg: dict[str, Any]
) -> np.ndarray:
    """Aplica los pasos activos en orden: recorte → componente principal → huecos.

    Con `apply_below_area_ratio` (E1-30) solo se post-procesan las máscaras crudas cuya área
    relativa es menor que ese valor; las demás se devuelven tal cual las da SAM2.
    """
    out = mask.astype(bool)
    limit = pp_cfg.get("apply_below_area_ratio")
    if limit is not None and out.mean() >= limit:
        return out
    if pp_cfg.get("crop_to_box", True):
        out = crop_to_box(out, box_xyxy, pp_cfg["box_margin"])
    if pp_cfg.get("largest_component", True):
        out = largest_component(out)
    if pp_cfg.get("fill_holes", True):
        out = fill_holes(out, pp_cfg["max_hole_ratio"])
    return out


In [ ]:
%%writefile src/scene/segmentation/sam2.py
"""Segmentación de instancias con SAM2 (vía `transformers`) usando las cajas detectadas como prompt.

El embedding de la imagen se calcula una sola vez y las cajas se procesan en lotes.
Con `multimask_output` SAM2 propone 3 máscaras por caja; queda la de mayor IoU predicho.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

from scene.detection.base import Detection
from scene.masks import rle
from scene.segmentation.postprocess import postprocess_mask


@dataclass
class SegmentedObject:
    detection: Detection
    mask: np.ndarray  # bool HxW, solo en memoria
    mask_score: float  # IoU predicho por SAM2
    raw_area: int = 0  # píxeles antes del post-proceso (para medir cuánto recorta)


def select_best_mask(masks: np.ndarray, iou_scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """masks (N, K, H, W) e iou_scores (N, K) → la máscara de mayor IoU predicho por caja."""
    best = iou_scores.argmax(axis=1)
    idx = np.arange(len(best))
    return masks[idx, best], iou_scores[idx, best]


class Sam2Segmenter:
    """SAM2 con cajas como prompt."""

    def __init__(
        self,
        model_id: str,
        device: str = "cpu",
        multimask_output: bool = True,
        box_batch_size: int = 32,
        mask_threshold: float = 0.0,
        postprocess: dict[str, Any] | None = None,
    ) -> None:
        import torch
        from transformers import Sam2Model, Sam2Processor

        self._torch = torch
        self.model_id = model_id
        self.device = device
        self.multimask_output = multimask_output
        self.box_batch_size = box_batch_size
        self.mask_threshold = mask_threshold
        self.pp_cfg = postprocess or {}
        self.processor = Sam2Processor.from_pretrained(model_id)
        self.model = Sam2Model.from_pretrained(model_id).to(device).eval()

    def params(self) -> dict[str, Any]:
        return {
            "model_id": self.model_id,
            "multimask_output": self.multimask_output,
            "box_batch_size": self.box_batch_size,
            "mask_threshold": self.mask_threshold,
            "postprocess": self.pp_cfg,
        }

    def segment(self, image: np.ndarray, dets: list[Detection]) -> list[SegmentedObject]:
        if not dets:
            return []
        torch = self._torch
        inputs = self.processor(images=image, return_tensors="pt")
        original_sizes = inputs["original_sizes"]
        with torch.inference_mode():
            embeddings = self.model.get_image_embeddings(inputs["pixel_values"].to(self.device))
        out: list[SegmentedObject] = []
        for start in range(0, len(dets), self.box_batch_size):
            chunk = dets[start : start + self.box_batch_size]
            boxes = self.processor(
                input_boxes=[[list(d.box_xyxy) for d in chunk]],
                original_sizes=original_sizes,
                return_tensors="pt",
            )["input_boxes"]
            with torch.inference_mode():
                pred = self.model(
                    image_embeddings=embeddings,
                    input_boxes=boxes.to(self.device),
                    multimask_output=self.multimask_output,
                )
            masks = self.processor.post_process_masks(
                pred.pred_masks.float().cpu(), original_sizes, mask_threshold=self.mask_threshold
            )[0].numpy()
            best, scores = select_best_mask(masks, pred.iou_scores[0].float().cpu().numpy())
            for d, m, s in zip(chunk, best, scores, strict=True):
                m = m.astype(bool)
                out.append(
                    SegmentedObject(
                        detection=d,
                        mask=postprocess_mask(m, d.box_xyxy, self.pp_cfg),
                        mask_score=float(s),
                        raw_area=int(m.sum()),
                    )
                )
        return out


def build_segmenter(cfg: dict, device: str) -> Sam2Segmenter:
    s = cfg["segmentation"]
    return Sam2Segmenter(
        model_id=s["model_id"],
        device=device,
        multimask_output=s["multimask_output"],
        box_batch_size=s["box_batch_size"],
        mask_threshold=s["mask_threshold"],
        postprocess=s["postprocess"],
    )


def save_segmented(objs: list[SegmentedObject], path: str | Path) -> None:
    """Caché por imagen: detección + RLE + score (no se guardan matrices completas)."""
    payload = [
        {
            "detection": o.detection.to_dict(),
            "mask": rle.encode(o.mask),
            "mask_score": o.mask_score,
            "raw_area": o.raw_area,
        }
        for o in objs
    ]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload), encoding="utf-8")


def load_segmented(path: str | Path) -> list[SegmentedObject]:
    return [
        SegmentedObject(
            detection=Detection.from_dict(o["detection"]),
            mask=rle.decode(o["mask"]),
            mask_score=o["mask_score"],
            raw_area=o["raw_area"],
        )
        for o in json.loads(Path(path).read_text(encoding="utf-8"))
    ]


In [ ]:
from pathlib import Path

_ = Path('src/scene/tracing/__init__.py').write_text('', encoding='utf-8')


In [ ]:
%%writefile src/scene/tracing/tracer.py
"""Trazas por (imagen, etapa) en `trace.jsonl`.

Uso:
    tracer = Tracer(ctx.trace_path, run_id=ctx.run_id, config_hash=ctx.config_hash)
    for img in images:
        with tracer.image_guard(img.image_id):          # un error no corta el run
            with tracer.stage(img.image_id, "detection", params=p, n_in=1) as s:
                dets = detector.detect(img)
                s.n_out = len(dets)
"""

from __future__ import annotations

import json
import sys
import time
import traceback
from collections.abc import Iterator
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any


@dataclass
class StageRecord:
    """Una línea de la traza. La etapa completa `n_out`, `artifact_path` y `extra`."""

    run_id: str
    config_hash: str
    image_id: str
    stage: str
    model_id: str | None = None
    params: dict[str, Any] = field(default_factory=dict)
    t_start: float = 0.0  # epoch en segundos
    t_end: float = 0.0
    duration_ms: float = 0.0
    status: str = "ok"  # ok | cached | error
    n_in: int | None = None
    n_out: int | None = None
    artifact_path: str | None = None
    error: str | None = None
    traceback: str | None = None
    gpu_mem_peak_mb: float | None = None
    # Datos propios de la etapa (por ejemplo, cuántas detecciones elimina cada filtro).
    extra: dict[str, Any] = field(default_factory=dict)


def _cuda_torch():
    """Devuelve torch solo si ya está importado y hay CUDA (el tracer no importa torch)."""
    torch = sys.modules.get("torch")
    if torch is not None and torch.cuda.is_available():
        return torch
    return None


class Tracer:
    """Escribe una línea JSONL por etapa ejecutada."""

    def __init__(self, path: str | Path, run_id: str, config_hash: str) -> None:
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id
        self.config_hash = config_hash
        self.failed_images: list[str] = []

    def _write(self, record: StageRecord) -> None:
        line = json.dumps(asdict(record), ensure_ascii=False, default=str)
        with self.path.open("a", encoding="utf-8") as f:
            f.write(line + "\n")

    @contextmanager
    def stage(
        self,
        image_id: str,
        stage: str,
        params: dict[str, Any] | None = None,
        model_id: str | None = None,
        n_in: int | None = None,
    ) -> Iterator[StageRecord]:
        """Mide y registra una etapa. Si falla, la registra con `status=error` y relanza."""
        rec = StageRecord(
            run_id=self.run_id,
            config_hash=self.config_hash,
            image_id=image_id,
            stage=stage,
            model_id=model_id,
            params=dict(params or {}),
            n_in=n_in,
        )
        torch = _cuda_torch()
        if torch is not None:
            torch.cuda.reset_peak_memory_stats()
        rec.t_start = time.time()
        t0 = time.perf_counter()
        try:
            yield rec
        except Exception as exc:
            rec.status = "error"
            rec.error = f"{type(exc).__name__}: {exc}"
            rec.traceback = traceback.format_exc()
            exc._traced = True  # para que image_guard no la registre dos veces
            raise
        finally:
            rec.duration_ms = (time.perf_counter() - t0) * 1000
            rec.t_end = time.time()
            if torch is not None:
                rec.gpu_mem_peak_mb = torch.cuda.max_memory_allocated() / 2**20
            self._write(rec)

    @contextmanager
    def image_guard(self, image_id: str) -> Iterator[None]:
        """Atrapa cualquier error de una imagen para que el run siga con la siguiente."""
        try:
            yield
        except Exception as exc:
            self.failed_images.append(image_id)
            if not getattr(exc, "_traced", False):
                # Error fuera de una etapa: se registra igual para no perderlo.
                self._write(
                    StageRecord(
                        run_id=self.run_id,
                        config_hash=self.config_hash,
                        image_id=image_id,
                        stage="image",
                        status="error",
                        t_start=time.time(),
                        t_end=time.time(),
                        error=f"{type(exc).__name__}: {exc}",
                        traceback=traceback.format_exc(),
                    )
                )


def read_trace(path: str | Path) -> list[dict[str, Any]]:
    """Lee `trace.jsonl` como lista de dicts."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


In [ ]:
%pip install -q -e ".[llm]"


In [ ]:

RUN_ORACLE_SEGMENTATION = True
RUN_THRESHOLD_TUNING = True
RUN_CONTROLLED_EXPERIMENT = True
RUN_LLM_BENCHMARK = True


In [ ]:

import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

BASE_CONFIGS = ["configs/base.yaml", "configs/kaggle.yaml"]
RUNS_DIR = Path("/kaggle/working/outputs/runs")


def run_script(script, *args, configs=BASE_CONFIGS):
    command = [sys.executable, script, "--config", *configs, *map(str, args)]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


def run_pipeline(*args, configs=BASE_CONFIGS):
    before = set(RUNS_DIR.glob("*")) if RUNS_DIR.exists() else set()
    run_script("scripts/run_pipeline.py", *args, configs=configs)
    candidates = set(RUNS_DIR.glob("*")) - before
    if not candidates:
        candidates = {path for path in RUNS_DIR.glob("*") if (path / "scenes").is_dir()}
    if not candidates:
        raise RuntimeError("La pipeline no creó un run")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


## 2. EDA — solo ground truth

Esta sección caracteriza las anotaciones de desarrollo. Sus resultados justifican los umbrales,
el tope de detecciones y el desglose por tamaño; no alimentan la inferencia por imagen.


In [ ]:

run_script("scripts/run_eda.py", "--out", "/kaggle/working/outputs/eda")
for figure in sorted(Path("/kaggle/working/outputs/eda/figures").glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure)))


### Hallazgos principales

- 120 imágenes, 1790 instancias y 120 categorías.
- El 71 % de los objetos ocupa menos del 1 % de la imagen: el recall de objetos chicos es el
  principal riesgo.
- La evaluación LVIS es federada; las categorías no verificadas se ignoran.
- El GT contiene objetos con dos etiquetas superpuestas, por lo que el NMS entre clases se deja
  apagado en la configuración final.


## 3. Metodología

**Percepción.** OWLv2 recibe las 120 categorías, aplica umbral global 0,18 y NMS por clase.
SAM2.1 hiera-large recibe exclusivamente las cajas predichas y produce máscaras.

**Representación.** Área, centroide, posición, tamaño, color y relaciones se derivan de las
máscaras predichas.

**QA.** El modo determinista traduce los campos estructurados al DSL. En el modo LLM,
Qwen2.5-3B recibe solo la pregunta, genera el mismo DSL validado y el ejecutor calcula la respuesta.
El descriptor recibe solo un resumen textual del SceneJSON.

**Trazabilidad.** Cada etapa escribe tiempos, entradas, salidas, errores y artefactos en JSONL.


## 4. Pipeline final sobre desarrollo

La inferencia siguiente no abre anotaciones. `--eval` ejecuta la evaluación recién después de
guardar las predicciones y mantiene la frontera en los paquetes `eval`/`eda`.


In [ ]:
DEV_RUN = run_pipeline("--split", "dev", "--eval")
print("Run dev:", DEV_RUN)


In [ ]:
run_script("scripts/run_eval.py", "--run", DEV_RUN)


In [ ]:

dev_metrics = {
    "detection": read_json(DEV_RUN / "metrics_detection.json")["summary"],
    "segmentation": read_json(DEV_RUN / "metrics_segmentation.json")["summary"],
    "scene": read_json(DEV_RUN / "metrics_scene.json"),
}
dev_metrics


## 5. Diagnóstico, ajuste y propagación de errores

Las cajas GT se usan solamente como oráculo de evaluación para aislar el error de SAM2. El
barrido selecciona el umbral en tuning y reporta una única vez sobre holdout. El embudo identifica
la primera etapa en la que se pierde cada instancia.


In [ ]:

if RUN_ORACLE_SEGMENTATION:
    run_script("scripts/eval_segmentation.py", "--run", DEV_RUN)
else:
    print("Oráculo omitido por configuración")

if RUN_THRESHOLD_TUNING:
    run_script("scripts/tune_thresholds.py")
else:
    print("Tuning omitido por configuración")

run_script("scripts/analyze_errors.py", "--run", DEV_RUN)
error_summary = read_json(DEV_RUN / "error_analysis" / "summary.json")
error_summary


In [ ]:

for figure in sorted((DEV_RUN / "error_analysis" / "gallery").glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure)))


## 6. Experimento controlado

Hipótesis: aplicar el post-proceso de SAM2 solo a máscaras chicas mejora su F1 sin perjudicar
medium/large. Se modifica una sola variable y se decide con holdout. El experimento previo fue
descartado: mejoró con cajas GT, pero el F1 end-to-end de máscaras bajó 0,614 → 0,609.


In [ ]:

EXPERIMENT_RUN = None
if RUN_CONTROLLED_EXPERIMENT:
    experiment_configs = BASE_CONFIGS + ["configs/experiments/e1_30_pp_solo_chicas.yaml"]
    EXPERIMENT_RUN = run_pipeline("--split", "dev", configs=experiment_configs)
    run_script(
        "scripts/compare_runs.py",
        "--base", DEV_RUN,
        "--exp", EXPERIMENT_RUN,
        "--name", "e1_30_pp_solo_chicas",
        "--hypothesis", "post-proceso solo en máscaras chicas sube el F1 de máscaras",
        "--change", "apply_below_area_ratio: null -> 0.00333",
        "--conclusion", "descartado: baja el F1 de máscaras en holdout",
    )
else:
    print("Experimento omitido por configuración")


## 7. Costo del LLM

Se comparan los candidatos con los mismos prompts, una pasada de calentamiento y tres repeticiones.
La decisión previa eligió Qwen2.5-3B fp16: ~5994 MiB y 5,3 s promedio, frente a 2,6 GiB y
10,9 s de Qwen3-4B 4-bit en una T4.


In [ ]:

if RUN_LLM_BENCHMARK:
    run_script(
        "scripts/benchmark_llm.py",
        "--output", "/kaggle/working/outputs/llm_benchmark.json",
    )
    display(read_json("/kaggle/working/outputs/llm_benchmark.json"))
else:
    print("Benchmark LLM omitido por configuración")


## 8. Inferencia de test/benchmark y preguntas

Estas 80 imágenes se procesan sin GT. El LLM nunca recibe la imagen ni el SceneJSON completo:
solo el texto de cada pregunta; el mismo ejecutor determinista calcula la respuesta.


In [ ]:

SUBMISSION_RUN = run_pipeline("--split", "test", "benchmark")
run_script("scripts/run_qa.py", "--run", SUBMISSION_RUN)
run_script("scripts/run_qa_llm.py", "--run", SUBMISSION_RUN)
print("Run submission:", SUBMISSION_RUN)


In [ ]:
run_script("scripts/eval_qa.py", "--run", SUBMISSION_RUN)


In [ ]:

SUBMISSION = Path("/kaggle/working/submission.csv")
run_script("scripts/make_submission.py", "--run", SUBMISSION_RUN, "--out", SUBMISSION)
qa_metrics = read_json(SUBMISSION_RUN / "metrics_qa.json")
qa_metrics


## 9. Resultados y discusión

En la corrida de referencia, el umbral 0,18 obtuvo F1 de cajas **0,653** y F1 de máscaras
**0,596** sobre las 120 imágenes. La consistencia estructural fue perfecta en 6905 objetos. El QA
determinista y LLM alcanzaron 72,5 % contra el oráculo; las 11 respuestas erróneas provinieron de
percepción, no de razonamiento. La submission aceptada obtuvo `PipelineScore = 0.6509`.

El embudo muestra que la mayor pérdida está en detección y se concentra en objetos chicos. Esto
motiva tiling tipo SAHI como extensión de Etapa 2, pero no se usa en la entrega de Etapa 1.


In [ ]:

from scene.io.submission import read_submission

submission_rows = read_submission(SUBMISSION)
assert len(submission_rows) == 80
required = [
    DEV_RUN / "run_manifest.json",
    DEV_RUN / "trace.jsonl",
    DEV_RUN / "metrics_detection.json",
    DEV_RUN / "metrics_segmentation.json",
    DEV_RUN / "metrics_scene.json",
    DEV_RUN / "error_analysis" / "summary.json",
    SUBMISSION_RUN / "metrics_qa.json",
    SUBMISSION,
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, missing

final_summary = {
    "group": '15',
    "course": '5K1',
    "kaggle_team": 'Grupo 15 - 5K1',
    "dev_run": str(DEV_RUN),
    "experiment_run": str(EXPERIMENT_RUN) if EXPERIMENT_RUN else None,
    "submission_run": str(SUBMISSION_RUN),
    "submission_rows": len(submission_rows),
    "reference_pipeline_score": 0.6509,
    "metrics": {"dev": dev_metrics, "qa": qa_metrics},
}
summary_path = Path("/kaggle/working/outputs/final_summary.json")
summary_path.write_text(json.dumps(final_summary, indent=2, default=str), encoding="utf-8")

import zipfile

artifact_zip = Path("/kaggle/working/grupo15_entrega_artifacts.zip")
artifact_files = [SUBMISSION, summary_path]
for run in (DEV_RUN, SUBMISSION_RUN):
    for pattern in ("metrics*.json", "*.csv", "trace.jsonl", "run_manifest.json"):
        artifact_files.extend(run.glob(pattern))
for directory in (
    Path("/kaggle/working/outputs/eda"),
    Path("/kaggle/working/outputs/tuning"),
    DEV_RUN / "error_analysis",
):
    if directory.exists():
        artifact_files.extend(path for path in directory.rglob("*") if path.is_file())
for optional in (
    Path("/kaggle/working/outputs/experiments.csv"),
    Path("/kaggle/working/outputs/llm_benchmark.json"),
):
    if optional.is_file():
        artifact_files.append(optional)
with zipfile.ZipFile(artifact_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(artifact_files)):
        archive.write(path, path.relative_to("/kaggle/working"))

print("Entrega validada. Resumen:", summary_path)
print("Artefactos para reporte/checklist:", artifact_zip)
display(final_summary)


## 10. Conclusiones y limitaciones

La pipeline supera la meta de F1 de detección, entrega máscaras y SceneJSON consistentes, y
responde mediante dos modos que comparten el mismo razonador verificable. La trazabilidad permite
atribuir los errores a la etapa correspondiente. Las principales limitaciones son el recall de
objetos chicos, la sensibilidad de SAM2 a cajas imprecisas y la dependencia de un léxico español
manual. El LLM no corrige percepción ni ve la imagen.
